In [ ]:
# ============================================================
# ΒΗΜΑ 7C.2:
# Αρχικοποίηση και τελικός έλεγχος του docking notebook
#
# Το κελί:
# - επιβεβαιώνει το σωστό Python kernel,
# - ελέγχει SMINA, OpenBabel, PyMOL και Meeko,
# - ελέγχει ότι υπάρχουν τα αρχεία 3BE2 και RAJ,
# - δεν εκτελεί ακόμη receptor ή ligand preparation.
# ============================================================

import os
import sys
import shutil
import subprocess
from pathlib import Path

import pandas as pd
import rdkit
import Bio
import meeko

from IPython.display import display


# ============================================================
# 1. Βασικές διαδρομές
# ============================================================

PROJECT_DIRECTORY = Path(
    "/home/afroditi/ptixiaki"
)

os.chdir(
    PROJECT_DIRECTORY
)

print(
    f"Working directory: {Path.cwd()}"
)


# ============================================================
# 2. Αρχεία που πρέπει να έχουν δημιουργηθεί
# από το προηγούμενο structural audit
# ============================================================

required_files = [
    "3BE2.pdb",
    "3BE2_RAJ_CRYSTAL_LIGAND.pdb",
    "3BE2_RAJ_CRYSTAL_COORDINATES.csv",
    "3BE2_RAJ_GEOMETRY.csv",
    "3BE2_CHAIN_A_PROTEIN_ONLY_UNPREPARED.pdb",
    "SELECTED_20_GENERATED_CANDIDATES.csv"
]


# ============================================================
# 3. Command-line εργαλεία
# ============================================================

required_commands = [
    "smina",
    "obabel",
    "pymol",
    "mk_prepare_ligand.py",
    "mk_prepare_receptor.py"
]


# ============================================================
# 4. Συνάρτηση ασφαλούς εκτέλεσης command
# ============================================================

def run_command_check(command):
    """
    Εκτελεί μία σύντομη εντολή ελέγχου και επιστρέφει
    την πρώτη γραμμή της εξόδου.
    """

    try:
        completed_process = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=30
        )

        output = (
            completed_process.stdout.strip()
            or completed_process.stderr.strip()
        )

        first_line = (
            output.splitlines()[0]
            if output
            else "No version text returned"
        )

        return (
            completed_process.returncode,
            first_line
        )

    except Exception as error:
        return (
            None,
            f"{type(error).__name__}: {error}"
        )


# ============================================================
# 5. Έλεγχος του Python kernel
# ============================================================

expected_environment_fragment = (
    "/miniconda3/envs/docking_env/"
)

correct_kernel = (
    expected_environment_fragment
    in sys.executable
)

print("\n" + "=" * 85)
print("PYTHON KERNEL")
print("=" * 85)

print(
    f"Python executable: {sys.executable}"
)

print(
    f"Python version: "
    f"{sys.version.replace(chr(10), ' ')}"
)

print(
    f"Correct docking kernel: {correct_kernel}"
)

if not correct_kernel:
    raise RuntimeError(
        "Το notebook δεν χρησιμοποιεί το kernel "
        "'Python (docking_env)'.\n"
        "Άλλαξε kernel πριν συνεχίσεις."
    )


# ============================================================
# 6. Έλεγχος command-line εργαλείων
# ============================================================

command_check_definitions = {
    "smina": [
        "smina",
        "--help"
    ],
    "obabel": [
        "obabel",
        "-V"
    ],
    "pymol": [
        "pymol",
        "-cq",
        "-d",
        "print('PyMOL OK'); quit"
    ],
    "mk_prepare_ligand.py": [
        "mk_prepare_ligand.py",
        "--help"
    ],
    "mk_prepare_receptor.py": [
        "mk_prepare_receptor.py",
        "--help"
    ]
}


command_rows = []

for command_name in required_commands:

    command_path = shutil.which(
        command_name
    )

    available = (
        command_path is not None
    )

    if available:
        return_code, output_text = run_command_check(
            command_check_definitions[
                command_name
            ]
        )
    else:
        return_code = None
        output_text = "Not found in PATH"

    command_rows.append({
        "Tool": command_name,
        "Available": available,
        "Path": command_path,
        "Return_Code": return_code,
        "Check_Output": output_text
    })


command_audit_df = pd.DataFrame(
    command_rows
)


print("\n" + "=" * 85)
print("DOCKING COMMANDS")
print("=" * 85)

display(
    command_audit_df
)


missing_commands = command_audit_df[
    ~command_audit_df["Available"]
]["Tool"].tolist()

if missing_commands:
    raise RuntimeError(
        "Λείπουν τα ακόλουθα εργαλεία:\n"
        f"{missing_commands}"
    )


# ============================================================
# 7. Έλεγχος απαιτούμενων αρχείων
# ============================================================

file_rows = []

for filename in required_files:

    file_path = PROJECT_DIRECTORY / filename

    exists = file_path.exists()

    file_rows.append({
        "File": filename,
        "Exists": exists,
        "Absolute_Path": str(file_path),
        "Size_Bytes": (
            file_path.stat().st_size
            if exists
            else 0
        )
    })


file_audit_df = pd.DataFrame(
    file_rows
)


print("\n" + "=" * 85)
print("REQUIRED FILES")
print("=" * 85)

display(
    file_audit_df
)


missing_files = file_audit_df[
    ~file_audit_df["Exists"]
]["File"].tolist()

if missing_files:
    raise FileNotFoundError(
        "Λείπουν τα ακόλουθα αρχεία:\n"
        f"{missing_files}"
    )


# ============================================================
# 8. Έλεγχος Python packages
# ============================================================

package_versions_df = pd.DataFrame([
    {
        "Package": "RDKit",
        "Version": rdkit.__version__
    },
    {
        "Package": "BioPython",
        "Version": Bio.__version__
    },
    {
        "Package": "Meeko",
        "Version": getattr(
            meeko,
            "__version__",
            "Version not exposed"
        )
    }
])


print("\n" + "=" * 85)
print("PYTHON PACKAGES")
print("=" * 85)

display(
    package_versions_df
)


# ============================================================
# 9. Αποθήκευση τελικού audit
# ============================================================

notebook_audit_output = (
    "DOCKING_NOTEBOOK_INITIALIZATION_AUDIT.csv"
)

combined_audit_df = pd.concat(
    [
        command_audit_df.assign(
            Category="Command"
        ),
        file_audit_df.rename(
            columns={
                "File": "Tool",
                "Exists": "Available",
                "Absolute_Path": "Path",
                "Size_Bytes": "Return_Code"
            }
        ).assign(
            Category="File",
            Check_Output=""
        )
    ],
    ignore_index=True
)

combined_audit_df.to_csv(
    notebook_audit_output,
    index=False
)


# ============================================================
# 10. Τελική αναφορά
# ============================================================

print("\n" + "-" * 85)
print("ΤΟ DOCKING NOTEBOOK ΕΙΝΑΙ ΕΤΟΙΜΟ")
print("-" * 85)

print(
    f"Kernel: {sys.executable}"
)

print(
    f"Project directory: {PROJECT_DIRECTORY}"
)

print(
    f"Audit file: {notebook_audit_output}"
)

print(
    "\nΔεν πραγματοποιήθηκε ακόμη receptor preparation, "
    "ligand preparation ή docking."
)

In [ ]:
# ============================================================
# ΒΗΜΑ 7C.3A:
# Καθαρισμός και προετοιμασία του receptor 3BE2
#
# Μεθοδολογία:
# 1. Διατήρηση μόνο της protein chain A
# 2. Διατήρηση κύριου AltLoc: κενό ή A
# 3. Αφαίρεση όλων των hydrogens από το αρχικό PDB
# 4. Προσθήκη hydrogens με PyMOL
# 5. Μετατροπή σε receptor PDBQT με Meeko
# 6. Έλεγχος ότι τα heavy atoms δεν μετακινήθηκαν
#
# Δεν πραγματοποιείται ακόμη:
# - ligand preparation,
# - docking,
# - δημιουργία docking box,
# - mutation reversal,
# - modelling των missing residues.
# ============================================================

import os
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 1. Βασικές διαδρομές
# ============================================================

PROJECT_DIRECTORY = Path(
    "/home/afroditi/ptixiaki"
)

os.chdir(
    PROJECT_DIRECTORY
)

INPUT_RECEPTOR = Path(
    "3BE2_CHAIN_A_PROTEIN_ONLY_UNPREPARED.pdb"
)

CLEAN_RECEPTOR_NO_H = Path(
    "3BE2_CHAIN_A_CLEAN_NO_H.pdb"
)

PYMOL_SCRIPT = Path(
    "PREPARE_3BE2_RECEPTOR_HYDROGENS.pml"
)

HYDROGENATED_RECEPTOR = Path(
    "3BE2_CHAIN_A_CLEAN_WITH_H.pdb"
)

MEEKO_OUTPUT_BASENAME = Path(
    "3BE2_RECEPTOR_PREPARED"
)

PREPARED_RECEPTOR_PDBQT = Path(
    "3BE2_RECEPTOR_PREPARED.pdbqt"
)

PREPARED_RECEPTOR_PDB = Path(
    "3BE2_RECEPTOR_MEEKO_PREPARED.pdb"
)

PREPARATION_LOG = Path(
    "3BE2_RECEPTOR_PREPARATION_LOG.txt"
)

PREPARATION_AUDIT = Path(
    "3BE2_RECEPTOR_PREPARATION_AUDIT.csv"
)


# ============================================================
# 2. Έλεγχος input και εργαλείων
# ============================================================

if not INPUT_RECEPTOR.exists():
    raise FileNotFoundError(
        f"Δεν βρέθηκε το αρχείο: {INPUT_RECEPTOR}"
    )

required_commands = [
    "pymol",
    "mk_prepare_receptor.py"
]

missing_commands = [
    command
    for command in required_commands
    if shutil.which(command) is None
]

if missing_commands:
    raise RuntimeError(
        "Λείπουν τα ακόλουθα εργαλεία:\n"
        f"{missing_commands}"
    )


# ============================================================
# 3. Βοηθητικές συναρτήσεις PDB
# ============================================================

def get_element_from_pdb_line(line):
    """
    Διαβάζει το χημικό στοιχείο από μία γραμμή PDB.

    Αν η στήλη element είναι κενή, γίνεται ασφαλές fallback
    από το atom name.
    """

    element = ""

    if len(line) >= 78:
        element = line[76:78].strip().upper()

    if not element:
        atom_name = line[12:16].strip()

        atom_name_without_digits = "".join(
            character
            for character in atom_name
            if not character.isdigit()
        )

        element = (
            atom_name_without_digits[0].upper()
            if atom_name_without_digits
            else ""
        )

    return element


def parse_pdb_atoms(pdb_path):
    """
    Διαβάζει ATOM και HETATM records από αρχείο PDB.
    """

    atom_rows = []

    with open(
        pdb_path,
        "r",
        encoding="utf-8",
        errors="replace"
    ) as pdb_handle:

        for line in pdb_handle:

            record_type = line[0:6].strip()

            if record_type not in {
                "ATOM",
                "HETATM"
            }:
                continue

            if len(line) < 54:
                continue

            try:
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
            except ValueError:
                continue

            atom_rows.append({
                "Record_Type": record_type,
                "Atom_Name": line[12:16].strip(),
                "AltLoc": line[16:17].strip(),
                "Residue_Name": line[17:20].strip(),
                "Chain_ID": line[21:22].strip(),
                "Residue_Number": line[22:26].strip(),
                "Insertion_Code": line[26:27].strip(),
                "X": x,
                "Y": y,
                "Z": z,
                "Element": get_element_from_pdb_line(line)
            })

    return pd.DataFrame(atom_rows)


def atom_identity_key(atom_row):
    """
    Δημιουργεί μοναδικό identifier για protein heavy atom.
    """

    return (
        atom_row["Chain_ID"],
        atom_row["Residue_Number"],
        atom_row["Insertion_Code"],
        atom_row["Residue_Name"],
        atom_row["Atom_Name"]
    )


# ============================================================
# 4. Δημιουργία καθαρού receptor χωρίς hydrogens
#
# Διατηρούμε μόνο:
# - ATOM records,
# - chain A,
# - AltLoc κενό ή A,
# - heavy atoms.
#
# Αν υπάρχουν και blank και A records για το ίδιο atom,
# δίνεται προτεραιότητα στο blank AltLoc.
# ============================================================

input_lines = INPUT_RECEPTOR.read_text(
    encoding="utf-8",
    errors="replace"
).splitlines()


selected_atoms = {}


for original_order, line in enumerate(input_lines):

    if not line.startswith("ATOM"):
        continue

    if len(line) < 54:
        continue

    chain_id = line[21:22].strip()
    altloc = line[16:17].strip()
    element = get_element_from_pdb_line(line)

    if chain_id != "A":
        continue

    if altloc not in {
        "",
        "A"
    }:
        continue

    if element in {
        "H",
        "D"
    }:
        continue

    atom_name = line[12:16].strip()
    residue_name = line[17:20].strip()
    residue_number = line[22:26].strip()
    insertion_code = line[26:27].strip()

    atom_key = (
        chain_id,
        residue_number,
        insertion_code,
        residue_name,
        atom_name
    )

    # Blank AltLoc έχει υψηλότερη προτεραιότητα από AltLoc A.
    priority = (
        0
        if altloc == ""
        else 1
    )

    # Καθαρίζουμε τη στήλη AltLoc πριν την αποθήκευση.
    cleaned_line = (
        line[:16]
        + " "
        + line[17:]
    )

    previous_entry = selected_atoms.get(
        atom_key
    )

    if (
        previous_entry is None
        or priority < previous_entry["Priority"]
    ):
        selected_atoms[atom_key] = {
            "Priority": priority,
            "Original_Order": original_order,
            "Line": cleaned_line
        }


ordered_clean_atoms = sorted(
    selected_atoms.values(),
    key=lambda entry: entry["Original_Order"]
)


clean_output_lines = [
    "REMARK Clean receptor extracted from 3BE2",
    "REMARK Chain A only",
    "REMARK Waters, ligands and other heteroatoms removed",
    "REMARK Main alternate location retained",
    "REMARK Hydrogens not yet added"
]

clean_output_lines.extend(
    entry["Line"]
    for entry in ordered_clean_atoms
)

clean_output_lines.extend([
    "TER",
    "END"
])


CLEAN_RECEPTOR_NO_H.write_text(
    "\n".join(clean_output_lines) + "\n",
    encoding="utf-8"
)


# ============================================================
# 5. Έλεγχος του καθαρού receptor
# ============================================================

clean_no_h_df = parse_pdb_atoms(
    CLEAN_RECEPTOR_NO_H
)

if clean_no_h_df.empty:
    raise RuntimeError(
        "Το καθαρό receptor PDB δεν περιέχει atoms."
    )

if (
    clean_no_h_df["Record_Type"] != "ATOM"
).any():
    raise RuntimeError(
        "Το καθαρό receptor περιέχει HETATM records."
    )

if set(
    clean_no_h_df["Chain_ID"].unique()
) != {"A"}:
    raise RuntimeError(
        "Το καθαρό receptor δεν περιέχει αποκλειστικά chain A."
    )

if clean_no_h_df[
    "Element"
].isin(["H", "D"]).any():
    raise RuntimeError(
        "Το καθαρό receptor περιέχει ήδη hydrogens."
    )


# ============================================================
# 6. Προσθήκη hydrogens με PyMOL
#
# Η χρήση του PyMOL ακολουθεί τη βασική διαδικασία
# που περιγράφεται στο paper.
# ============================================================

pymol_script_text = f"""
reinitialize
load {CLEAN_RECEPTOR_NO_H.resolve().as_posix()}, receptor
remove hydro
h_add receptor
save {HYDROGENATED_RECEPTOR.resolve().as_posix()}, receptor
quit
""".strip()


PYMOL_SCRIPT.write_text(
    pymol_script_text + "\n",
    encoding="utf-8"
)


pymol_process = subprocess.run(
    [
        "pymol",
        "-cq",
        str(PYMOL_SCRIPT.resolve())
    ],
    capture_output=True,
    text=True,
    cwd=PROJECT_DIRECTORY,
    timeout=300
)


pymol_log = (
    "PYMOL COMMAND\n"
    + " ".join(pymol_process.args)
    + "\n\nPYMOL STDOUT\n"
    + pymol_process.stdout
    + "\n\nPYMOL STDERR\n"
    + pymol_process.stderr
)


if pymol_process.returncode != 0:
    PREPARATION_LOG.write_text(
        pymol_log,
        encoding="utf-8"
    )

    raise RuntimeError(
        "Η προσθήκη hydrogens με PyMOL απέτυχε.\n"
        f"Δες το αρχείο: {PREPARATION_LOG}"
    )


if (
    not HYDROGENATED_RECEPTOR.exists()
    or HYDROGENATED_RECEPTOR.stat().st_size == 0
):
    raise RuntimeError(
        "Το PyMOL ολοκληρώθηκε, αλλά δεν δημιουργήθηκε "
        "το hydrogenated receptor."
    )


# ============================================================
# 7. Έλεγχος του hydrogenated receptor
# ============================================================

hydrogenated_df = parse_pdb_atoms(
    HYDROGENATED_RECEPTOR
)

hydrogen_count = int(
    hydrogenated_df[
        "Element"
    ].isin(["H", "D"]).sum()
)

if hydrogen_count == 0:
    raise RuntimeError(
        "Το hydrogenated PDB δεν περιέχει hydrogens."
    )


# ============================================================
# 8. Έλεγχος ότι δεν μετακινήθηκαν τα heavy atoms
# ============================================================

clean_heavy_df = clean_no_h_df[
    ~clean_no_h_df["Element"].isin(
        ["H", "D"]
    )
].copy()

hydrogenated_heavy_df = hydrogenated_df[
    ~hydrogenated_df["Element"].isin(
        ["H", "D"]
    )
].copy()


clean_coordinates = {
    atom_identity_key(atom_row): np.array(
        [
            atom_row["X"],
            atom_row["Y"],
            atom_row["Z"]
        ],
        dtype=float
    )
    for _, atom_row in clean_heavy_df.iterrows()
}


hydrogenated_coordinates = {
    atom_identity_key(atom_row): np.array(
        [
            atom_row["X"],
            atom_row["Y"],
            atom_row["Z"]
        ],
        dtype=float
    )
    for _, atom_row in hydrogenated_heavy_df.iterrows()
}


clean_keys = set(
    clean_coordinates
)

hydrogenated_keys = set(
    hydrogenated_coordinates
)

missing_after_pymol = sorted(
    clean_keys - hydrogenated_keys
)

extra_after_pymol = sorted(
    hydrogenated_keys - clean_keys
)


if missing_after_pymol or extra_after_pymol:
    raise RuntimeError(
        "Η λίστα των heavy atoms άλλαξε μετά το PyMOL.\n"
        f"Missing atoms: {missing_after_pymol[:10]}\n"
        f"Extra atoms: {extra_after_pymol[:10]}"
    )


heavy_atom_displacements = [
    float(
        np.linalg.norm(
            clean_coordinates[key]
            - hydrogenated_coordinates[key]
        )
    )
    for key in clean_keys
]


maximum_heavy_atom_displacement = (
    max(heavy_atom_displacements)
    if heavy_atom_displacements
    else np.nan
)


# ============================================================
# 9. Δημιουργία receptor PDBQT με Meeko
#
# Δεν χρησιμοποιούμε --delete_bad_res.
# Αν υπάρχει προβληματικό residue, σταματάμε και το
# εξετάζουμε αντί να το διαγράψουμε αυτόματα.
# ============================================================

meeko_command = [
    "mk_prepare_receptor.py",
    "--read_pdb",
    str(HYDROGENATED_RECEPTOR.resolve()),
    "-o",
    str(MEEKO_OUTPUT_BASENAME.resolve()),
    "-p",
    str(PREPARED_RECEPTOR_PDBQT.resolve()),
    "--write_pdb",
    str(PREPARED_RECEPTOR_PDB.resolve())
]


meeko_process = subprocess.run(
    meeko_command,
    capture_output=True,
    text=True,
    cwd=PROJECT_DIRECTORY,
    timeout=600
)


meeko_log = (
    "\n\n"
    + "=" * 80
    + "\nMEEKO COMMAND\n"
    + " ".join(meeko_command)
    + "\n\nMEEKO STDOUT\n"
    + meeko_process.stdout
    + "\n\nMEEKO STDERR\n"
    + meeko_process.stderr
)


PREPARATION_LOG.write_text(
    pymol_log + meeko_log,
    encoding="utf-8"
)


if meeko_process.returncode != 0:
    print(meeko_process.stdout)
    print(meeko_process.stderr)

    raise RuntimeError(
        "Η προετοιμασία του receptor με Meeko απέτυχε.\n"
        "Δεν διαγράφηκε αυτόματα κανένα residue.\n"
        f"Δες το log: {PREPARATION_LOG}"
    )


if (
    not PREPARED_RECEPTOR_PDBQT.exists()
    or PREPARED_RECEPTOR_PDBQT.stat().st_size == 0
):
    raise RuntimeError(
        "Το Meeko ολοκληρώθηκε χωρίς να δημιουργήσει "
        "έγκυρο receptor PDBQT."
    )


# ============================================================
# 10. Βασικός έλεγχος του PDBQT
# ============================================================

pdbqt_lines = PREPARED_RECEPTOR_PDBQT.read_text(
    encoding="utf-8",
    errors="replace"
).splitlines()

pdbqt_atom_lines = [
    line
    for line in pdbqt_lines
    if line.startswith(("ATOM", "HETATM"))
]

if not pdbqt_atom_lines:
    raise RuntimeError(
        "Το receptor PDBQT δεν περιέχει atom records."
    )


for forbidden_residue in [
    "RAJ",
    "HOH",
    "WAT",
    "DOD"
]:
    if any(
        forbidden_residue in line
        for line in pdbqt_atom_lines
    ):
        raise RuntimeError(
            f"Το receptor PDBQT περιέχει το μη επιτρεπτό "
            f"residue: {forbidden_residue}"
        )


# ============================================================
# 11. Δημιουργία audit
# ============================================================

input_df = parse_pdb_atoms(
    INPUT_RECEPTOR
)

audit_df = pd.DataFrame([
    {
        "Parameter": "Input receptor",
        "Value": str(INPUT_RECEPTOR)
    },
    {
        "Parameter": "Input atom count",
        "Value": int(len(input_df))
    },
    {
        "Parameter": "Clean heavy-atom count",
        "Value": int(len(clean_heavy_df))
    },
    {
        "Parameter": "Hydrogenated total atom count",
        "Value": int(len(hydrogenated_df))
    },
    {
        "Parameter": "Hydrogens added by PyMOL",
        "Value": hydrogen_count
    },
    {
        "Parameter": "Heavy-atom identity preserved",
        "Value": (
            len(missing_after_pymol) == 0
            and len(extra_after_pymol) == 0
        )
    },
    {
        "Parameter": "Maximum heavy-atom displacement Angstrom",
        "Value": maximum_heavy_atom_displacement
    },
    {
        "Parameter": "Meeko return code",
        "Value": meeko_process.returncode
    },
    {
        "Parameter": "PDBQT atom record count",
        "Value": int(len(pdbqt_atom_lines))
    },
    {
        "Parameter": "RAJ present in receptor PDBQT",
        "Value": False
    },
    {
        "Parameter": "Water present in receptor PDBQT",
        "Value": False
    },
    {
        "Parameter": "Mutation reversal performed",
        "Value": False
    },
    {
        "Parameter": "Missing-residue modelling performed",
        "Value": False
    },
    {
        "Parameter": "Prepared receptor PDBQT",
        "Value": str(PREPARED_RECEPTOR_PDBQT)
    }
])


audit_df.to_csv(
    PREPARATION_AUDIT,
    index=False
)


# ============================================================
# 12. Εμφάνιση αποτελεσμάτων
# ============================================================

print("=" * 90)
print("3BE2 RECEPTOR PREPARATION AUDIT")
print("=" * 90)

display(
    audit_df
)


print("\nΑρχεία που δημιουργήθηκαν:")

created_files = [
    CLEAN_RECEPTOR_NO_H,
    PYMOL_SCRIPT,
    HYDROGENATED_RECEPTOR,
    PREPARED_RECEPTOR_PDB,
    PREPARED_RECEPTOR_PDBQT,
    PREPARATION_LOG,
    PREPARATION_AUDIT
]

for file_path in created_files:

    if file_path.exists():
        print(
            f"- {file_path} "
            f"({file_path.stat().st_size / 1024:.1f} KB)"
        )


print("\n" + "-" * 90)
print("Η ΠΡΟΕΤΟΙΜΑΣΙΑ ΤΟΥ RECEPTOR ΟΛΟΚΛΗΡΩΘΗΚΕ")
print("-" * 90)

print(
    "Δεν πραγματοποιήθηκε ακόμη ligand preparation ή docking."
)

In [ ]:
# ============================================================
# ΒΗΜΑ 7C.3B:
# Προετοιμασία του crystallographic ligand RAJ για redocking
#
# Στόχοι:
# 1. Λήψη του επίσημου RAJ ideal SDF από την RCSB
# 2. Ανάθεση σωστών bond orders στη crystallographic δομή
# 3. Διατήρηση των πειραματικών heavy-atom coordinates
# 4. Προσθήκη explicit hydrogens
# 5. Βελτιστοποίηση μόνο των hydrogen coordinates
# 6. Παραγωγή ligand PDBQT με Meeko
#
# Δεν πραγματοποιείται ακόμη:
# - docking,
# - RMSD calculation,
# - προετοιμασία των 20 generated candidates,
# - αλλαγή protonation state σε pH 7.4.
# ============================================================

import os
import shutil
import subprocess
from pathlib import Path
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdMolDescriptors

from IPython.display import display


# ============================================================
# 1. Βασικές διαδρομές
# ============================================================

PROJECT_DIRECTORY = Path(
    "/home/afroditi/ptixiaki"
)

os.chdir(
    PROJECT_DIRECTORY
)

CRYSTAL_LIGAND_PDB = Path(
    "3BE2_RAJ_CRYSTAL_LIGAND.pdb"
)

CRYSTAL_GEOMETRY_CSV = Path(
    "3BE2_RAJ_GEOMETRY.csv"
)

RCSB_IDEAL_SDF = Path(
    "RAJ_IDEAL_CCD.sdf"
)

REFERENCE_HEAVY_SDF = Path(
    "3BE2_RAJ_REFERENCE_CRYSTAL_HEAVY.sdf"
)

REFERENCE_WITH_H_SDF = Path(
    "3BE2_RAJ_REFERENCE_CRYSTAL_WITH_H.sdf"
)

REFERENCE_LIGAND_PDBQT = Path(
    "3BE2_RAJ_REFERENCE_CRYSTAL.pdbqt"
)

ATOM_COORDINATE_AUDIT = Path(
    "3BE2_RAJ_REFERENCE_ATOM_COORDINATES.csv"
)

LIGAND_PREPARATION_AUDIT = Path(
    "3BE2_RAJ_LIGAND_PREPARATION_AUDIT.csv"
)

LIGAND_PREPARATION_LOG = Path(
    "3BE2_RAJ_LIGAND_PREPARATION_LOG.txt"
)

RCSB_IDEAL_SDF_URL = (
    "https://files.rcsb.org/ligands/download/RAJ_ideal.sdf"
)


# ============================================================
# 2. Έλεγχος input και εργαλείων
# ============================================================

required_files = [
    CRYSTAL_LIGAND_PDB,
    CRYSTAL_GEOMETRY_CSV
]

missing_files = [
    str(file_path)
    for file_path in required_files
    if not file_path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Λείπουν τα ακόλουθα input αρχεία:\n"
        f"{missing_files}"
    )


if shutil.which("mk_prepare_ligand.py") is None:
    raise RuntimeError(
        "Δεν βρέθηκε το mk_prepare_ligand.py στο PATH."
    )


# ============================================================
# 3. Λήψη του επίσημου RAJ ideal SDF
# ============================================================

def download_file(
    url,
    output_path
):
    """
    Κατεβάζει ένα αρχείο μόνο όταν δεν υπάρχει ήδη ή
    όταν το υπάρχον αρχείο είναι απροσδόκητα μικρό.
    """

    if (
        output_path.exists()
        and output_path.stat().st_size > 500
    ):
        print(
            f"Το επίσημο CCD αρχείο υπάρχει ήδη: "
            f"{output_path}"
        )
        return

    request = Request(
        url,
        headers={
            "User-Agent": (
                "Mozilla/5.0 thesis-redocking-workflow"
            )
        }
    )

    try:

        with urlopen(
            request,
            timeout=60
        ) as response:

            file_content = response.read()

    except Exception as error:

        raise RuntimeError(
            "Απέτυχε η λήψη του επίσημου RAJ ideal SDF.\n"
            f"Αρχικό σφάλμα: {type(error).__name__}: {error}"
        )


    if len(file_content) < 500:

        raise RuntimeError(
            "Το αρχείο που λήφθηκε είναι απροσδόκητα μικρό "
            "και δεν θα χρησιμοποιηθεί."
        )


    output_path.write_bytes(
        file_content
    )

    print(
        f"Κατέβηκε: {output_path} "
        f"({output_path.stat().st_size / 1024:.1f} KB)"
    )


download_file(
    RCSB_IDEAL_SDF_URL,
    RCSB_IDEAL_SDF
)


# ============================================================
# 4. Ανάγνωση του επίσημου CCD template
# ============================================================

template_supplier = Chem.SDMolSupplier(
    str(RCSB_IDEAL_SDF),
    removeHs=False,
    sanitize=True
)

template_molecules = [
    molecule
    for molecule in template_supplier
    if molecule is not None
]

if len(template_molecules) != 1:

    raise RuntimeError(
        "Το RAJ ideal SDF δεν περιέχει ακριβώς "
        "ένα έγκυρο μόριο."
    )


template_with_h = template_molecules[0]

template_heavy = Chem.RemoveHs(
    template_with_h
)


template_formula = rdMolDescriptors.CalcMolFormula(
    template_with_h
)

template_formal_charge = int(
    Chem.GetFormalCharge(
        template_with_h
    )
)

template_total_atom_count = int(
    template_with_h.GetNumAtoms()
)

template_heavy_atom_count = int(
    template_heavy.GetNumHeavyAtoms()
)

template_hydrogen_count = int(
    sum(
        atom.GetAtomicNum() == 1
        for atom in template_with_h.GetAtoms()
    )
)


print("=" * 90)
print("RCSB RAJ CCD TEMPLATE")
print("=" * 90)

print(
    f"Formula: {template_formula}"
)

print(
    f"Formal charge: {template_formal_charge}"
)

print(
    f"Total atoms: {template_total_atom_count}"
)

print(
    f"Heavy atoms: {template_heavy_atom_count}"
)

print(
    f"Explicit hydrogens: {template_hydrogen_count}"
)


# ============================================================
# 5. Ανάγνωση του crystallographic PDB ligand
#
# Αρχικά χρησιμοποιούμε μόνο τα CONECT records.
# Αν αυτά δεν επαρκούν, δοκιμάζουμε proximity bonding.
# ============================================================

def load_crystal_ligand(
    pdb_path,
    proximity_bonding
):
    """
    Διαβάζει το crystallographic PDB ligand χωρίς να
    απαιτεί σωστά bond orders στο PDB.
    """

    molecule = Chem.MolFromPDBFile(
        str(pdb_path),
        sanitize=False,
        removeHs=False,
        proximityBonding=proximity_bonding
    )

    if molecule is None:
        return None

    try:

        molecule = Chem.RemoveHs(
            molecule,
            sanitize=False
        )

    except Exception:
        pass

    return molecule


crystal_raw = load_crystal_ligand(
    CRYSTAL_LIGAND_PDB,
    proximity_bonding=False
)


if crystal_raw is None:

    crystal_raw = load_crystal_ligand(
        CRYSTAL_LIGAND_PDB,
        proximity_bonding=True
    )


if crystal_raw is None:

    raise RuntimeError(
        "Το RDKit δεν μπόρεσε να διαβάσει τον "
        "crystallographic RAJ."
    )


crystal_raw_heavy_count = int(
    crystal_raw.GetNumHeavyAtoms()
)


if (
    crystal_raw_heavy_count
    != template_heavy_atom_count
):

    raise RuntimeError(
        "Ο αριθμός heavy atoms δεν συμφωνεί μεταξύ "
        "του crystallographic RAJ και του CCD template.\n"
        f"Crystal: {crystal_raw_heavy_count}\n"
        f"CCD: {template_heavy_atom_count}"
    )


# ============================================================
# 6. Ανάθεση σωστών bond orders από το CCD template
# ============================================================

bond_assignment_method = (
    "PDB CONECT records"
)

try:

    crystal_with_bond_orders = (
        AllChem.AssignBondOrdersFromTemplate(
            template_heavy,
            crystal_raw
        )
    )

except Exception as first_error:

    print(
        "Η ανάθεση bond orders με αποκλειστική χρήση "
        "CONECT απέτυχε. Δοκιμή proximity bonding..."
    )

    crystal_raw_proximity = load_crystal_ligand(
        CRYSTAL_LIGAND_PDB,
        proximity_bonding=True
    )

    if crystal_raw_proximity is None:

        raise RuntimeError(
            "Απέτυχε και η εναλλακτική ανάγνωση "
            "με proximity bonding."
        )

    try:

        crystal_with_bond_orders = (
            AllChem.AssignBondOrdersFromTemplate(
                template_heavy,
                crystal_raw_proximity
            )
        )

        bond_assignment_method = (
            "RDKit proximity bonding fallback"
        )

    except Exception as second_error:

        raise RuntimeError(
            "Δεν ήταν δυνατή η ανάθεση bond orders "
            "στον crystallographic RAJ.\n"
            f"Πρώτη προσπάθεια: {first_error}\n"
            f"Δεύτερη προσπάθεια: {second_error}"
        )


# Πλήρες sanitization μετά την ανάθεση της χημείας.
try:

    Chem.SanitizeMol(
        crystal_with_bond_orders
    )

except Exception as error:

    raise RuntimeError(
        "Το μόριο απέτυχε στο RDKit sanitization "
        "μετά την ανάθεση bond orders.\n"
        f"{type(error).__name__}: {error}"
    )


# ============================================================
# 7. Έλεγχος χημικής ταυτότητας
# ============================================================

template_canonical_smiles = Chem.MolToSmiles(
    template_heavy,
    canonical=True,
    isomericSmiles=True
)

crystal_canonical_smiles = Chem.MolToSmiles(
    crystal_with_bond_orders,
    canonical=True,
    isomericSmiles=True
)

chemical_identity_match = (
    template_canonical_smiles
    == crystal_canonical_smiles
)


if not chemical_identity_match:

    raise RuntimeError(
        "Η χημική ταυτότητα του crystallographic ligand "
        "δεν συμφωνεί με το επίσημο CCD template.\n"
        f"CCD SMILES: {template_canonical_smiles}\n"
        f"Crystal SMILES: {crystal_canonical_smiles}"
    )


crystal_formula_before_h = (
    rdMolDescriptors.CalcMolFormula(
        crystal_with_bond_orders
    )
)

crystal_formal_charge = int(
    Chem.GetFormalCharge(
        crystal_with_bond_orders
    )
)


# ============================================================
# 8. Αποθήκευση reference heavy-atom SDF
#
# Αυτό είναι το αρχείο αναφοράς για το μελλοντικό RMSD.
# ============================================================

crystal_with_bond_orders.SetProp(
    "_Name",
    "3BE2_RAJ_CRYSTAL_REFERENCE_HEAVY"
)

crystal_with_bond_orders.SetProp(
    "PDB_ID",
    "3BE2"
)

crystal_with_bond_orders.SetProp(
    "CCD_ID",
    "RAJ"
)

crystal_with_bond_orders.SetProp(
    "Coordinate_Source",
    "Experimental crystallographic coordinates from 3BE2"
)

crystal_with_bond_orders.SetProp(
    "Bond_Order_Source",
    "RCSB Chemical Component Dictionary RAJ ideal SDF"
)


heavy_sdf_writer = Chem.SDWriter(
    str(REFERENCE_HEAVY_SDF)
)

heavy_sdf_writer.write(
    crystal_with_bond_orders
)

heavy_sdf_writer.close()


# ============================================================
# 9. Αποθήκευση των αρχικών heavy-atom coordinates
# ============================================================

initial_conformer = (
    crystal_with_bond_orders.GetConformer()
)

initial_heavy_coordinates = np.array([
    [
        initial_conformer.GetAtomPosition(
            atom_index
        ).x,
        initial_conformer.GetAtomPosition(
            atom_index
        ).y,
        initial_conformer.GetAtomPosition(
            atom_index
        ).z
    ]
    for atom_index in range(
        crystal_with_bond_orders.GetNumAtoms()
    )
])


# ============================================================
# 10. Προσθήκη explicit hydrogens
# ============================================================

crystal_with_h = Chem.AddHs(
    crystal_with_bond_orders,
    addCoords=True
)


expected_heavy_atom_count = (
    crystal_with_bond_orders.GetNumAtoms()
)

actual_total_atom_count = (
    crystal_with_h.GetNumAtoms()
)

actual_hydrogen_count = int(
    sum(
        atom.GetAtomicNum() == 1
        for atom in crystal_with_h.GetAtoms()
    )
)


if (
    crystal_with_h.GetNumHeavyAtoms()
    != template_heavy_atom_count
):

    raise RuntimeError(
        "Ο αριθμός heavy atoms άλλαξε μετά "
        "την προσθήκη hydrogens."
    )


# ============================================================
# 11. Βελτιστοποίηση μόνο των hydrogens
#
# Όλα τα heavy atoms προστίθενται ως fixed points.
# Πρώτη επιλογή είναι το MMFF94.
# Αν δεν υποστηρίζεται, χρησιμοποιείται UFF.
# ============================================================

optimization_method = None
optimization_status = None


if AllChem.MMFFHasAllMoleculeParams(
    crystal_with_h
):

    mmff_properties = (
        AllChem.MMFFGetMoleculeProperties(
            crystal_with_h,
            mmffVariant="MMFF94"
        )
    )

    force_field = (
        AllChem.MMFFGetMoleculeForceField(
            crystal_with_h,
            mmff_properties
        )
    )

    optimization_method = "MMFF94"

else:

    if not AllChem.UFFHasAllMoleculeParams(
        crystal_with_h
    ):
        raise RuntimeError(
            "Ο RAJ δεν υποστηρίζεται πλήρως ούτε "
            "από MMFF94 ούτε από UFF."
        )

    force_field = (
        AllChem.UFFGetMoleculeForceField(
            crystal_with_h
        )
    )

    optimization_method = "UFF"


# Κρατάμε ακίνητα όλα τα heavy atoms.
for atom_index in range(
    expected_heavy_atom_count
):

    force_field.AddFixedPoint(
        atom_index
    )


force_field.Initialize()

optimization_status = int(
    force_field.Minimize(
        maxIts=1000
    )
)


# Status:
# 0 = σύγκλιση
# 1 = δεν συγκλίνει πλήρως εντός του maxIts
if optimization_status not in {
    0,
    1
}:
    raise RuntimeError(
        "Η hydrogen-only optimization επέστρεψε "
        f"μη αναμενόμενο status: {optimization_status}"
    )


# ============================================================
# 12. Έλεγχος διατήρησης heavy-atom coordinates
# ============================================================

optimized_conformer = (
    crystal_with_h.GetConformer()
)

optimized_heavy_coordinates = np.array([
    [
        optimized_conformer.GetAtomPosition(
            atom_index
        ).x,
        optimized_conformer.GetAtomPosition(
            atom_index
        ).y,
        optimized_conformer.GetAtomPosition(
            atom_index
        ).z
    ]
    for atom_index in range(
        expected_heavy_atom_count
    )
])


heavy_coordinate_displacements = np.linalg.norm(
    initial_heavy_coordinates
    - optimized_heavy_coordinates,
    axis=1
)

maximum_heavy_displacement = float(
    heavy_coordinate_displacements.max()
)

mean_heavy_displacement = float(
    heavy_coordinate_displacements.mean()
)


if maximum_heavy_displacement > 1e-5:

    raise RuntimeError(
        "Τα crystallographic heavy atoms μετακινήθηκαν "
        "κατά τη hydrogen-only optimization.\n"
        f"Maximum displacement: "
        f"{maximum_heavy_displacement:.8f} Å"
    )


# ============================================================
# 13. Έλεγχος του ligand centroid
# ============================================================

calculated_center = (
    optimized_heavy_coordinates.mean(
        axis=0
    )
)

geometry_df = pd.read_csv(
    CRYSTAL_GEOMETRY_CSV
)


required_geometry_columns = [
    "Center_X",
    "Center_Y",
    "Center_Z"
]

missing_geometry_columns = [
    column
    for column in required_geometry_columns
    if column not in geometry_df.columns
]

if missing_geometry_columns:

    raise RuntimeError(
        "Το geometry CSV δεν περιέχει τις στήλες:\n"
        f"{missing_geometry_columns}"
    )


stored_center = geometry_df.loc[
    0,
    required_geometry_columns
].to_numpy(
    dtype=float
)


center_difference = float(
    np.linalg.norm(
        calculated_center
        - stored_center
    )
)


if center_difference > 0.01:

    raise RuntimeError(
        "Το ligand center δεν συμφωνεί με το προηγούμενο "
        "structural audit.\n"
        f"Difference: {center_difference:.6f} Å"
    )


# ============================================================
# 14. Αποθήκευση του SDF με explicit hydrogens
# ============================================================

crystal_with_h.SetProp(
    "_Name",
    "3BE2_RAJ_CRYSTAL_REFERENCE_WITH_H"
)

crystal_with_h.SetProp(
    "PDB_ID",
    "3BE2"
)

crystal_with_h.SetProp(
    "CCD_ID",
    "RAJ"
)

crystal_with_h.SetProp(
    "Formal_Charge",
    str(
        Chem.GetFormalCharge(
            crystal_with_h
        )
    )
)

crystal_with_h.SetProp(
    "Hydrogen_Optimization_Method",
    optimization_method
)

crystal_with_h.SetProp(
    "Heavy_Atoms_Fixed_During_Optimization",
    "True"
)


with_h_sdf_writer = Chem.SDWriter(
    str(REFERENCE_WITH_H_SDF)
)

with_h_sdf_writer.write(
    crystal_with_h
)

with_h_sdf_writer.close()


# ============================================================
# 15. Δημιουργία PDBQT με Meeko
# ============================================================

meeko_command = [
    "mk_prepare_ligand.py",
    "-i",
    str(
        REFERENCE_WITH_H_SDF.resolve()
    ),
    "-o",
    str(
        REFERENCE_LIGAND_PDBQT.resolve()
    )
]


meeko_process = subprocess.run(
    meeko_command,
    capture_output=True,
    text=True,
    cwd=PROJECT_DIRECTORY,
    timeout=300
)


log_text = (
    "MEEKO COMMAND\n"
    + " ".join(meeko_command)
    + "\n\nMEEKO STDOUT\n"
    + meeko_process.stdout
    + "\n\nMEEKO STDERR\n"
    + meeko_process.stderr
)


LIGAND_PREPARATION_LOG.write_text(
    log_text,
    encoding="utf-8"
)


if meeko_process.returncode != 0:

    print(
        meeko_process.stdout
    )

    print(
        meeko_process.stderr
    )

    raise RuntimeError(
        "Η προετοιμασία του RAJ με Meeko απέτυχε.\n"
        f"Δες το log: {LIGAND_PREPARATION_LOG}"
    )


if (
    not REFERENCE_LIGAND_PDBQT.exists()
    or REFERENCE_LIGAND_PDBQT.stat().st_size == 0
):

    raise RuntimeError(
        "Το Meeko δεν δημιούργησε έγκυρο ligand PDBQT."
    )


# ============================================================
# 16. Έλεγχος PDBQT
# ============================================================

pdbqt_lines = (
    REFERENCE_LIGAND_PDBQT.read_text(
        encoding="utf-8",
        errors="replace"
    )
    .splitlines()
)


pdbqt_atom_lines = [
    line
    for line in pdbqt_lines
    if line.startswith(
        ("ATOM", "HETATM")
    )
]

root_count = sum(
    line.strip() == "ROOT"
    for line in pdbqt_lines
)

branch_count = sum(
    line.startswith("BRANCH")
    for line in pdbqt_lines
)

torsdof_values = []

for line in pdbqt_lines:

    if line.startswith("TORSDOF"):

        parts = line.split()

        if len(parts) >= 2:

            try:
                torsdof_values.append(
                    int(parts[1])
                )
            except ValueError:
                pass


if not pdbqt_atom_lines:
    raise RuntimeError(
        "Το ligand PDBQT δεν περιέχει atom records."
    )

if root_count != 1:
    raise RuntimeError(
        f"Αναμενόταν ένα ROOT record, βρέθηκαν {root_count}."
    )

if len(torsdof_values) != 1:
    raise RuntimeError(
        "Δεν βρέθηκε ακριβώς ένα έγκυρο TORSDOF record."
    )


# ============================================================
# 17. Αναλυτικό coordinate audit
# ============================================================

coordinate_rows = []

for atom_index in range(
    expected_heavy_atom_count
):

    atom = crystal_with_h.GetAtomWithIdx(
        atom_index
    )

    atom_info = atom.GetPDBResidueInfo()

    atom_name = (
        atom_info.GetName().strip()
        if atom_info is not None
        else f"Atom_{atom_index}"
    )

    coordinate_rows.append({
        "Atom_Index": atom_index,
        "Atom_Name": atom_name,
        "Element": atom.GetSymbol(),
        "Initial_X": initial_heavy_coordinates[
            atom_index,
            0
        ],
        "Initial_Y": initial_heavy_coordinates[
            atom_index,
            1
        ],
        "Initial_Z": initial_heavy_coordinates[
            atom_index,
            2
        ],
        "Prepared_X": optimized_heavy_coordinates[
            atom_index,
            0
        ],
        "Prepared_Y": optimized_heavy_coordinates[
            atom_index,
            1
        ],
        "Prepared_Z": optimized_heavy_coordinates[
            atom_index,
            2
        ],
        "Displacement_Angstrom": (
            heavy_coordinate_displacements[
                atom_index
            ]
        )
    })


coordinate_audit_df = pd.DataFrame(
    coordinate_rows
)

coordinate_audit_df.to_csv(
    ATOM_COORDINATE_AUDIT,
    index=False
)


# ============================================================
# 18. Δημιουργία συνοπτικού preparation audit
# ============================================================

prepared_formula = (
    rdMolDescriptors.CalcMolFormula(
        crystal_with_h
    )
)

prepared_formal_charge = int(
    Chem.GetFormalCharge(
        crystal_with_h
    )
)


audit_df = pd.DataFrame([
    {
        "Parameter": "Ligand",
        "Value": "RAJ"
    },
    {
        "Parameter": "Structure source",
        "Value": "3BE2 crystallographic coordinates"
    },
    {
        "Parameter": "Bond-order source",
        "Value": "RCSB RAJ ideal CCD SDF"
    },
    {
        "Parameter": "Bond-assignment method",
        "Value": bond_assignment_method
    },
    {
        "Parameter": "CCD formula",
        "Value": template_formula
    },
    {
        "Parameter": "Prepared formula",
        "Value": prepared_formula
    },
    {
        "Parameter": "CCD formal charge",
        "Value": template_formal_charge
    },
    {
        "Parameter": "Prepared formal charge",
        "Value": prepared_formal_charge
    },
    {
        "Parameter": "Chemical identity matches CCD",
        "Value": chemical_identity_match
    },
    {
        "Parameter": "Heavy-atom count",
        "Value": template_heavy_atom_count
    },
    {
        "Parameter": "Explicit hydrogen count",
        "Value": actual_hydrogen_count
    },
    {
        "Parameter": "Total prepared atom count",
        "Value": actual_total_atom_count
    },
    {
        "Parameter": "Hydrogen optimization method",
        "Value": optimization_method
    },
    {
        "Parameter": "Optimization status",
        "Value": optimization_status
    },
    {
        "Parameter": "Maximum heavy-atom displacement Angstrom",
        "Value": maximum_heavy_displacement
    },
    {
        "Parameter": "Mean heavy-atom displacement Angstrom",
        "Value": mean_heavy_displacement
    },
    {
        "Parameter": "Ligand-center difference Angstrom",
        "Value": center_difference
    },
    {
        "Parameter": "Meeko return code",
        "Value": meeko_process.returncode
    },
    {
        "Parameter": "PDBQT atom record count",
        "Value": len(pdbqt_atom_lines)
    },
    {
        "Parameter": "PDBQT BRANCH record count",
        "Value": branch_count
    },
    {
        "Parameter": "PDBQT TORSDOF",
        "Value": torsdof_values[0]
    },
    {
        "Parameter": "Reference heavy SDF",
        "Value": str(REFERENCE_HEAVY_SDF)
    },
    {
        "Parameter": "Prepared ligand SDF",
        "Value": str(REFERENCE_WITH_H_SDF)
    },
    {
        "Parameter": "Prepared ligand PDBQT",
        "Value": str(REFERENCE_LIGAND_PDBQT)
    }
])


audit_df.to_csv(
    LIGAND_PREPARATION_AUDIT,
    index=False
)


# ============================================================
# 19. Εμφάνιση αποτελεσμάτων
# ============================================================

print("\n" + "=" * 90)
print("3BE2 RAJ LIGAND PREPARATION AUDIT")
print("=" * 90)

display(
    audit_df
)


print("\nΑρχεία που δημιουργήθηκαν:")

created_files = [
    RCSB_IDEAL_SDF,
    REFERENCE_HEAVY_SDF,
    REFERENCE_WITH_H_SDF,
    REFERENCE_LIGAND_PDBQT,
    ATOM_COORDINATE_AUDIT,
    LIGAND_PREPARATION_AUDIT,
    LIGAND_PREPARATION_LOG
]

for file_path in created_files:

    if file_path.exists():

        print(
            f"- {file_path} "
            f"({file_path.stat().st_size / 1024:.1f} KB)"
        )


print("\n" + "-" * 90)
print("Η ΠΡΟΕΤΟΙΜΑΣΙΑ ΤΟΥ RAJ ΟΛΟΚΛΗΡΩΘΗΚΕ")
print("-" * 90)

print(
    "Οι crystallographic heavy-atom coordinates "
    "διατηρήθηκαν για το μελλοντικό RMSD."
)

print(
    "Δεν πραγματοποιήθηκε ακόμη redocking."
)

In [ ]:
# ============================================================
# ΒΗΜΑ 7C.4:
# Καθορισμός docking box, redocking του RAJ και RMSD validation
#
# Μεθοδολογία:
# 1. Υπολογισμός του box από τα crystallographic heavy atoms
# 2. Buffer 4 Å σε κάθε πλευρά, σύμφωνα με το SMINA autobox
# 3. Redocking του RAJ με SMINA
# 4. Σταθερό seed και ένα CPU για αναπαραγωγιμότητα
# 5. Υπολογισμός direct και symmetry-corrected RMSD
# 6. Χωριστή αξιολόγηση sampling και ranking success
#
# Δεν πραγματοποιείται ακόμη:
# - docking των 20 generated molecules,
# - αλλαγή receptor preparation,
# - επανάληψη με διαφορετικές παραμέτρους,
# - επιλογή αποτελέσματος με βάση το καλύτερο RMSD.
# ============================================================

import os
import re
import time
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment

from rdkit import Chem
from rdkit.Geometry import Point3D

from IPython.display import display


# ============================================================
# 1. Βασικές διαδρομές
# ============================================================

PROJECT_DIRECTORY = Path(
    "/home/afroditi/ptixiaki"
)

os.chdir(
    PROJECT_DIRECTORY
)

RECEPTOR_PDBQT = Path(
    "3BE2_RECEPTOR_PREPARED.pdbqt"
)

REFERENCE_LIGAND_PDBQT = Path(
    "3BE2_RAJ_REFERENCE_CRYSTAL.pdbqt"
)

REFERENCE_HEAVY_SDF = Path(
    "3BE2_RAJ_REFERENCE_CRYSTAL_HEAVY.sdf"
)

REDOCKED_PDBQT = Path(
    "3BE2_RAJ_REDOCKED.pdbqt"
)

REDOCKING_LOG = Path(
    "3BE2_RAJ_REDOCKING.log"
)

REDOCKING_COMMAND_FILE = Path(
    "3BE2_RAJ_REDOCKING_COMMAND.txt"
)

DOCKING_BOX_OUTPUT = Path(
    "3BE2_RAJ_DOCKING_BOX.csv"
)

POSE_RMSD_OUTPUT = Path(
    "3BE2_RAJ_REDOCKING_POSE_RMSD.csv"
)

VALIDATION_SUMMARY_OUTPUT = Path(
    "3BE2_RAJ_REDOCKING_VALIDATION_SUMMARY.csv"
)

ALL_POSES_SDF = Path(
    "3BE2_RAJ_REDOCKED_POSES.sdf"
)

TOP_SCORING_POSE_SDF = Path(
    "3BE2_RAJ_TOP_SCORING_POSE.sdf"
)

BEST_RMSD_POSE_SDF = Path(
    "3BE2_RAJ_BEST_RMSD_POSE.sdf"
)


# ============================================================
# 2. Προκαθορισμένες παράμετροι
# ============================================================

# SMINA autobox default:
# 4 Å σε κάθε πλευρά του crystallographic ligand.
AUTOBOX_PADDING_ANGSTROM = 4.0

# SMINA default global-search parameters.
EXHAUSTIVENESS = 8
NUM_MODES = 9
ENERGY_RANGE_KCAL_MOL = 3

# Προσαρμογή για αναπαραγωγιμότητα.
RANDOM_SEED = 42
CPU_COUNT = 1

# Προκαθορισμένα όρια validation.
STRICT_RMSD_THRESHOLD = 1.5
STANDARD_RMSD_THRESHOLD = 2.0


# ============================================================
# 3. Έλεγχος input αρχείων και SMINA
# ============================================================

required_files = [
    RECEPTOR_PDBQT,
    REFERENCE_LIGAND_PDBQT,
    REFERENCE_HEAVY_SDF
]

missing_files = [
    str(file_path)
    for file_path in required_files
    if not file_path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Λείπουν τα ακόλουθα input αρχεία:\n"
        f"{missing_files}"
    )


smina_path = shutil.which(
    "smina"
)

if smina_path is None:
    raise RuntimeError(
        "Δεν βρέθηκε το SMINA στο ενεργό docking environment."
    )


# ============================================================
# 4. Ανάγνωση του crystallographic heavy-atom SDF
# ============================================================

reference_supplier = Chem.SDMolSupplier(
    str(REFERENCE_HEAVY_SDF),
    removeHs=True,
    sanitize=True
)

reference_molecules = [
    molecule
    for molecule in reference_supplier
    if molecule is not None
]

if len(reference_molecules) != 1:
    raise RuntimeError(
        "Το reference heavy SDF δεν περιέχει ακριβώς "
        "ένα έγκυρο μόριο."
    )


reference_molecule = reference_molecules[0]

if reference_molecule.GetNumConformers() != 1:
    raise RuntimeError(
        "Το reference molecule δεν περιέχει ακριβώς "
        "ένα conformer."
    )


reference_atom_count = int(
    reference_molecule.GetNumAtoms()
)

reference_heavy_atom_count = int(
    reference_molecule.GetNumHeavyAtoms()
)

if reference_atom_count != reference_heavy_atom_count:
    raise RuntimeError(
        "Το reference heavy SDF περιέχει hydrogens, "
        "ενώ αναμενόταν heavy-atom-only δομή."
    )


reference_conformer = (
    reference_molecule.GetConformer()
)

reference_coordinates = np.array([
    [
        reference_conformer.GetAtomPosition(
            atom_index
        ).x,
        reference_conformer.GetAtomPosition(
            atom_index
        ).y,
        reference_conformer.GetAtomPosition(
            atom_index
        ).z
    ]
    for atom_index in range(
        reference_atom_count
    )
], dtype=float)


# ============================================================
# 5. Υπολογισμός του docking box
#
# Το docking-box center είναι το μέσο των ελάχιστων και
# μέγιστων συντεταγμένων, όχι το centroid του ligand.
# ============================================================

coordinate_minimum = (
    reference_coordinates.min(
        axis=0
    )
)

coordinate_maximum = (
    reference_coordinates.max(
        axis=0
    )
)

ligand_span = (
    coordinate_maximum
    - coordinate_minimum
)

box_center = (
    coordinate_minimum
    + coordinate_maximum
) / 2.0

box_size = (
    ligand_span
    + 2.0 * AUTOBOX_PADDING_ANGSTROM
)

ligand_centroid = (
    reference_coordinates.mean(
        axis=0
    )
)


box_df = pd.DataFrame([
    {
        "PDB_ID": "3BE2",
        "Ligand": "RAJ",
        "Box_Definition": (
            "Crystallographic heavy-atom bounding box "
            "plus 4 Angstrom on each side"
        ),
        "Autobox_Padding_Each_Side_Angstrom":
            AUTOBOX_PADDING_ANGSTROM,
        "Ligand_Min_X": coordinate_minimum[0],
        "Ligand_Max_X": coordinate_maximum[0],
        "Ligand_Min_Y": coordinate_minimum[1],
        "Ligand_Max_Y": coordinate_maximum[1],
        "Ligand_Min_Z": coordinate_minimum[2],
        "Ligand_Max_Z": coordinate_maximum[2],
        "Ligand_Span_X": ligand_span[0],
        "Ligand_Span_Y": ligand_span[1],
        "Ligand_Span_Z": ligand_span[2],
        "Box_Center_X": box_center[0],
        "Box_Center_Y": box_center[1],
        "Box_Center_Z": box_center[2],
        "Box_Size_X": box_size[0],
        "Box_Size_Y": box_size[1],
        "Box_Size_Z": box_size[2],
        "Ligand_Centroid_X": ligand_centroid[0],
        "Ligand_Centroid_Y": ligand_centroid[1],
        "Ligand_Centroid_Z": ligand_centroid[2]
    }
])

box_df.to_csv(
    DOCKING_BOX_OUTPUT,
    index=False
)


print("=" * 95)
print("3BE2 RAJ DOCKING BOX")
print("=" * 95)

display(
    box_df
)


# ============================================================
# 6. Δημιουργία και καταγραφή SMINA command
#
# Χρησιμοποιούμε --autobox_ligand ώστε το SMINA να ορίσει
# το search space απευθείας από τον crystallographic RAJ.
#
# Οι default παράμετροι δηλώνονται ρητά.
# ============================================================

smina_command = [
    smina_path,
    "--receptor",
    str(RECEPTOR_PDBQT.resolve()),
    "--ligand",
    str(REFERENCE_LIGAND_PDBQT.resolve()),
    "--autobox_ligand",
    str(REFERENCE_LIGAND_PDBQT.resolve()),
    "--autobox_add",
    str(AUTOBOX_PADDING_ANGSTROM),
    "--exhaustiveness",
    str(EXHAUSTIVENESS),
    "--num_modes",
    str(NUM_MODES),
    "--energy_range",
    str(ENERGY_RANGE_KCAL_MOL),
    "--seed",
    str(RANDOM_SEED),
    "--cpu",
    str(CPU_COUNT),
    "--out",
    str(REDOCKED_PDBQT.resolve()),
    "--log",
    str(REDOCKING_LOG.resolve())
]


command_text = (
    " ".join(
        smina_command
    )
)

REDOCKING_COMMAND_FILE.write_text(
    command_text + "\n",
    encoding="utf-8"
)


print("\n" + "=" * 95)
print("SMINA REDOCKING COMMAND")
print("=" * 95)

print(
    command_text
)


# ============================================================
# 7. Αφαίρεση προηγούμενων outputs
#
# Αποφεύγουμε να αναλυθούν κατά λάθος αποτελέσματα
# από παλαιότερη εκτέλεση.
# ============================================================

for previous_output in [
    REDOCKED_PDBQT,
    REDOCKING_LOG
]:
    if previous_output.exists():
        previous_output.unlink()


# ============================================================
# 8. Εκτέλεση redocking
# ============================================================

start_time = time.perf_counter()

smina_process = subprocess.run(
    smina_command,
    capture_output=True,
    text=True,
    cwd=PROJECT_DIRECTORY,
    timeout=3600
)

runtime_seconds = (
    time.perf_counter()
    - start_time
)


if smina_process.returncode != 0:

    print(
        "SMINA STDOUT:"
    )
    print(
        smina_process.stdout
    )

    print(
        "\nSMINA STDERR:"
    )
    print(
        smina_process.stderr
    )

    raise RuntimeError(
        "Το redocking με SMINA απέτυχε.\n"
        f"Return code: {smina_process.returncode}\n"
        f"Δες επίσης το αρχείο: {REDOCKING_LOG}"
    )


if (
    not REDOCKED_PDBQT.exists()
    or REDOCKED_PDBQT.stat().st_size == 0
):
    raise RuntimeError(
        "Το SMINA ολοκληρώθηκε, αλλά δεν δημιουργήθηκε "
        "έγκυρο redocked PDBQT."
    )


print(
    f"\nSMINA runtime: {runtime_seconds:.2f} seconds"
)


# ============================================================
# 9. Βοηθητική συνάρτηση ανάγνωσης PDBQT atom
# ============================================================

def parse_pdbqt_atom_line(line):
    """
    Αναλύει μία ATOM/HETATM γραμμή PDBQT.
    """

    if not line.startswith(
        ("ATOM", "HETATM")
    ):
        return None

    if len(line) < 54:
        return None

    try:
        x = float(
            line[30:38]
        )
        y = float(
            line[38:46]
        )
        z = float(
            line[46:54]
        )
    except ValueError:
        return None

    atom_name = (
        line[12:16].strip()
    )

    split_fields = (
        line.split()
    )

    atom_type = (
        split_fields[-1]
        if split_fields
        else ""
    )

    return {
        "Atom_Name": atom_name,
        "Atom_Type": atom_type,
        "X": x,
        "Y": y,
        "Z": z,
        "Original_Line": line
    }


def is_pdbqt_hydrogen(atom_record):
    """
    Αναγνωρίζει hydrogen atom από τον PDBQT atom type.
    """

    atom_type = (
        atom_record["Atom_Type"]
        .strip()
        .upper()
    )

    return atom_type.startswith(
        "H"
    )


# ============================================================
# 10. Αντιστοίχιση του PDBQT atom order με το RDKit atom order
#
# Το PDBQT μπορεί να έχει διαφορετική σειρά atoms από το SDF.
# Η αντιστοίχιση γίνεται από τις αρχικές crystallographic
# συντεταγμένες με global one-to-one assignment.
# ============================================================

reference_pdbqt_lines = (
    REFERENCE_LIGAND_PDBQT.read_text(
        encoding="utf-8",
        errors="replace"
    )
    .splitlines()
)


reference_pdbqt_atoms = []

for line in reference_pdbqt_lines:

    atom_record = parse_pdbqt_atom_line(
        line
    )

    if atom_record is None:
        continue

    if is_pdbqt_hydrogen(
        atom_record
    ):
        continue

    reference_pdbqt_atoms.append(
        atom_record
    )


if (
    len(reference_pdbqt_atoms)
    != reference_heavy_atom_count
):
    raise RuntimeError(
        "Ο αριθμός heavy atoms του reference PDBQT "
        "δεν συμφωνεί με το reference SDF.\n"
        f"PDBQT: {len(reference_pdbqt_atoms)}\n"
        f"SDF: {reference_heavy_atom_count}"
    )


reference_pdbqt_coordinates = np.array([
    [
        atom_record["X"],
        atom_record["Y"],
        atom_record["Z"]
    ]
    for atom_record in reference_pdbqt_atoms
], dtype=float)


# Πίνακας αποστάσεων μεταξύ:
# - PDBQT atom order,
# - RDKit/SDF atom order.
distance_matrix = np.linalg.norm(
    reference_pdbqt_coordinates[
        :,
        np.newaxis,
        :
    ]
    -
    reference_coordinates[
        np.newaxis,
        :,
        :
    ],
    axis=2
)


pdbqt_indices, rdkit_indices = (
    linear_sum_assignment(
        distance_matrix
    )
)


mapping_distances = (
    distance_matrix[
        pdbqt_indices,
        rdkit_indices
    ]
)


maximum_mapping_distance = float(
    mapping_distances.max()
)


if maximum_mapping_distance > 0.05:
    raise RuntimeError(
        "Δεν ήταν δυνατή η ασφαλής αντιστοίχιση "
        "PDBQT και SDF atoms.\n"
        f"Maximum mapping distance: "
        f"{maximum_mapping_distance:.6f} Å"
    )


pdbqt_to_rdkit_atom_mapping = {
    int(pdbqt_index): int(rdkit_index)
    for pdbqt_index, rdkit_index
    in zip(
        pdbqt_indices,
        rdkit_indices
    )
}


reference_pdbqt_atom_signature = [
    (
        atom_record["Atom_Name"],
        atom_record["Atom_Type"]
    )
    for atom_record in reference_pdbqt_atoms
]


# ============================================================
# 11. Διαχωρισμός των SMINA output poses
# ============================================================

redocked_lines = (
    REDOCKED_PDBQT.read_text(
        encoding="utf-8",
        errors="replace"
    )
    .splitlines()
)


def split_pdbqt_models(lines):
    """
    Διαχωρίζει ένα multi-model PDBQT σε ανεξάρτητες poses.
    """

    models = []
    current_model = []
    inside_model = False
    model_number = None

    for line in lines:

        if line.startswith("MODEL"):

            if current_model:
                models.append({
                    "Model_Number": model_number,
                    "Lines": current_model
                })

            inside_model = True
            current_model = [
                line
            ]

            model_parts = line.split()

            model_number = (
                int(model_parts[1])
                if len(model_parts) >= 2
                and model_parts[1].isdigit()
                else len(models) + 1
            )

            continue

        if line.startswith("ENDMDL"):

            current_model.append(
                line
            )

            models.append({
                "Model_Number": model_number,
                "Lines": current_model
            })

            current_model = []
            inside_model = False
            model_number = None

            continue

        if inside_model:
            current_model.append(
                line
            )


    if not models:

        # Fallback για PDBQT χωρίς MODEL/ENDMDL.
        models = [{
            "Model_Number": 1,
            "Lines": list(lines)
        }]


    return models


pose_models = split_pdbqt_models(
    redocked_lines
)


if len(pose_models) == 0:
    raise RuntimeError(
        "Δεν βρέθηκαν docking poses στο SMINA output."
    )


# ============================================================
# 12. Εξαγωγή SMINA score από κάθε pose
# ============================================================

def extract_pose_score(model_lines):
    """
    Αναζητά SMINA/Vina affinity score μέσα στα REMARK records.
    """

    score_patterns = [
        r"minimizedAffinity\s+(-?\d+(?:\.\d+)?)",
        r"VINA\s+RESULT:\s+(-?\d+(?:\.\d+)?)",
        r"Affinity:\s+(-?\d+(?:\.\d+)?)"
    ]

    for line in model_lines:

        for pattern in score_patterns:

            match = re.search(
                pattern,
                line,
                flags=re.IGNORECASE
            )

            if match:

                return float(
                    match.group(1)
                )

    return np.nan


# ============================================================
# 13. Δημιουργία symmetry mappings
#
# Τα mappings χρησιμοποιούνται για symmetry-corrected RMSD.
# Δεν πραγματοποιείται fitting ή spatial alignment.
# ============================================================

automorphism_matches = (
    reference_molecule.GetSubstructMatches(
        reference_molecule,
        uniquify=False,
        useChirality=True,
        maxMatches=100000
    )
)


if len(automorphism_matches) == 0:

    automorphism_matches = [
        tuple(
            range(
                reference_atom_count
            )
        )
    ]


automorphism_count = int(
    len(automorphism_matches)
)


# ============================================================
# 14. Βοηθητική συνάρτηση RMSD χωρίς alignment
# ============================================================

def calculate_direct_rmsd(
    reference_xyz,
    pose_xyz
):
    """
    Υπολογίζει heavy-atom RMSD χωρίς fitting ή alignment.
    """

    squared_distances = np.sum(
        (
            reference_xyz
            - pose_xyz
        ) ** 2,
        axis=1
    )

    return float(
        np.sqrt(
            np.mean(
                squared_distances
            )
        )
    )


def calculate_symmetry_corrected_rmsd(
    reference_xyz,
    pose_xyz,
    symmetry_mappings
):
    """
    Υπολογίζει το μικρότερο direct RMSD μεταξύ όλων
    των έγκυρων molecular automorphisms.
    """

    best_rmsd = np.inf

    for symmetry_mapping in symmetry_mappings:

        mapped_pose_coordinates = (
            pose_xyz[
                np.asarray(
                    symmetry_mapping,
                    dtype=int
                )
            ]
        )

        current_rmsd = calculate_direct_rmsd(
            reference_xyz,
            mapped_pose_coordinates
        )

        if current_rmsd < best_rmsd:
            best_rmsd = current_rmsd

    return float(
        best_rmsd
    )


# ============================================================
# 15. Ανάλυση κάθε docking pose
# ============================================================

pose_rows = []
pose_molecules = []


for pose_position, pose_model in enumerate(
    pose_models,
    start=1
):

    model_lines = (
        pose_model["Lines"]
    )

    pose_atoms = []

    for line in model_lines:

        atom_record = parse_pdbqt_atom_line(
            line
        )

        if atom_record is None:
            continue

        if is_pdbqt_hydrogen(
            atom_record
        ):
            continue

        pose_atoms.append(
            atom_record
        )


    if (
        len(pose_atoms)
        != reference_heavy_atom_count
    ):
        raise RuntimeError(
            f"Η pose {pose_position} περιέχει "
            f"{len(pose_atoms)} heavy atoms, ενώ "
            f"αναμένονταν {reference_heavy_atom_count}."
        )


    pose_atom_signature = [
        (
            atom_record["Atom_Name"],
            atom_record["Atom_Type"]
        )
        for atom_record in pose_atoms
    ]


    if (
        pose_atom_signature
        != reference_pdbqt_atom_signature
    ):
        raise RuntimeError(
            f"Η σειρά ή η ταυτότητα των atoms άλλαξε "
            f"στη pose {pose_position}. Δεν μπορεί να "
            "υπολογιστεί ασφαλές RMSD."
        )


    pose_coordinates_pdbqt_order = np.array([
        [
            atom_record["X"],
            atom_record["Y"],
            atom_record["Z"]
        ]
        for atom_record in pose_atoms
    ], dtype=float)


    # Μετατροπή σε RDKit/reference atom order.
    pose_coordinates_rdkit_order = np.zeros(
        (
            reference_atom_count,
            3
        ),
        dtype=float
    )


    for (
        pdbqt_atom_index,
        rdkit_atom_index
    ) in pdbqt_to_rdkit_atom_mapping.items():

        pose_coordinates_rdkit_order[
            rdkit_atom_index
        ] = (
            pose_coordinates_pdbqt_order[
                pdbqt_atom_index
            ]
        )


    direct_rmsd = calculate_direct_rmsd(
        reference_coordinates,
        pose_coordinates_rdkit_order
    )


    symmetry_corrected_rmsd = (
        calculate_symmetry_corrected_rmsd(
            reference_coordinates,
            pose_coordinates_rdkit_order,
            automorphism_matches
        )
    )


    affinity_score = extract_pose_score(
        model_lines
    )


    pose_identifier = int(
        pose_model["Model_Number"]
        if pose_model["Model_Number"] is not None
        else pose_position
    )


    pose_rows.append({
        "Pose_ID": pose_identifier,
        "Output_Position": pose_position,
        "SMINA_Affinity_kcal_mol": affinity_score,
        "Direct_Heavy_Atom_RMSD_Angstrom": direct_rmsd,
        "Symmetry_Corrected_Heavy_Atom_RMSD_Angstrom":
            symmetry_corrected_rmsd,
        "Within_1_5_Angstrom": (
            symmetry_corrected_rmsd
            <= STRICT_RMSD_THRESHOLD
        ),
        "Within_2_0_Angstrom": (
            symmetry_corrected_rmsd
            <= STANDARD_RMSD_THRESHOLD
        )
    })


    # Δημιουργία RDKit molecule για μελλοντική απεικόνιση.
    pose_molecule = Chem.Mol(
        reference_molecule
    )

    pose_molecule.RemoveAllConformers()

    pose_conformer = Chem.Conformer(
        reference_atom_count
    )

    for atom_index, coordinates in enumerate(
        pose_coordinates_rdkit_order
    ):

        pose_conformer.SetAtomPosition(
            atom_index,
            Point3D(
                float(coordinates[0]),
                float(coordinates[1]),
                float(coordinates[2])
            )
        )


    pose_molecule.AddConformer(
        pose_conformer,
        assignId=True
    )

    pose_molecule.SetProp(
        "_Name",
        f"3BE2_RAJ_REDOCKED_POSE_{pose_identifier}"
    )

    pose_molecule.SetProp(
        "Pose_ID",
        str(pose_identifier)
    )

    pose_molecule.SetProp(
        "SMINA_Affinity_kcal_mol",
        (
            f"{affinity_score:.6f}"
            if not np.isnan(affinity_score)
            else "NaN"
        )
    )

    pose_molecule.SetProp(
        "Direct_RMSD_Angstrom",
        f"{direct_rmsd:.6f}"
    )

    pose_molecule.SetProp(
        "Symmetry_Corrected_RMSD_Angstrom",
        f"{symmetry_corrected_rmsd:.6f}"
    )

    pose_molecules.append(
        pose_molecule
    )


pose_results_df = pd.DataFrame(
    pose_rows
)


if pose_results_df.empty:
    raise RuntimeError(
        "Δεν δημιουργήθηκε πίνακας RMSD."
    )


# ============================================================
# 16. Αν δεν βρέθηκαν scores στα pose remarks,
# δοκιμάζουμε να τα διαβάσουμε από το SMINA log
# ============================================================

def extract_scores_from_smina_log(
    log_path
):
    """
    Εξάγει τον πίνακα mode/affinity από το SMINA log.
    """

    if not log_path.exists():
        return {}

    log_lines = log_path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    score_dictionary = {}

    for line in log_lines:

        match = re.match(
            r"^\s*(\d+)\s+(-?\d+(?:\.\d+)?)\s+",
            line
        )

        if match:

            mode_number = int(
                match.group(1)
            )

            affinity = float(
                match.group(2)
            )

            score_dictionary[
                mode_number
            ] = affinity

    return score_dictionary


if pose_results_df[
    "SMINA_Affinity_kcal_mol"
].isna().any():

    log_scores = extract_scores_from_smina_log(
        REDOCKING_LOG
    )

    for dataframe_index, pose_row in (
        pose_results_df.iterrows()
    ):

        if not np.isnan(
            pose_row[
                "SMINA_Affinity_kcal_mol"
            ]
        ):
            continue

        pose_id = int(
            pose_row["Pose_ID"]
        )

        if pose_id in log_scores:

            pose_results_df.loc[
                dataframe_index,
                "SMINA_Affinity_kcal_mol"
            ] = log_scores[
                pose_id
            ]


# ============================================================
# 17. Αποθήκευση όλων των poses σε SDF
# ============================================================

all_pose_writer = Chem.SDWriter(
    str(ALL_POSES_SDF)
)

for pose_index, pose_molecule in enumerate(
    pose_molecules
):

    result_row = pose_results_df.iloc[
        pose_index
    ]

    pose_molecule.SetProp(
        "SMINA_Affinity_kcal_mol",
        str(
            result_row[
                "SMINA_Affinity_kcal_mol"
            ]
        )
    )

    all_pose_writer.write(
        pose_molecule
    )

all_pose_writer.close()


# ============================================================
# 18. Εντοπισμός top-scoring και best-RMSD poses
#
# Σημαντικό:
# Η best-RMSD pose δεν χρησιμοποιείται ως top prediction.
# Καταγράφεται μόνο για να διαχωριστεί sampling από ranking.
# ============================================================

valid_score_mask = (
    pose_results_df[
        "SMINA_Affinity_kcal_mol"
    ].notna()
)


if valid_score_mask.any():

    top_scoring_dataframe_index = int(
        pose_results_df.loc[
            valid_score_mask,
            "SMINA_Affinity_kcal_mol"
        ].idxmin()
    )

else:

    # Fallback μόνο αν δεν υπάρχουν καθόλου scores.
    top_scoring_dataframe_index = int(
        pose_results_df.index[0]
    )


best_rmsd_dataframe_index = int(
    pose_results_df[
        "Symmetry_Corrected_Heavy_Atom_RMSD_Angstrom"
    ].idxmin()
)


top_scoring_row = pose_results_df.loc[
    top_scoring_dataframe_index
]

best_rmsd_row = pose_results_df.loc[
    best_rmsd_dataframe_index
]


# ============================================================
# 19. Αποθήκευση top-scoring και best-RMSD poses
# ============================================================

def write_single_molecule_sdf(
    molecule,
    output_path
):
    """
    Αποθηκεύει ένα RDKit molecule σε SDF.
    """

    writer = Chem.SDWriter(
        str(output_path)
    )

    writer.write(
        molecule
    )

    writer.close()


write_single_molecule_sdf(
    pose_molecules[
        top_scoring_dataframe_index
    ],
    TOP_SCORING_POSE_SDF
)

write_single_molecule_sdf(
    pose_molecules[
        best_rmsd_dataframe_index
    ],
    BEST_RMSD_POSE_SDF
)


# ============================================================
# 20. Αξιολόγηση sampling και ranking
# ============================================================

top_scoring_symmetry_rmsd = float(
    top_scoring_row[
        "Symmetry_Corrected_Heavy_Atom_RMSD_Angstrom"
    ]
)

best_symmetry_rmsd = float(
    best_rmsd_row[
        "Symmetry_Corrected_Heavy_Atom_RMSD_Angstrom"
    ]
)


ranking_success_1_5 = (
    top_scoring_symmetry_rmsd
    <= STRICT_RMSD_THRESHOLD
)

ranking_success_2_0 = (
    top_scoring_symmetry_rmsd
    <= STANDARD_RMSD_THRESHOLD
)

sampling_success_1_5 = (
    best_symmetry_rmsd
    <= STRICT_RMSD_THRESHOLD
)

sampling_success_2_0 = (
    best_symmetry_rmsd
    <= STANDARD_RMSD_THRESHOLD
)


if ranking_success_1_5:

    validation_interpretation = (
        "Strong top-ranked crystallographic pose recovery"
    )

elif ranking_success_2_0:

    validation_interpretation = (
        "Top-ranked pose recovered within 2 Angstrom"
    )

elif sampling_success_2_0:

    validation_interpretation = (
        "Near-native pose sampled, but SMINA ranking did not "
        "place it first"
    )

else:

    validation_interpretation = (
        "Redocking protocol did not recover a pose within "
        "2 Angstrom"
    )


# ============================================================
# 21. Δημιουργία summary
# ============================================================

validation_summary_df = pd.DataFrame([
    {
        "Parameter": "Receptor",
        "Value": str(RECEPTOR_PDBQT)
    },
    {
        "Parameter": "Reference ligand",
        "Value": str(REFERENCE_LIGAND_PDBQT)
    },
    {
        "Parameter": "Search-space method",
        "Value": "SMINA autobox from crystallographic RAJ"
    },
    {
        "Parameter": "Autobox padding each side Angstrom",
        "Value": AUTOBOX_PADDING_ANGSTROM
    },
    {
        "Parameter": "Box center X",
        "Value": box_center[0]
    },
    {
        "Parameter": "Box center Y",
        "Value": box_center[1]
    },
    {
        "Parameter": "Box center Z",
        "Value": box_center[2]
    },
    {
        "Parameter": "Box size X",
        "Value": box_size[0]
    },
    {
        "Parameter": "Box size Y",
        "Value": box_size[1]
    },
    {
        "Parameter": "Box size Z",
        "Value": box_size[2]
    },
    {
        "Parameter": "Random seed",
        "Value": RANDOM_SEED
    },
    {
        "Parameter": "CPU count",
        "Value": CPU_COUNT
    },
    {
        "Parameter": "Exhaustiveness",
        "Value": EXHAUSTIVENESS
    },
    {
        "Parameter": "Requested num modes",
        "Value": NUM_MODES
    },
    {
        "Parameter": "Generated pose count",
        "Value": len(pose_results_df)
    },
    {
        "Parameter": "Energy range kcal/mol",
        "Value": ENERGY_RANGE_KCAL_MOL
    },
    {
        "Parameter": "Runtime seconds",
        "Value": runtime_seconds
    },
    {
        "Parameter": "Heavy-atom count",
        "Value": reference_heavy_atom_count
    },
    {
        "Parameter": "Molecular automorphism count",
        "Value": automorphism_count
    },
    {
        "Parameter": "Maximum PDBQT-SDF mapping distance Angstrom",
        "Value": maximum_mapping_distance
    },
    {
        "Parameter": "Top-scoring pose ID",
        "Value": int(
            top_scoring_row["Pose_ID"]
        )
    },
    {
        "Parameter": "Top-scoring affinity kcal/mol",
        "Value": top_scoring_row[
            "SMINA_Affinity_kcal_mol"
        ]
    },
    {
        "Parameter": "Top-scoring pose direct RMSD Angstrom",
        "Value": top_scoring_row[
            "Direct_Heavy_Atom_RMSD_Angstrom"
        ]
    },
    {
        "Parameter": (
            "Top-scoring pose symmetry-corrected RMSD Angstrom"
        ),
        "Value": top_scoring_symmetry_rmsd
    },
    {
        "Parameter": "Ranking success within 1.5 Angstrom",
        "Value": ranking_success_1_5
    },
    {
        "Parameter": "Ranking success within 2.0 Angstrom",
        "Value": ranking_success_2_0
    },
    {
        "Parameter": "Best-RMSD pose ID",
        "Value": int(
            best_rmsd_row["Pose_ID"]
        )
    },
    {
        "Parameter": "Best symmetry-corrected RMSD Angstrom",
        "Value": best_symmetry_rmsd
    },
    {
        "Parameter": "Sampling success within 1.5 Angstrom",
        "Value": sampling_success_1_5
    },
    {
        "Parameter": "Sampling success within 2.0 Angstrom",
        "Value": sampling_success_2_0
    },
    {
        "Parameter": "Validation interpretation",
        "Value": validation_interpretation
    }
])


# ============================================================
# 22. Αποθήκευση αποτελεσμάτων
# ============================================================

pose_results_df = pose_results_df.sort_values(
    by=[
        "SMINA_Affinity_kcal_mol",
        "Pose_ID"
    ],
    ascending=[
        True,
        True
    ],
    na_position="last"
).reset_index(
    drop=True
)


pose_results_df.to_csv(
    POSE_RMSD_OUTPUT,
    index=False
)

validation_summary_df.to_csv(
    VALIDATION_SUMMARY_OUTPUT,
    index=False
)


# ============================================================
# 23. Εμφάνιση αποτελεσμάτων
# ============================================================

print("\n" + "=" * 95)
print("3BE2 RAJ REDOCKING POSES")
print("=" * 95)

display(
    pose_results_df
)


print("\n" + "=" * 95)
print("3BE2 RAJ REDOCKING VALIDATION SUMMARY")
print("=" * 95)

display(
    validation_summary_df
)


# ============================================================
# 24. Τελική αναφορά
# ============================================================

print("\n" + "-" * 95)
print("ΤΟ REDOCKING ΤΟΥ RAJ ΟΛΟΚΛΗΡΩΘΗΚΕ")
print("-" * 95)

print(
    f"Top-scoring pose symmetry-corrected RMSD: "
    f"{top_scoring_symmetry_rmsd:.4f} Å"
)

print(
    f"Best sampled symmetry-corrected RMSD: "
    f"{best_symmetry_rmsd:.4f} Å"
)

print(
    f"Ranking success ≤ 2.0 Å: "
    f"{ranking_success_2_0}"
)

print(
    f"Sampling success ≤ 2.0 Å: "
    f"{sampling_success_2_0}"
)

print(
    f"Interpretation: {validation_interpretation}"
)


print("\nΑρχεία που δημιουργήθηκαν:")

created_files = [
    DOCKING_BOX_OUTPUT,
    REDOCKING_COMMAND_FILE,
    REDOCKING_LOG,
    REDOCKED_PDBQT,
    POSE_RMSD_OUTPUT,
    VALIDATION_SUMMARY_OUTPUT,
    ALL_POSES_SDF,
    TOP_SCORING_POSE_SDF,
    BEST_RMSD_POSE_SDF
]

for file_path in created_files:

    if file_path.exists():

        print(
            f"- {file_path} "
            f"({file_path.stat().st_size / 1024:.1f} KB)"
        )


print(
    "\nΔεν πραγματοποιήθηκε ακόμη docking "
    "των 20 generated candidates."
)

In [ ]:
# ============================================================
# ΒΗΜΑ 7C.5:
# Οπτικός έλεγχος του redocking του RAJ
#
# Δημιουργούνται δύο εικόνες:
# 1. Crystallographic RAJ vs top-scoring pose
# 2. Crystallographic RAJ vs best-RMSD pose
#
# Χρώματα:
# - Πράσινο: crystallographic reference RAJ
# - Magenta: top-scoring pose
# - Cyan: best-RMSD pose
# - Γκρι: residues του binding pocket
#
# Δεν πραγματοποιείται νέο docking ή αλλαγή παραμέτρων.
# ============================================================

import os
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display, Image


# ============================================================
# 1. Βασικές διαδρομές
# ============================================================

PROJECT_DIRECTORY = Path(
    "/home/afroditi/ptixiaki"
)

os.chdir(
    PROJECT_DIRECTORY
)

RECEPTOR_PDB = Path(
    "3BE2_RECEPTOR_MEEKO_PREPARED.pdb"
)

REFERENCE_LIGAND_SDF = Path(
    "3BE2_RAJ_REFERENCE_CRYSTAL_HEAVY.sdf"
)

TOP_SCORING_POSE_SDF = Path(
    "3BE2_RAJ_TOP_SCORING_POSE.sdf"
)

BEST_RMSD_POSE_SDF = Path(
    "3BE2_RAJ_BEST_RMSD_POSE.sdf"
)

POSE_RESULTS_CSV = Path(
    "3BE2_RAJ_REDOCKING_POSE_RMSD.csv"
)

VALIDATION_SUMMARY_CSV = Path(
    "3BE2_RAJ_REDOCKING_VALIDATION_SUMMARY.csv"
)

TOP_OVERLAY_SCRIPT = Path(
    "3BE2_RAJ_TOP_SCORING_OVERLAY.pml"
)

BEST_OVERLAY_SCRIPT = Path(
    "3BE2_RAJ_BEST_RMSD_OVERLAY.pml"
)

TOP_OVERLAY_PNG = Path(
    "3BE2_RAJ_TOP_SCORING_OVERLAY.png"
)

BEST_OVERLAY_PNG = Path(
    "3BE2_RAJ_BEST_RMSD_OVERLAY.png"
)

VISUAL_VALIDATION_SUMMARY = Path(
    "3BE2_RAJ_REDOCKING_VISUAL_VALIDATION_SUMMARY.csv"
)

VISUALIZATION_LOG = Path(
    "3BE2_RAJ_REDOCKING_VISUALIZATION_LOG.txt"
)


# ============================================================
# 2. Έλεγχος input
# ============================================================

required_files = [
    RECEPTOR_PDB,
    REFERENCE_LIGAND_SDF,
    TOP_SCORING_POSE_SDF,
    BEST_RMSD_POSE_SDF,
    POSE_RESULTS_CSV,
    VALIDATION_SUMMARY_CSV
]

missing_files = [
    str(file_path)
    for file_path in required_files
    if not file_path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Λείπουν τα ακόλουθα αρχεία:\n"
        f"{missing_files}"
    )


pymol_path = shutil.which(
    "pymol"
)

if pymol_path is None:
    raise RuntimeError(
        "Δεν βρέθηκε το PyMOL στο ενεργό docking environment."
    )


# ============================================================
# 3. Ανάγνωση των αριθμητικών αποτελεσμάτων
# ============================================================

pose_results_df = pd.read_csv(
    POSE_RESULTS_CSV
)

validation_summary_df = pd.read_csv(
    VALIDATION_SUMMARY_CSV
)

summary_values = dict(
    zip(
        validation_summary_df["Parameter"],
        validation_summary_df["Value"]
    )
)


top_pose_id = int(
    float(
        summary_values["Top-scoring pose ID"]
    )
)

best_rmsd_pose_id = int(
    float(
        summary_values["Best-RMSD pose ID"]
    )
)


top_pose_row = pose_results_df[
    pose_results_df["Pose_ID"] == top_pose_id
].iloc[0]

best_rmsd_pose_row = pose_results_df[
    pose_results_df["Pose_ID"] == best_rmsd_pose_id
].iloc[0]


top_pose_score = float(
    top_pose_row[
        "SMINA_Affinity_kcal_mol"
    ]
)

best_rmsd_pose_score = float(
    best_rmsd_pose_row[
        "SMINA_Affinity_kcal_mol"
    ]
)

score_difference = float(
    abs(
        top_pose_score
        - best_rmsd_pose_score
    )
)


# ============================================================
# 4. Συνάρτηση δημιουργίας PyMOL script
# ============================================================

def create_overlay_script(
    docked_pose_path,
    output_png_path,
    docked_color,
    docked_object_name,
    title_description
):
    """
    Δημιουργεί PyMOL script για overlay του crystallographic
    ligand με μία redocked pose.
    """

    return f"""
reinitialize

load {RECEPTOR_PDB.resolve().as_posix()}, receptor
load {REFERENCE_LIGAND_SDF.resolve().as_posix()}, reference_raj
load {docked_pose_path.resolve().as_posix()}, {docked_object_name}

hide everything

show cartoon, receptor
color gray80, receptor

select binding_pocket, byres (
    receptor within 5.0 of reference_raj
)

show sticks, binding_pocket
color gray60, binding_pocket

show sticks, reference_raj
color green, reference_raj

show sticks, {docked_object_name}
color {docked_color}, {docked_object_name}

set stick_radius, 0.22
set cartoon_transparency, 0.70
set transparency, 0.15, binding_pocket

set bg_rgb, [1.0, 1.0, 1.0]
set ray_opaque_background, off
set orthoscopic, on
set antialias, 2
set ray_trace_mode, 1
set ambient, 0.45
set specular, 0.25
set depth_cue, 0

orient reference_raj or {docked_object_name}
zoom reference_raj or {docked_object_name}, 7

ray 1800, 1400
png {output_png_path.resolve().as_posix()}, dpi=300

quit
""".strip()


# ============================================================
# 5. Overlay της top-scoring pose
# ============================================================

top_script_text = create_overlay_script(
    docked_pose_path=TOP_SCORING_POSE_SDF,
    output_png_path=TOP_OVERLAY_PNG,
    docked_color="magenta",
    docked_object_name="top_scoring_pose",
    title_description="Top-scoring redocked pose"
)

TOP_OVERLAY_SCRIPT.write_text(
    top_script_text + "\n",
    encoding="utf-8"
)


top_process = subprocess.run(
    [
        pymol_path,
        "-cq",
        str(TOP_OVERLAY_SCRIPT.resolve())
    ],
    capture_output=True,
    text=True,
    cwd=PROJECT_DIRECTORY,
    timeout=600
)


# ============================================================
# 6. Overlay της best-RMSD pose
# ============================================================

best_script_text = create_overlay_script(
    docked_pose_path=BEST_RMSD_POSE_SDF,
    output_png_path=BEST_OVERLAY_PNG,
    docked_color="cyan",
    docked_object_name="best_rmsd_pose",
    title_description="Best-RMSD redocked pose"
)

BEST_OVERLAY_SCRIPT.write_text(
    best_script_text + "\n",
    encoding="utf-8"
)


best_process = subprocess.run(
    [
        pymol_path,
        "-cq",
        str(BEST_OVERLAY_SCRIPT.resolve())
    ],
    capture_output=True,
    text=True,
    cwd=PROJECT_DIRECTORY,
    timeout=600
)


# ============================================================
# 7. Αποθήκευση log
# ============================================================

visualization_log_text = (
    "TOP-SCORING OVERLAY COMMAND\n"
    + " ".join(top_process.args)
    + "\n\nTOP-SCORING STDOUT\n"
    + top_process.stdout
    + "\n\nTOP-SCORING STDERR\n"
    + top_process.stderr
    + "\n\n"
    + "=" * 90
    + "\n\nBEST-RMSD OVERLAY COMMAND\n"
    + " ".join(best_process.args)
    + "\n\nBEST-RMSD STDOUT\n"
    + best_process.stdout
    + "\n\nBEST-RMSD STDERR\n"
    + best_process.stderr
)

VISUALIZATION_LOG.write_text(
    visualization_log_text,
    encoding="utf-8"
)


if top_process.returncode != 0:
    raise RuntimeError(
        "Απέτυχε η δημιουργία του top-scoring overlay.\n"
        f"Δες το log: {VISUALIZATION_LOG}"
    )

if best_process.returncode != 0:
    raise RuntimeError(
        "Απέτυχε η δημιουργία του best-RMSD overlay.\n"
        f"Δες το log: {VISUALIZATION_LOG}"
    )


for image_path in [
    TOP_OVERLAY_PNG,
    BEST_OVERLAY_PNG
]:

    if (
        not image_path.exists()
        or image_path.stat().st_size == 0
    ):
        raise RuntimeError(
            f"Δεν δημιουργήθηκε έγκυρη εικόνα: {image_path}"
        )


# ============================================================
# 8. Δημιουργία συνοπτικού πίνακα
# ============================================================

visual_summary_df = pd.DataFrame([
    {
        "Pose_Type": "Top-scoring pose",
        "Pose_ID": top_pose_id,
        "SMINA_Affinity_kcal_mol": top_pose_score,
        "Symmetry_Corrected_RMSD_Angstrom": float(
            top_pose_row[
                "Symmetry_Corrected_Heavy_Atom_RMSD_Angstrom"
            ]
        ),
        "Within_2_Angstrom": bool(
            float(
                top_pose_row[
                    "Symmetry_Corrected_Heavy_Atom_RMSD_Angstrom"
                ]
            ) <= 2.0
        ),
        "Image_File": str(
            TOP_OVERLAY_PNG
        )
    },
    {
        "Pose_Type": "Best-RMSD pose",
        "Pose_ID": best_rmsd_pose_id,
        "SMINA_Affinity_kcal_mol": best_rmsd_pose_score,
        "Symmetry_Corrected_RMSD_Angstrom": float(
            best_rmsd_pose_row[
                "Symmetry_Corrected_Heavy_Atom_RMSD_Angstrom"
            ]
        ),
        "Within_2_Angstrom": bool(
            float(
                best_rmsd_pose_row[
                    "Symmetry_Corrected_Heavy_Atom_RMSD_Angstrom"
                ]
            ) <= 2.0
        ),
        "Image_File": str(
            BEST_OVERLAY_PNG
        )
    }
])


visual_summary_df["Score_Difference_Between_Two_Poses"] = (
    score_difference
)

visual_summary_df.to_csv(
    VISUAL_VALIDATION_SUMMARY,
    index=False
)


# ============================================================
# 9. Εμφάνιση αποτελεσμάτων
# ============================================================

print("=" * 95)
print("3BE2 RAJ REDOCKING VISUAL VALIDATION")
print("=" * 95)

display(
    visual_summary_df
)


print("\nTOP-SCORING POSE OVERLAY")
print(
    "Πράσινο: crystallographic RAJ | "
    "Magenta: top-scoring pose"
)

display(
    Image(
        filename=str(TOP_OVERLAY_PNG)
    )
)


print("\nBEST-RMSD POSE OVERLAY")
print(
    "Πράσινο: crystallographic RAJ | "
    "Cyan: best-RMSD pose"
)

display(
    Image(
        filename=str(BEST_OVERLAY_PNG)
    )
)


# ============================================================
# 10. Τελική αναφορά
# ============================================================

print("\n" + "-" * 95)
print("Ο ΟΠΤΙΚΟΣ ΕΛΕΓΧΟΣ ΤΟΥ REDOCKING ΟΛΟΚΛΗΡΩΘΗΚΕ")
print("-" * 95)

print(
    f"Top-scoring pose RMSD: "
    f"{float(top_pose_row['Symmetry_Corrected_Heavy_Atom_RMSD_Angstrom']):.4f} Å"
)

print(
    f"Best-RMSD pose RMSD: "
    f"{float(best_rmsd_pose_row['Symmetry_Corrected_Heavy_Atom_RMSD_Angstrom']):.4f} Å"
)

print(
    f"Score difference: "
    f"{score_difference:.4f} kcal/mol"
)

print(
    "\nΔεν τροποποιήθηκε καμία παράμετρος του docking."
)

In [ ]:
# ============================================================
# ΒΗΜΑ 7D.1:
# Προετοιμασία των 20 generated candidates για docking
#
# Διαδικασία:
# 1. Ανάγνωση των 20 επιλεγμένων generated molecules
# 2. Έλεγχος εγκυρότητας και χημικής ταυτότητας των SMILES
# 3. Παραγωγή 3D conformer με OpenBabel
# 4. Προσθήκη explicit hydrogens
# 5. Βελτιστοποίηση με MMFF94
# 6. Παραγωγή ligand PDBQT με Meeko
# 7. Έλεγχος χημικής ταυτότητας και αρχείων
# 8. Καταγραφή SHA256 checksums για αναπαραγωγιμότητα
#
# Δεν πραγματοποιείται ακόμη docking.
#
# Σημαντική προσαρμογή:
# Χρησιμοποιείται η protonation/charge κατάσταση που
# κωδικοποιείται στο αρχικό SMILES. Δεν πραγματοποιείται
# pH-dependent protonation enumeration.
# ============================================================

import os
import re
import json
import hashlib
import shutil
import subprocess
from pathlib import Path

import pandas as pd

from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

from IPython.display import display


# ============================================================
# 1. Βασικές διαδρομές
# ============================================================

PROJECT_DIRECTORY = Path(
    "/home/afroditi/ptixiaki"
)

os.chdir(
    PROJECT_DIRECTORY
)

INPUT_CANDIDATES_CSV = Path(
    "SELECTED_20_GENERATED_CANDIDATES.csv"
)

OUTPUT_DIRECTORY = Path(
    "SELECTED_20_DOCKING_INPUTS"
)

PREPARATION_AUDIT_OUTPUT = Path(
    "SELECTED_20_LIGAND_PREPARATION_AUDIT.csv"
)

PREPARATION_METADATA_OUTPUT = Path(
    "SELECTED_20_LIGAND_PREPARATION_METADATA.csv"
)

COMBINED_PREPARED_SDF = Path(
    "SELECTED_20_GENERATED_CANDIDATES_3D.sdf"
)


# ============================================================
# 2. Σταθερές προετοιμασίας
# ============================================================

EXPECTED_CANDIDATE_COUNT = 20

OPENBABEL_FORCE_FIELD = "MMFF94"

OPENBABEL_MINIMIZATION_STEPS = 500


# ============================================================
# 3. Έλεγχος εργαλείων και input
# ============================================================

if not INPUT_CANDIDATES_CSV.exists():

    raise FileNotFoundError(
        f"Δεν βρέθηκε το αρχείο: {INPUT_CANDIDATES_CSV}"
    )


required_commands = [
    "obabel",
    "mk_prepare_ligand.py"
]


missing_commands = [
    command
    for command in required_commands
    if shutil.which(command) is None
]


if missing_commands:

    raise RuntimeError(
        "Λείπουν τα ακόλουθα εργαλεία:\n"
        f"{missing_commands}"
    )


OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 4. Βοηθητικές συναρτήσεις
# ============================================================

def calculate_sha256(
    file_path
):
    """
    Υπολογίζει SHA256 checksum αρχείου.
    """

    sha256_hash = hashlib.sha256()

    with open(
        file_path,
        "rb"
    ) as file_handle:

        for block in iter(
            lambda: file_handle.read(1024 * 1024),
            b""
        ):
            sha256_hash.update(
                block
            )

    return sha256_hash.hexdigest()


def canonicalize_smiles(
    smiles
):
    """
    Ελέγχει και κανονικοποιεί ένα SMILES με RDKit.
    """

    molecule = Chem.MolFromSmiles(
        str(smiles)
    )

    if molecule is None:
        return None, None

    canonical_smiles = Chem.MolToSmiles(
        molecule,
        canonical=True,
        isomericSmiles=True
    )

    return molecule, canonical_smiles


def find_smiles_column(
    dataframe
):
    """
    Εντοπίζει με ασφαλή σειρά προτεραιότητας
    τη στήλη που περιέχει τα SMILES.
    """

    candidate_columns = [
        "Canonical_SMILES",
        "canonical_smiles",
        "Canonical_Smiles",
        "SMILES",
        "smiles",
        "Generated_SMILES",
        "generated_smiles"
    ]

    for column_name in candidate_columns:

        if column_name in dataframe.columns:
            return column_name

    raise KeyError(
        "Δεν εντοπίστηκε στήλη SMILES.\n"
        f"Διαθέσιμες στήλες: {list(dataframe.columns)}"
    )


def extract_torsdof(
    pdbqt_path
):
    """
    Εξάγει την τιμή TORSDOF από ligand PDBQT.
    """

    torsdof_values = []

    for line in pdbqt_path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines():

        if line.startswith("TORSDOF"):

            parts = line.split()

            if len(parts) >= 2:

                try:
                    torsdof_values.append(
                        int(parts[1])
                    )
                except ValueError:
                    pass

    if len(torsdof_values) != 1:
        return None

    return torsdof_values[0]


def run_subprocess(
    command,
    timeout=600
):
    """
    Εκτελεί command-line εργαλείο και επιστρέφει
    το CompletedProcess.
    """

    return subprocess.run(
        command,
        capture_output=True,
        text=True,
        cwd=PROJECT_DIRECTORY,
        timeout=timeout
    )


# ============================================================
# 5. Ανάγνωση των 20 candidates
# ============================================================

candidates_df = pd.read_csv(
    INPUT_CANDIDATES_CSV
)


if len(candidates_df) != EXPECTED_CANDIDATE_COUNT:

    raise RuntimeError(
        f"Αναμένονταν {EXPECTED_CANDIDATE_COUNT} candidates, "
        f"αλλά βρέθηκαν {len(candidates_df)}."
    )


smiles_column = find_smiles_column(
    candidates_df
)


print("=" * 95)
print("SELECTED GENERATED CANDIDATES")
print("=" * 95)

print(
    f"Candidates: {len(candidates_df)}"
)

print(
    f"SMILES column: {smiles_column}"
)

print(
    f"Columns: {list(candidates_df.columns)}"
)


# ============================================================
# 6. Προκαταρκτικός έλεγχος SMILES
# ============================================================

canonical_smiles_list = []


for dataframe_index, row in candidates_df.iterrows():

    input_smiles = str(
        row[smiles_column]
    ).strip()

    molecule, canonical_smiles = canonicalize_smiles(
        input_smiles
    )

    if molecule is None:

        raise ValueError(
            f"Μη έγκυρο SMILES στη γραμμή "
            f"{dataframe_index + 1}:\n{input_smiles}"
        )

    canonical_smiles_list.append(
        canonical_smiles
    )


if len(set(canonical_smiles_list)) != EXPECTED_CANDIDATE_COUNT:

    raise RuntimeError(
        "Τα 20 επιλεγμένα candidates δεν είναι "
        "χημικά μοναδικά μετά το canonicalization."
    )


candidates_df[
    "Docking_Canonical_SMILES"
] = canonical_smiles_list


# ============================================================
# 7. Προετοιμασία κάθε candidate
# ============================================================

preparation_rows = []

prepared_molecules = []


for dataframe_index, candidate_row in candidates_df.iterrows():

    selection_order = (
        int(candidate_row["Selection_Order"])
        if (
            "Selection_Order" in candidates_df.columns
            and pd.notna(candidate_row["Selection_Order"])
        )
        else dataframe_index + 1
    )

    candidate_id = (
        f"CAND_{selection_order:02d}"
    )

    input_smiles = str(
        candidate_row[smiles_column]
    ).strip()

    expected_canonical_smiles = str(
        candidate_row["Docking_Canonical_SMILES"]
    )


    candidate_directory = (
        OUTPUT_DIRECTORY / candidate_id
    )

    candidate_directory.mkdir(
        parents=True,
        exist_ok=True
    )


    smiles_file = (
        candidate_directory
        / f"{candidate_id}.smi"
    )

    sdf_file = (
        candidate_directory
        / f"{candidate_id}_3D.sdf"
    )

    pdbqt_file = (
        candidate_directory
        / f"{candidate_id}.pdbqt"
    )

    openbabel_log_file = (
        candidate_directory
        / f"{candidate_id}_OPENBABEL.log"
    )

    meeko_log_file = (
        candidate_directory
        / f"{candidate_id}_MEEKO.log"
    )


    # Αφαίρεση παλαιών outputs του ίδιου candidate.
    for previous_output in [
        sdf_file,
        pdbqt_file,
        openbabel_log_file,
        meeko_log_file
    ]:

        if previous_output.exists():
            previous_output.unlink()


    smiles_file.write_text(
        f"{input_smiles}\t{candidate_id}\n",
        encoding="utf-8"
    )


    preparation_record = {
        "Candidate_ID": candidate_id,
        "Selection_Order": selection_order,
        "Input_SMILES": input_smiles,
        "Expected_Canonical_SMILES":
            expected_canonical_smiles,
        "OpenBabel_Success": False,
        "Chemical_Identity_Preserved": False,
        "Meeko_Success": False,
        "Preparation_Success": False,
        "Error": ""
    }


    try:

        # ====================================================
        # 7A. Παραγωγή 3D conformer με OpenBabel
        # ====================================================

        openbabel_command = [
            "obabel",
            str(smiles_file.resolve()),
            "-O",
            str(sdf_file.resolve()),
            "--gen3d",
            "-h",
            "--minimize",
            "--ff",
            OPENBABEL_FORCE_FIELD,
            "--steps",
            str(
                OPENBABEL_MINIMIZATION_STEPS
            )
        ]


        openbabel_process = run_subprocess(
            openbabel_command,
            timeout=900
        )


        openbabel_log_text = (
            "COMMAND\n"
            + " ".join(openbabel_command)
            + "\n\nSTDOUT\n"
            + openbabel_process.stdout
            + "\n\nSTDERR\n"
            + openbabel_process.stderr
        )


        openbabel_log_file.write_text(
            openbabel_log_text,
            encoding="utf-8"
        )


        if openbabel_process.returncode != 0:

            raise RuntimeError(
                "Το OpenBabel επέστρεψε return code "
                f"{openbabel_process.returncode}."
            )


        if (
            not sdf_file.exists()
            or sdf_file.stat().st_size == 0
        ):

            raise RuntimeError(
                "Δεν δημιουργήθηκε έγκυρο 3D SDF."
            )


        preparation_record[
            "OpenBabel_Success"
        ] = True


        # ====================================================
        # 7B. Έλεγχος του 3D SDF με RDKit
        # ====================================================

        sdf_supplier = Chem.SDMolSupplier(
            str(sdf_file),
            removeHs=False,
            sanitize=True
        )


        sdf_molecules = [
            molecule
            for molecule in sdf_supplier
            if molecule is not None
        ]


        if len(sdf_molecules) != 1:

            raise RuntimeError(
                "Το 3D SDF δεν περιέχει ακριβώς "
                "ένα έγκυρο molecule."
            )


        prepared_molecule = sdf_molecules[0]


        if prepared_molecule.GetNumConformers() != 1:

            raise RuntimeError(
                "Το molecule δεν περιέχει ακριβώς "
                "ένα 3D conformer."
            )


        conformer = prepared_molecule.GetConformer()


        if not conformer.Is3D():

            raise RuntimeError(
                "Το OpenBabel output δεν αναγνωρίζεται "
                "ως τρισδιάστατη δομή."
            )


        prepared_heavy_molecule = Chem.RemoveHs(
            prepared_molecule
        )


        prepared_canonical_smiles = Chem.MolToSmiles(
            prepared_heavy_molecule,
            canonical=True,
            isomericSmiles=True
        )


        chemical_identity_preserved = (
            prepared_canonical_smiles
            == expected_canonical_smiles
        )


        preparation_record[
            "Prepared_Canonical_SMILES"
        ] = prepared_canonical_smiles

        preparation_record[
            "Chemical_Identity_Preserved"
        ] = chemical_identity_preserved


        if not chemical_identity_preserved:

            raise RuntimeError(
                "Η χημική ταυτότητα άλλαξε κατά "
                "την παραγωγή 3D conformer."
            )


        heavy_atom_count = int(
            prepared_molecule.GetNumHeavyAtoms()
        )

        explicit_hydrogen_count = int(
            sum(
                atom.GetAtomicNum() == 1
                for atom in prepared_molecule.GetAtoms()
            )
        )

        total_atom_count = int(
            prepared_molecule.GetNumAtoms()
        )

        formal_charge = int(
            Chem.GetFormalCharge(
                prepared_molecule
            )
        )

        molecular_formula = (
            rdMolDescriptors.CalcMolFormula(
                prepared_molecule
            )
        )


        preparation_record.update({
            "Molecular_Formula": molecular_formula,
            "Formal_Charge": formal_charge,
            "Heavy_Atom_Count": heavy_atom_count,
            "Explicit_Hydrogen_Count":
                explicit_hydrogen_count,
            "Total_Atom_Count": total_atom_count,
            "SDF_File": str(sdf_file),
            "SDF_SHA256": calculate_sha256(
                sdf_file
            )
        })


        # ====================================================
        # 7C. Παραγωγή PDBQT με Meeko
        # ====================================================

        meeko_command = [
            "mk_prepare_ligand.py",
            "-i",
            str(sdf_file.resolve()),
            "-o",
            str(pdbqt_file.resolve())
        ]


        meeko_process = run_subprocess(
            meeko_command,
            timeout=600
        )


        meeko_log_text = (
            "COMMAND\n"
            + " ".join(meeko_command)
            + "\n\nSTDOUT\n"
            + meeko_process.stdout
            + "\n\nSTDERR\n"
            + meeko_process.stderr
        )


        meeko_log_file.write_text(
            meeko_log_text,
            encoding="utf-8"
        )


        if meeko_process.returncode != 0:

            raise RuntimeError(
                "Το Meeko επέστρεψε return code "
                f"{meeko_process.returncode}."
            )


        if (
            not pdbqt_file.exists()
            or pdbqt_file.stat().st_size == 0
        ):

            raise RuntimeError(
                "Δεν δημιουργήθηκε έγκυρο ligand PDBQT."
            )


        pdbqt_lines = pdbqt_file.read_text(
            encoding="utf-8",
            errors="replace"
        ).splitlines()


        pdbqt_atom_lines = [
            line
            for line in pdbqt_lines
            if line.startswith(
                ("ATOM", "HETATM")
            )
        ]


        root_count = sum(
            line.strip() == "ROOT"
            for line in pdbqt_lines
        )


        torsdof = extract_torsdof(
            pdbqt_file
        )


        if not pdbqt_atom_lines:

            raise RuntimeError(
                "Το PDBQT δεν περιέχει atom records."
            )


        if root_count != 1:

            raise RuntimeError(
                f"Αναμενόταν 1 ROOT, βρέθηκαν {root_count}."
            )


        if torsdof is None:

            raise RuntimeError(
                "Δεν βρέθηκε έγκυρη τιμή TORSDOF."
            )


        preparation_record.update({
            "Meeko_Success": True,
            "Preparation_Success": True,
            "PDBQT_Atom_Record_Count":
                len(pdbqt_atom_lines),
            "PDBQT_TORSDOF": torsdof,
            "PDBQT_File": str(pdbqt_file),
            "PDBQT_SHA256": calculate_sha256(
                pdbqt_file
            ),
            "SMILES_File": str(smiles_file),
            "OpenBabel_Log": str(
                openbabel_log_file
            ),
            "Meeko_Log": str(
                meeko_log_file
            )
        })


        # Properties για το ενιαίο SDF.
        prepared_molecule.SetProp(
            "_Name",
            candidate_id
        )

        prepared_molecule.SetProp(
            "Candidate_ID",
            candidate_id
        )

        prepared_molecule.SetProp(
            "Selection_Order",
            str(selection_order)
        )

        prepared_molecule.SetProp(
            "Input_SMILES",
            input_smiles
        )

        prepared_molecule.SetProp(
            "Canonical_SMILES",
            expected_canonical_smiles
        )

        prepared_molecules.append(
            prepared_molecule
        )


        print(
            f"{candidate_id}: OK | "
            f"Heavy atoms={heavy_atom_count} | "
            f"TORSDOF={torsdof}"
        )


    except Exception as error:

        preparation_record[
            "Error"
        ] = (
            f"{type(error).__name__}: {error}"
        )

        print(
            f"{candidate_id}: FAILED | "
            f"{preparation_record['Error']}"
        )


    preparation_rows.append(
        preparation_record
    )


# ============================================================
# 8. Δημιουργία audit
# ============================================================

preparation_audit_df = pd.DataFrame(
    preparation_rows
)


preparation_audit_df.to_csv(
    PREPARATION_AUDIT_OUTPUT,
    index=False
)


# ============================================================
# 9. Αποθήκευση ενιαίου SDF
# ============================================================

if prepared_molecules:

    combined_sdf_writer = Chem.SDWriter(
        str(COMBINED_PREPARED_SDF)
    )

    for prepared_molecule in prepared_molecules:

        combined_sdf_writer.write(
            prepared_molecule
        )

    combined_sdf_writer.close()


# ============================================================
# 10. Metadata της διαδικασίας
# ============================================================

openbabel_version_process = subprocess.run(
    [
        "obabel",
        "-V"
    ],
    capture_output=True,
    text=True
)


openbabel_version = (
    openbabel_version_process.stdout.strip()
    or openbabel_version_process.stderr.strip()
)


metadata_df = pd.DataFrame([
    {
        "Parameter": "Input file",
        "Value": str(INPUT_CANDIDATES_CSV)
    },
    {
        "Parameter": "Candidate count",
        "Value": len(candidates_df)
    },
    {
        "Parameter": "SMILES column",
        "Value": smiles_column
    },
    {
        "Parameter": "3D conformer generator",
        "Value": "OpenBabel --gen3d"
    },
    {
        "Parameter": "OpenBabel version",
        "Value": openbabel_version
    },
    {
        "Parameter": "Hydrogen handling",
        "Value": "Explicit hydrogens added with OpenBabel -h"
    },
    {
        "Parameter": "Geometry force field",
        "Value": OPENBABEL_FORCE_FIELD
    },
    {
        "Parameter": "Minimization steps",
        "Value": OPENBABEL_MINIMIZATION_STEPS
    },
    {
        "Parameter": "Ligand PDBQT preparation",
        "Value": "Meeko mk_prepare_ligand.py"
    },
    {
        "Parameter": "pH protonation enumeration",
        "Value": False
    },
    {
        "Parameter": "Input protonation source",
        "Value": "Original selected SMILES"
    },
    {
        "Parameter": "Successful preparations",
        "Value": int(
            preparation_audit_df[
                "Preparation_Success"
            ].sum()
        )
    },
    {
        "Parameter": "Failed preparations",
        "Value": int(
            (
                ~preparation_audit_df[
                    "Preparation_Success"
                ]
            ).sum()
        )
    },
    {
        "Parameter": "Combined prepared SDF",
        "Value": str(COMBINED_PREPARED_SDF)
    }
])


metadata_df.to_csv(
    PREPARATION_METADATA_OUTPUT,
    index=False
)


# ============================================================
# 11. Εμφάνιση αποτελεσμάτων
# ============================================================

print("\n" + "=" * 95)
print("SELECTED 20 LIGAND PREPARATION AUDIT")
print("=" * 95)

display(
    preparation_audit_df
)


print("\n" + "=" * 95)
print("PREPARATION METADATA")
print("=" * 95)

display(
    metadata_df
)


# ============================================================
# 12. Τελικός έλεγχος
# ============================================================

failed_preparations_df = preparation_audit_df[
    ~preparation_audit_df[
        "Preparation_Success"
    ]
]


print("\n" + "-" * 95)

if failed_preparations_df.empty:

    print(
        "Η ΠΡΟΕΤΟΙΜΑΣΙΑ ΚΑΙ ΤΩΝ 20 CANDIDATES "
        "ΟΛΟΚΛΗΡΩΘΗΚΕ"
    )

    print(
        f"Prepared molecules: "
        f"{len(prepared_molecules)}/{EXPECTED_CANDIDATE_COUNT}"
    )

    print(
        f"Output directory: {OUTPUT_DIRECTORY}"
    )

    print(
        f"Combined SDF: {COMBINED_PREPARED_SDF}"
    )

else:

    print(
        "Η ΠΡΟΕΤΟΙΜΑΣΙΑ ΔΕΝ ΟΛΟΚΛΗΡΩΘΗΚΕ "
        "ΓΙΑ ΟΛΑ ΤΑ CANDIDATES"
    )

    display(
        failed_preparations_df[
            [
                "Candidate_ID",
                "Input_SMILES",
                "Error"
            ]
        ]
    )

    raise RuntimeError(
        f"Απέτυχαν "
        f"{len(failed_preparations_df)} candidates. "
        "Δεν πρέπει να προχωρήσουμε σε docking "
        "πριν ελεγχθούν."
    )


print("-" * 95)

print(
    "\nΔεν πραγματοποιήθηκε ακόμη docking των candidates."
)

In [ ]:
# ============================================================
# ΔΙΑΓΝΩΣΤΙΚΟΣ ΕΛΕΓΧΟΣ ΤΩΝ 3 CANDIDATES ΠΟΥ ΑΠΕΤΥΧΑΝ
#
# Το κελί:
# - διαβάζει το υπάρχον preparation audit,
# - εμφανίζει μόνο τα αποτυχημένα candidates,
# - εμφανίζει το τέλος των OpenBabel και Meeko logs,
# - δεν τροποποιεί και δεν επαναλαμβάνει κανέναν υπολογισμό.
# ============================================================

from pathlib import Path

import pandas as pd
from IPython.display import display


PROJECT_DIRECTORY = Path(
    "/home/afroditi/ptixiaki"
)

AUDIT_FILE = (
    PROJECT_DIRECTORY
    / "SELECTED_20_LIGAND_PREPARATION_AUDIT.csv"
)

OUTPUT_DIRECTORY = (
    PROJECT_DIRECTORY
    / "SELECTED_20_DOCKING_INPUTS"
)

DIAGNOSTIC_OUTPUT = (
    PROJECT_DIRECTORY
    / "SELECTED_20_FAILED_PREPARATION_DIAGNOSTIC.txt"
)


# ============================================================
# 1. Έλεγχος ότι υπάρχει το audit
# ============================================================

if not AUDIT_FILE.exists():
    raise FileNotFoundError(
        f"Δεν βρέθηκε το αρχείο: {AUDIT_FILE}"
    )


audit_df = pd.read_csv(
    AUDIT_FILE
)


# ============================================================
# 2. Ασφαλής μετατροπή της Preparation_Success σε boolean
# ============================================================

def convert_to_boolean(value):
    """
    Μετατρέπει True/False ή αντίστοιχες συμβολοσειρές
    σε πραγματικές boolean τιμές.
    """

    if isinstance(value, bool):
        return value

    return str(value).strip().lower() in {
        "true",
        "1",
        "yes"
    }


audit_df["Preparation_Success_Boolean"] = (
    audit_df["Preparation_Success"].apply(
        convert_to_boolean
    )
)


failed_df = audit_df[
    ~audit_df["Preparation_Success_Boolean"]
].copy()


print("=" * 100)
print("FAILED CANDIDATE PREPARATIONS")
print("=" * 100)

print(
    f"Αποτυχημένα candidates: {len(failed_df)}"
)


if failed_df.empty:
    print(
        "Δεν βρέθηκαν αποτυχημένες προετοιμασίες."
    )

else:

    preferred_columns = [
        "Candidate_ID",
        "Selection_Order",
        "Input_SMILES",
        "Expected_Canonical_SMILES",
        "Prepared_Canonical_SMILES",
        "OpenBabel_Success",
        "Chemical_Identity_Preserved",
        "Meeko_Success",
        "Preparation_Success",
        "Error"
    ]

    existing_columns = [
        column
        for column in preferred_columns
        if column in failed_df.columns
    ]

    pd.set_option(
        "display.max_colwidth",
        None
    )

    display(
        failed_df[
            existing_columns
        ]
    )


# ============================================================
# 3. Συνάρτηση ανάγνωσης του τέλους ενός log
# ============================================================

def read_log_tail(
    log_path,
    maximum_lines=40
):
    """
    Επιστρέφει τις τελευταίες γραμμές ενός log.
    """

    if not log_path.exists():
        return (
            f"Το log δεν δημιουργήθηκε: {log_path}"
        )

    lines = log_path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    return "\n".join(
        lines[-maximum_lines:]
    )


# ============================================================
# 4. Εμφάνιση των OpenBabel και Meeko logs
# ============================================================

diagnostic_sections = []


for _, candidate_row in failed_df.iterrows():

    candidate_id = str(
        candidate_row["Candidate_ID"]
    )

    candidate_directory = (
        OUTPUT_DIRECTORY
        / candidate_id
    )

    openbabel_log = (
        candidate_directory
        / f"{candidate_id}_OPENBABEL.log"
    )

    meeko_log = (
        candidate_directory
        / f"{candidate_id}_MEEKO.log"
    )

    error_text = str(
        candidate_row.get(
            "Error",
            ""
        )
    )

    section_header = (
        "\n"
        + "=" * 100
        + f"\n{candidate_id}\n"
        + "=" * 100
    )

    openbabel_text = (
        "\n\nOPENBABEL LOG — ΤΕΛΕΥΤΑΙΕΣ ΓΡΑΜΜΕΣ\n"
        + "-" * 100
        + "\n"
        + read_log_tail(
            openbabel_log
        )
    )

    meeko_text = (
        "\n\nMEEKO LOG — ΤΕΛΕΥΤΑΙΕΣ ΓΡΑΜΜΕΣ\n"
        + "-" * 100
        + "\n"
        + read_log_tail(
            meeko_log
        )
    )

    candidate_diagnostic = (
        section_header
        + "\n\nERROR FROM AUDIT\n"
        + "-" * 100
        + "\n"
        + error_text
        + openbabel_text
        + meeko_text
    )

    diagnostic_sections.append(
        candidate_diagnostic
    )

    print(
        candidate_diagnostic
    )


# ============================================================
# 5. Αποθήκευση του συνολικού diagnostic
# ============================================================

diagnostic_text = (
    "FAILED LIGAND PREPARATION DIAGNOSTIC\n"
    + "\n".join(
        diagnostic_sections
    )
)


DIAGNOSTIC_OUTPUT.write_text(
    diagnostic_text,
    encoding="utf-8"
)


print("\n" + "=" * 100)
print("Ο ΔΙΑΓΝΩΣΤΙΚΟΣ ΕΛΕΓΧΟΣ ΟΛΟΚΛΗΡΩΘΗΚΕ")
print("=" * 100)

print(
    f"Αποθηκεύτηκε στο:\n{DIAGNOSTIC_OUTPUT}"
)

In [ ]:
# ============================================================
# ΒΗΜΑ 7D.1B:
# Stereochemistry-aware έλεγχος και ολοκλήρωση των
# CAND_07, CAND_11 και CAND_15
#
# Αιτία προηγούμενης διακοπής:
# Τα αρχικά SMILES είχαν ακαθόριστη στερεοχημεία.
# Το OpenBabel, κατά την παραγωγή 3D συντεταγμένων,
# ανέθεσε συγκεκριμένες διατάξεις @/@@.
#
# Το κελί:
# 1. Δεν επαναλαμβάνει το OpenBabel 3D generation
# 2. Ελέγχει non-isomeric molecular identity
# 3. Ελέγχει formula, charge και heavy atoms
# 4. Επιβεβαιώνει ότι το input δεν είχε assigned stereo
# 5. Εκτελεί Meeko μόνο για τα 3 αποτυχημένα candidates
# 6. Ανανεώνει τα audits και το combined SDF
#
# Δεν πραγματοποιείται docking.
# ============================================================

import os
import hashlib
import shutil
import subprocess
from pathlib import Path

import pandas as pd

from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

from IPython.display import display


# ============================================================
# 1. Βασικές διαδρομές
# ============================================================

PROJECT_DIRECTORY = Path(
    "/home/afroditi/ptixiaki"
)

os.chdir(
    PROJECT_DIRECTORY
)

PREPARATION_AUDIT_FILE = Path(
    "SELECTED_20_LIGAND_PREPARATION_AUDIT.csv"
)

PREPARATION_METADATA_FILE = Path(
    "SELECTED_20_LIGAND_PREPARATION_METADATA.csv"
)

OUTPUT_DIRECTORY = Path(
    "SELECTED_20_DOCKING_INPUTS"
)

COMBINED_PREPARED_SDF = Path(
    "SELECTED_20_GENERATED_CANDIDATES_3D.sdf"
)

STEREOCHEMISTRY_AUDIT_FILE = Path(
    "SELECTED_20_STEREOCHEMISTRY_AUDIT.csv"
)

STEREOCHEMISTRY_METADATA_FILE = Path(
    "SELECTED_20_STEREOCHEMISTRY_METADATA.csv"
)


# ============================================================
# 2. Έλεγχος input και εργαλείων
# ============================================================

if not PREPARATION_AUDIT_FILE.exists():
    raise FileNotFoundError(
        f"Δεν βρέθηκε το audit: {PREPARATION_AUDIT_FILE}"
    )

if not OUTPUT_DIRECTORY.exists():
    raise FileNotFoundError(
        f"Δεν βρέθηκε ο φάκελος: {OUTPUT_DIRECTORY}"
    )

if shutil.which("mk_prepare_ligand.py") is None:
    raise RuntimeError(
        "Δεν βρέθηκε το mk_prepare_ligand.py "
        "στο ενεργό environment."
    )


# ============================================================
# 3. Βοηθητικές συναρτήσεις
# ============================================================

def convert_to_boolean(value):
    """
    Μετατρέπει τιμές CSV σε πραγματικό boolean.
    """

    if isinstance(value, bool):
        return value

    return str(value).strip().lower() in {
        "true",
        "1",
        "yes"
    }


def calculate_sha256(file_path):
    """
    Υπολογίζει SHA256 checksum αρχείου.
    """

    sha256_hash = hashlib.sha256()

    with open(file_path, "rb") as file_handle:

        for block in iter(
            lambda: file_handle.read(1024 * 1024),
            b""
        ):
            sha256_hash.update(block)

    return sha256_hash.hexdigest()


def canonical_smiles(
    molecule,
    isomeric
):
    """
    Δημιουργεί canonical SMILES με ή χωρίς stereochemistry.
    """

    return Chem.MolToSmiles(
        molecule,
        canonical=True,
        isomericSmiles=isomeric
    )


def get_unassigned_and_assigned_centers(molecule):
    """
    Επιστρέφει τα assigned και unassigned tetrahedral
    stereocenters ενός molecule.
    """

    molecule_copy = Chem.Mol(molecule)

    Chem.AssignStereochemistry(
        molecule_copy,
        cleanIt=True,
        force=True
    )

    centers = Chem.FindMolChiralCenters(
        molecule_copy,
        force=True,
        includeUnassigned=True,
        includeCIP=True,
        useLegacyImplementation=False
    )

    assigned_centers = [
        {
            "Atom_Index": int(atom_index),
            "Assignment": str(assignment)
        }
        for atom_index, assignment in centers
        if assignment != "?"
    ]

    unassigned_centers = [
        {
            "Atom_Index": int(atom_index),
            "Assignment": "?"
        }
        for atom_index, assignment in centers
        if assignment == "?"
    ]

    return (
        assigned_centers,
        unassigned_centers
    )


def get_assigned_double_bond_stereo_count(molecule):
    """
    Μετρά double bonds με ρητά καθορισμένη E/Z
    στερεοχημεία.
    """

    assigned_stereo_values = {
        Chem.BondStereo.STEREOE,
        Chem.BondStereo.STEREOZ,
        Chem.BondStereo.STEREOCIS,
        Chem.BondStereo.STEREOTRANS
    }

    return int(
        sum(
            bond.GetStereo()
            in assigned_stereo_values
            for bond in molecule.GetBonds()
        )
    )


def extract_torsdof(pdbqt_path):
    """
    Εξάγει το μοναδικό TORSDOF από PDBQT.
    """

    torsdof_values = []

    for line in pdbqt_path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines():

        if not line.startswith("TORSDOF"):
            continue

        parts = line.split()

        if len(parts) < 2:
            continue

        try:
            torsdof_values.append(
                int(parts[1])
            )
        except ValueError:
            pass

    if len(torsdof_values) != 1:
        return None

    return torsdof_values[0]


def load_single_sdf_molecule(sdf_path):
    """
    Διαβάζει ακριβώς ένα έγκυρο molecule από SDF.
    """

    supplier = Chem.SDMolSupplier(
        str(sdf_path),
        removeHs=False,
        sanitize=True
    )

    molecules = [
        molecule
        for molecule in supplier
        if molecule is not None
    ]

    if len(molecules) != 1:
        raise RuntimeError(
            f"Το {sdf_path} δεν περιέχει ακριβώς "
            "ένα έγκυρο molecule."
        )

    return molecules[0]


# ============================================================
# 4. Ανάγνωση του προηγούμενου audit
# ============================================================

audit_df = pd.read_csv(
    PREPARATION_AUDIT_FILE
)

audit_df["Preparation_Success_Boolean"] = (
    audit_df["Preparation_Success"].apply(
        convert_to_boolean
    )
)

failed_indices = audit_df.index[
    ~audit_df["Preparation_Success_Boolean"]
].tolist()


print("=" * 100)
print("STEREOCHEMISTRY-AWARE PREPARATION REPAIR")
print("=" * 100)

print(
    f"Candidates προς έλεγχο: "
    f"{len(failed_indices)}"
)

print(
    audit_df.loc[
        failed_indices,
        "Candidate_ID"
    ].tolist()
)


if len(failed_indices) != 3:
    raise RuntimeError(
        "Αναμένονταν ακριβώς 3 αποτυχημένα candidates, "
        f"αλλά βρέθηκαν {len(failed_indices)}."
    )


# ============================================================
# 5. Δημιουργία απαραίτητων νέων στηλών
# ============================================================

new_columns_with_defaults = {
    "Input_NonIsomeric_Canonical_SMILES": "",
    "Prepared_NonIsomeric_Canonical_SMILES": "",
    "Constitutional_Identity_Preserved": False,
    "Molecular_Formula_Preserved": False,
    "Formal_Charge_Preserved": False,
    "Heavy_Atom_Count_Preserved": False,
    "Stereo_Only_Difference": False,
    "Input_Assigned_Stereocenter_Count": 0,
    "Input_Unassigned_Stereocenter_Count": 0,
    "Prepared_Assigned_Stereocenter_Count": 0,
    "Input_Assigned_Double_Bond_Stereo_Count": 0,
    "Prepared_Assigned_Double_Bond_Stereo_Count": 0,
    "Stereochemistry_Status": "",
    "Identity_Acceptance_Basis": ""
}


for column_name, default_value in (
    new_columns_with_defaults.items()
):

    if column_name not in audit_df.columns:
        audit_df[column_name] = default_value


# ============================================================
# 6. Επεξεργασία μόνο των 3 αποτυχημένων candidates
# ============================================================

stereo_rows = []


for dataframe_index in failed_indices:

    candidate_row = audit_df.loc[
        dataframe_index
    ]

    candidate_id = str(
        candidate_row["Candidate_ID"]
    )

    input_smiles = str(
        candidate_row["Input_SMILES"]
    ).strip()

    candidate_directory = (
        OUTPUT_DIRECTORY
        / candidate_id
    )

    sdf_file = (
        candidate_directory
        / f"{candidate_id}_3D.sdf"
    )

    pdbqt_file = (
        candidate_directory
        / f"{candidate_id}.pdbqt"
    )

    meeko_log_file = (
        candidate_directory
        / f"{candidate_id}_MEEKO.log"
    )


    print("\n" + "-" * 100)
    print(f"Έλεγχος {candidate_id}")
    print("-" * 100)


    if not sdf_file.exists():

        raise FileNotFoundError(
            f"Δεν βρέθηκε το υπάρχον 3D SDF: {sdf_file}"
        )


    # ========================================================
    # 6A. Ανάγνωση input SMILES και prepared SDF
    # ========================================================

    input_molecule = Chem.MolFromSmiles(
        input_smiles
    )

    if input_molecule is None:
        raise RuntimeError(
            f"Το input SMILES του {candidate_id} "
            "δεν διαβάζεται από το RDKit."
        )


    prepared_molecule = load_single_sdf_molecule(
        sdf_file
    )

    prepared_heavy_molecule = Chem.RemoveHs(
        prepared_molecule
    )


    # ========================================================
    # 6B. Isomeric και non-isomeric identity
    # ========================================================

    input_isomeric_smiles = canonical_smiles(
        input_molecule,
        isomeric=True
    )

    prepared_isomeric_smiles = canonical_smiles(
        prepared_heavy_molecule,
        isomeric=True
    )

    input_nonisomeric_smiles = canonical_smiles(
        input_molecule,
        isomeric=False
    )

    prepared_nonisomeric_smiles = canonical_smiles(
        prepared_heavy_molecule,
        isomeric=False
    )


    constitutional_identity_preserved = (
        input_nonisomeric_smiles
        == prepared_nonisomeric_smiles
    )


    # ========================================================
    # 6C. Formula, charge και atom-count identity
    # ========================================================

    input_formula = (
        rdMolDescriptors.CalcMolFormula(
            input_molecule
        )
    )

    prepared_formula = (
        rdMolDescriptors.CalcMolFormula(
            prepared_molecule
        )
    )

    molecular_formula_preserved = (
        input_formula == prepared_formula
    )


    input_formal_charge = int(
        Chem.GetFormalCharge(
            input_molecule
        )
    )

    prepared_formal_charge = int(
        Chem.GetFormalCharge(
            prepared_molecule
        )
    )

    formal_charge_preserved = (
        input_formal_charge
        == prepared_formal_charge
    )


    input_heavy_atom_count = int(
        input_molecule.GetNumHeavyAtoms()
    )

    prepared_heavy_atom_count = int(
        prepared_molecule.GetNumHeavyAtoms()
    )

    heavy_atom_count_preserved = (
        input_heavy_atom_count
        == prepared_heavy_atom_count
    )


    # ========================================================
    # 6D. Stereochemistry audit
    # ========================================================

    (
        input_assigned_centers,
        input_unassigned_centers
    ) = get_unassigned_and_assigned_centers(
        input_molecule
    )

    (
        prepared_assigned_centers,
        prepared_unassigned_centers
    ) = get_unassigned_and_assigned_centers(
        prepared_heavy_molecule
    )


    input_assigned_double_bonds = (
        get_assigned_double_bond_stereo_count(
            input_molecule
        )
    )

    prepared_assigned_double_bonds = (
        get_assigned_double_bond_stereo_count(
            prepared_heavy_molecule
        )
    )


    input_has_assigned_stereo = (
        len(input_assigned_centers) > 0
        or input_assigned_double_bonds > 0
    )


    isomeric_smiles_differ = (
        input_isomeric_smiles
        != prepared_isomeric_smiles
    )


    stereo_only_difference = (
        constitutional_identity_preserved
        and molecular_formula_preserved
        and formal_charge_preserved
        and heavy_atom_count_preserved
        and isomeric_smiles_differ
        and not input_has_assigned_stereo
    )


    # ========================================================
    # 6E. Αυστηρός τελικός έλεγχος
    # ========================================================

    if not constitutional_identity_preserved:

        raise RuntimeError(
            f"{candidate_id}: Η molecular connectivity "
            "άλλαξε πραγματικά.\n"
            f"Input non-isomeric: {input_nonisomeric_smiles}\n"
            f"Prepared non-isomeric: "
            f"{prepared_nonisomeric_smiles}"
        )


    if not molecular_formula_preserved:

        raise RuntimeError(
            f"{candidate_id}: Ο μοριακός τύπος άλλαξε.\n"
            f"Input: {input_formula}\n"
            f"Prepared: {prepared_formula}"
        )


    if not formal_charge_preserved:

        raise RuntimeError(
            f"{candidate_id}: Το formal charge άλλαξε.\n"
            f"Input: {input_formal_charge}\n"
            f"Prepared: {prepared_formal_charge}"
        )


    if not heavy_atom_count_preserved:

        raise RuntimeError(
            f"{candidate_id}: Ο αριθμός heavy atoms άλλαξε."
        )


    if input_has_assigned_stereo:

        raise RuntimeError(
            f"{candidate_id}: Το input είχε ήδη καθορισμένη "
            "στερεοχημεία. Δεν επιτρέπεται αυτόματη "
            "αντικατάστασή της."
        )


    if not stereo_only_difference:

        raise RuntimeError(
            f"{candidate_id}: Η διαφορά δεν μπορεί να "
            "χαρακτηριστεί με ασφάλεια ως stereo-only."
        )


    print(
        "Constitutional identity: OK"
    )

    print(
        f"Formula: {input_formula} — OK"
    )

    print(
        f"Formal charge: {input_formal_charge} — OK"
    )

    print(
        f"Input unassigned stereocenters: "
        f"{len(input_unassigned_centers)}"
    )

    print(
        f"Prepared assigned stereocenters: "
        f"{len(prepared_assigned_centers)}"
    )


    # ========================================================
    # 6F. Παραγωγή PDBQT με Meeko
    # ========================================================

    if pdbqt_file.exists():
        pdbqt_file.unlink()

    if meeko_log_file.exists():
        meeko_log_file.unlink()


    meeko_command = [
        "mk_prepare_ligand.py",
        "-i",
        str(sdf_file.resolve()),
        "-o",
        str(pdbqt_file.resolve())
    ]


    meeko_process = subprocess.run(
        meeko_command,
        capture_output=True,
        text=True,
        cwd=PROJECT_DIRECTORY,
        timeout=600
    )


    meeko_log_text = (
        "COMMAND\n"
        + " ".join(meeko_command)
        + "\n\nSTDOUT\n"
        + meeko_process.stdout
        + "\n\nSTDERR\n"
        + meeko_process.stderr
    )


    meeko_log_file.write_text(
        meeko_log_text,
        encoding="utf-8"
    )


    if meeko_process.returncode != 0:

        raise RuntimeError(
            f"{candidate_id}: Το Meeko απέτυχε με "
            f"return code {meeko_process.returncode}.\n"
            f"Δες: {meeko_log_file}"
        )


    if (
        not pdbqt_file.exists()
        or pdbqt_file.stat().st_size == 0
    ):

        raise RuntimeError(
            f"{candidate_id}: Δεν δημιουργήθηκε PDBQT."
        )


    pdbqt_lines = pdbqt_file.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()


    pdbqt_atom_lines = [
        line
        for line in pdbqt_lines
        if line.startswith(
            ("ATOM", "HETATM")
        )
    ]


    root_count = sum(
        line.strip() == "ROOT"
        for line in pdbqt_lines
    )


    torsdof = extract_torsdof(
        pdbqt_file
    )


    if not pdbqt_atom_lines:
        raise RuntimeError(
            f"{candidate_id}: Το PDBQT δεν περιέχει atoms."
        )


    if root_count != 1:
        raise RuntimeError(
            f"{candidate_id}: Αναμενόταν ένα ROOT, "
            f"βρέθηκαν {root_count}."
        )


    if torsdof is None:
        raise RuntimeError(
            f"{candidate_id}: Δεν βρέθηκε έγκυρο TORSDOF."
        )


    # ========================================================
    # 6G. Ενημέρωση του preparation audit
    # ========================================================

    audit_df.loc[
        dataframe_index,
        "Prepared_Canonical_SMILES"
    ] = prepared_isomeric_smiles

    audit_df.loc[
        dataframe_index,
        "Input_NonIsomeric_Canonical_SMILES"
    ] = input_nonisomeric_smiles

    audit_df.loc[
        dataframe_index,
        "Prepared_NonIsomeric_Canonical_SMILES"
    ] = prepared_nonisomeric_smiles

    audit_df.loc[
        dataframe_index,
        "Constitutional_Identity_Preserved"
    ] = constitutional_identity_preserved

    audit_df.loc[
        dataframe_index,
        "Molecular_Formula_Preserved"
    ] = molecular_formula_preserved

    audit_df.loc[
        dataframe_index,
        "Formal_Charge_Preserved"
    ] = formal_charge_preserved

    audit_df.loc[
        dataframe_index,
        "Heavy_Atom_Count_Preserved"
    ] = heavy_atom_count_preserved

    audit_df.loc[
        dataframe_index,
        "Stereo_Only_Difference"
    ] = stereo_only_difference

    audit_df.loc[
        dataframe_index,
        "Input_Assigned_Stereocenter_Count"
    ] = len(input_assigned_centers)

    audit_df.loc[
        dataframe_index,
        "Input_Unassigned_Stereocenter_Count"
    ] = len(input_unassigned_centers)

    audit_df.loc[
        dataframe_index,
        "Prepared_Assigned_Stereocenter_Count"
    ] = len(prepared_assigned_centers)

    audit_df.loc[
        dataframe_index,
        "Input_Assigned_Double_Bond_Stereo_Count"
    ] = input_assigned_double_bonds

    audit_df.loc[
        dataframe_index,
        "Prepared_Assigned_Double_Bond_Stereo_Count"
    ] = prepared_assigned_double_bonds

    audit_df.loc[
        dataframe_index,
        "Stereochemistry_Status"
    ] = (
        "Unspecified in generated SMILES; "
        "specific 3D realization assigned by OpenBabel"
    )

    audit_df.loc[
        dataframe_index,
        "Identity_Acceptance_Basis"
    ] = (
        "Non-isomeric canonical SMILES, molecular formula, "
        "formal charge and heavy-atom count preserved"
    )

    audit_df.loc[
        dataframe_index,
        "Chemical_Identity_Preserved"
    ] = True

    audit_df.loc[
        dataframe_index,
        "Meeko_Success"
    ] = True

    audit_df.loc[
        dataframe_index,
        "Preparation_Success"
    ] = True

    audit_df.loc[
        dataframe_index,
        "Preparation_Success_Boolean"
    ] = True

    audit_df.loc[
        dataframe_index,
        "Molecular_Formula"
    ] = prepared_formula

    audit_df.loc[
        dataframe_index,
        "Formal_Charge"
    ] = prepared_formal_charge

    audit_df.loc[
        dataframe_index,
        "Heavy_Atom_Count"
    ] = prepared_heavy_atom_count

    audit_df.loc[
        dataframe_index,
        "Explicit_Hydrogen_Count"
    ] = int(
        sum(
            atom.GetAtomicNum() == 1
            for atom in prepared_molecule.GetAtoms()
        )
    )

    audit_df.loc[
        dataframe_index,
        "Total_Atom_Count"
    ] = prepared_molecule.GetNumAtoms()

    audit_df.loc[
        dataframe_index,
        "SDF_File"
    ] = str(sdf_file)

    audit_df.loc[
        dataframe_index,
        "SDF_SHA256"
    ] = calculate_sha256(
        sdf_file
    )

    audit_df.loc[
        dataframe_index,
        "PDBQT_Atom_Record_Count"
    ] = len(pdbqt_atom_lines)

    audit_df.loc[
        dataframe_index,
        "PDBQT_TORSDOF"
    ] = torsdof

    audit_df.loc[
        dataframe_index,
        "PDBQT_File"
    ] = str(pdbqt_file)

    audit_df.loc[
        dataframe_index,
        "PDBQT_SHA256"
    ] = calculate_sha256(
        pdbqt_file
    )

    audit_df.loc[
        dataframe_index,
        "Meeko_Log"
    ] = str(meeko_log_file)

    audit_df.loc[
        dataframe_index,
        "Error"
    ] = ""


    stereo_rows.append({
        "Candidate_ID": candidate_id,
        "Selection_Order": int(
            candidate_row["Selection_Order"]
        ),
        "Input_Isomeric_SMILES":
            input_isomeric_smiles,
        "Prepared_Isomeric_SMILES":
            prepared_isomeric_smiles,
        "Input_NonIsomeric_SMILES":
            input_nonisomeric_smiles,
        "Prepared_NonIsomeric_SMILES":
            prepared_nonisomeric_smiles,
        "Constitutional_Identity_Preserved":
            constitutional_identity_preserved,
        "Formula_Input": input_formula,
        "Formula_Prepared": prepared_formula,
        "Formula_Preserved":
            molecular_formula_preserved,
        "Formal_Charge_Input":
            input_formal_charge,
        "Formal_Charge_Prepared":
            prepared_formal_charge,
        "Formal_Charge_Preserved":
            formal_charge_preserved,
        "Heavy_Atoms_Input":
            input_heavy_atom_count,
        "Heavy_Atoms_Prepared":
            prepared_heavy_atom_count,
        "Heavy_Atom_Count_Preserved":
            heavy_atom_count_preserved,
        "Input_Assigned_Stereocenters":
            len(input_assigned_centers),
        "Input_Unassigned_Stereocenters":
            len(input_unassigned_centers),
        "Prepared_Assigned_Stereocenters":
            len(prepared_assigned_centers),
        "Input_Assigned_Double_Bond_Stereo":
            input_assigned_double_bonds,
        "Prepared_Assigned_Double_Bond_Stereo":
            prepared_assigned_double_bonds,
        "Stereo_Only_Difference":
            stereo_only_difference,
        "PDBQT_TORSDOF": torsdof,
        "Preparation_Success": True
    })


    print(
        f"{candidate_id}: COMPLETED | "
        f"TORSDOF={torsdof}"
    )


# ============================================================
# 7. Αποθήκευση ενημερωμένου audit
# ============================================================

audit_df = audit_df.drop(
    columns=[
        "Preparation_Success_Boolean"
    ],
    errors="ignore"
)


audit_df.to_csv(
    PREPARATION_AUDIT_FILE,
    index=False
)


stereochemistry_audit_df = pd.DataFrame(
    stereo_rows
)


stereochemistry_audit_df.to_csv(
    STEREOCHEMISTRY_AUDIT_FILE,
    index=False
)


# ============================================================
# 8. Τελικός έλεγχος και των 20 PDBQT
# ============================================================

audit_success_mask = (
    audit_df["Preparation_Success"]
    .apply(convert_to_boolean)
)


failed_after_repair_df = audit_df[
    ~audit_success_mask
]


missing_pdbqt_candidates = []


for _, row in audit_df.iterrows():

    candidate_id = str(
        row["Candidate_ID"]
    )

    pdbqt_path = (
        OUTPUT_DIRECTORY
        / candidate_id
        / f"{candidate_id}.pdbqt"
    )

    if (
        not pdbqt_path.exists()
        or pdbqt_path.stat().st_size == 0
    ):
        missing_pdbqt_candidates.append(
            candidate_id
        )


if not failed_after_repair_df.empty:

    raise RuntimeError(
        "Παραμένουν αποτυχημένες προετοιμασίες:\n"
        f"{failed_after_repair_df['Candidate_ID'].tolist()}"
    )


if missing_pdbqt_candidates:

    raise RuntimeError(
        "Λείπουν PDBQT αρχεία για:\n"
        f"{missing_pdbqt_candidates}"
    )


# ============================================================
# 9. Αναδημιουργία combined SDF με όλα τα 20 molecules
# ============================================================

combined_writer = Chem.SDWriter(
    str(COMBINED_PREPARED_SDF)
)


combined_molecule_count = 0


sorted_audit_df = audit_df.sort_values(
    by="Selection_Order"
)


for _, row in sorted_audit_df.iterrows():

    candidate_id = str(
        row["Candidate_ID"]
    )

    candidate_sdf = (
        OUTPUT_DIRECTORY
        / candidate_id
        / f"{candidate_id}_3D.sdf"
    )

    molecule = load_single_sdf_molecule(
        candidate_sdf
    )

    molecule.SetProp(
        "_Name",
        candidate_id
    )

    molecule.SetProp(
        "Candidate_ID",
        candidate_id
    )

    molecule.SetProp(
        "Selection_Order",
        str(
            int(row["Selection_Order"])
        )
    )

    molecule.SetProp(
        "Input_SMILES",
        str(row["Input_SMILES"])
    )

    molecule.SetProp(
        "Prepared_Canonical_SMILES",
        str(row["Prepared_Canonical_SMILES"])
    )

    stereochemistry_status = str(
        row.get(
            "Stereochemistry_Status",
            ""
        )
    )

    if (
        stereochemistry_status
        and stereochemistry_status.lower() != "nan"
    ):
        molecule.SetProp(
            "Stereochemistry_Status",
            stereochemistry_status
        )

    combined_writer.write(
        molecule
    )

    combined_molecule_count += 1


combined_writer.close()


if combined_molecule_count != 20:
    raise RuntimeError(
        "Το combined SDF δεν περιέχει 20 molecules."
    )


# ============================================================
# 10. Ενημέρωση preparation metadata
# ============================================================

if PREPARATION_METADATA_FILE.exists():

    metadata_df = pd.read_csv(
        PREPARATION_METADATA_FILE
    )

else:

    metadata_df = pd.DataFrame(
        columns=[
            "Parameter",
            "Value"
        ]
    )


def set_metadata_value(
    dataframe,
    parameter,
    value
):
    """
    Ενημερώνει ή προσθέτει μία metadata τιμή.
    """

    parameter_mask = (
        dataframe["Parameter"] == parameter
    )

    if parameter_mask.any():

        dataframe.loc[
            parameter_mask,
            "Value"
        ] = value

    else:

        new_row = pd.DataFrame([
            {
                "Parameter": parameter,
                "Value": value
            }
        ])

        dataframe = pd.concat(
            [
                dataframe,
                new_row
            ],
            ignore_index=True
        )

    return dataframe


metadata_df = set_metadata_value(
    metadata_df,
    "Successful preparations",
    20
)

metadata_df = set_metadata_value(
    metadata_df,
    "Failed preparations",
    0
)

metadata_df = set_metadata_value(
    metadata_df,
    "Candidates with stereo-only 3D assignment",
    len(stereochemistry_audit_df)
)

metadata_df = set_metadata_value(
    metadata_df,
    "Stereo handling",
    (
        "Unspecified input stereochemistry retained as "
        "unspecified at the molecular-record level; "
        "one OpenBabel 3D stereochemical realization "
        "used for docking"
    )
)

metadata_df = set_metadata_value(
    metadata_df,
    "Combined prepared SDF molecule count",
    combined_molecule_count
)

metadata_df.to_csv(
    PREPARATION_METADATA_FILE,
    index=False
)


# ============================================================
# 11. Δημιουργία ειδικού stereochemistry metadata
# ============================================================

stereo_metadata_df = pd.DataFrame([
    {
        "Parameter": "Affected candidate count",
        "Value": len(
            stereochemistry_audit_df
        )
    },
    {
        "Parameter": "Affected candidates",
        "Value": ", ".join(
            stereochemistry_audit_df[
                "Candidate_ID"
            ].tolist()
        )
    },
    {
        "Parameter": "Original stereochemistry",
        "Value": (
            "Unspecified in generated canonical SMILES"
        )
    },
    {
        "Parameter": "3D stereochemistry",
        "Value": (
            "One concrete realization assigned during "
            "OpenBabel 3D generation"
        )
    },
    {
        "Parameter": "Acceptance criteria",
        "Value": (
            "Non-isomeric canonical SMILES, molecular "
            "formula, formal charge and heavy-atom count "
            "must all be preserved"
        )
    },
    {
        "Parameter": "Stereoisomer enumeration performed",
        "Value": False
    },
    {
        "Parameter": "Methodological interpretation",
        "Value": (
            "Paper-aligned single-conformer preparation; "
            "stereochemical ambiguity retained as a limitation"
        )
    }
])


stereo_metadata_df.to_csv(
    STEREOCHEMISTRY_METADATA_FILE,
    index=False
)


# ============================================================
# 12. Εμφάνιση αποτελεσμάτων
# ============================================================

print("\n" + "=" * 100)
print("STEREOCHEMISTRY AUDIT ΤΩΝ 3 CANDIDATES")
print("=" * 100)

display(
    stereochemistry_audit_df
)


final_summary_df = pd.DataFrame([
    {
        "Parameter": "Total candidates",
        "Value": len(audit_df)
    },
    {
        "Parameter": "Successful preparations",
        "Value": int(
            audit_success_mask.sum()
        )
    },
    {
        "Parameter": "Failed preparations",
        "Value": int(
            (~audit_success_mask).sum()
        )
    },
    {
        "Parameter": "Available PDBQT files",
        "Value": (
            len(audit_df)
            - len(missing_pdbqt_candidates)
        )
    },
    {
        "Parameter": "Combined SDF molecules",
        "Value": combined_molecule_count
    },
    {
        "Parameter": "Stereo-only accepted cases",
        "Value": len(
            stereochemistry_audit_df
        )
    }
])


print("\n" + "=" * 100)
print("FINAL PREPARATION SUMMARY")
print("=" * 100)

display(
    final_summary_df
)


print("\n" + "-" * 100)
print(
    "Η ΠΡΟΕΤΟΙΜΑΣΙΑ ΚΑΙ ΤΩΝ 20 CANDIDATES "
    "ΟΛΟΚΛΗΡΩΘΗΚΕ"
)
print("-" * 100)

print(
    "Prepared PDBQT files: 20/20"
)

print(
    f"Combined SDF: {COMBINED_PREPARED_SDF}"
)

print(
    f"Stereochemistry audit: "
    f"{STEREOCHEMISTRY_AUDIT_FILE}"
)

print(
    "\nΔεν πραγματοποιήθηκε ακόμη docking."
)

In [ ]:
# ============================================================
# ΔΙΟΡΘΩΣΗ PANDAS 3 ΚΑΙ ΟΛΟΚΛΗΡΩΣΗ ΤΩΝ METADATA
#
# Το προηγούμενο κελί είχε ήδη:
# - ελέγξει τα 3 stereochemistry cases,
# - δημιουργήσει τα PDBQT τους,
# - ενημερώσει το preparation audit,
# - δημιουργήσει το combined SDF.
#
# Το παρόν κελί:
# 1. Επιβεβαιώνει ότι υπάρχουν 20 επιτυχημένες προετοιμασίες
# 2. Επιβεβαιώνει ότι υπάρχουν 20 PDBQT αρχεία
# 3. Ελέγχει το combined SDF
# 4. Διορθώνει το pandas string-dtype πρόβλημα
# 5. Ολοκληρώνει τα metadata
#
# Δεν πραγματοποιεί νέο OpenBabel generation ή docking.
# ============================================================

import os
from pathlib import Path

import pandas as pd

from rdkit import Chem
from IPython.display import display


# ============================================================
# 1. Βασικές διαδρομές
# ============================================================

PROJECT_DIRECTORY = Path(
    "/home/afroditi/ptixiaki"
)

os.chdir(
    PROJECT_DIRECTORY
)

PREPARATION_AUDIT_FILE = Path(
    "SELECTED_20_LIGAND_PREPARATION_AUDIT.csv"
)

PREPARATION_METADATA_FILE = Path(
    "SELECTED_20_LIGAND_PREPARATION_METADATA.csv"
)

STEREOCHEMISTRY_AUDIT_FILE = Path(
    "SELECTED_20_STEREOCHEMISTRY_AUDIT.csv"
)

STEREOCHEMISTRY_METADATA_FILE = Path(
    "SELECTED_20_STEREOCHEMISTRY_METADATA.csv"
)

COMBINED_PREPARED_SDF = Path(
    "SELECTED_20_GENERATED_CANDIDATES_3D.sdf"
)

OUTPUT_DIRECTORY = Path(
    "SELECTED_20_DOCKING_INPUTS"
)

FINAL_SUMMARY_FILE = Path(
    "SELECTED_20_FINAL_PREPARATION_SUMMARY.csv"
)


# ============================================================
# 2. Βοηθητική μετατροπή σε boolean
# ============================================================

def convert_to_boolean(value):
    """
    Μετατρέπει τιμές από CSV σε πραγματικό boolean.
    """

    if isinstance(value, bool):
        return value

    return str(value).strip().lower() in {
        "true",
        "1",
        "yes"
    }


# ============================================================
# 3. Ανάγνωση του ενημερωμένου preparation audit
# ============================================================

if not PREPARATION_AUDIT_FILE.exists():
    raise FileNotFoundError(
        f"Δεν βρέθηκε: {PREPARATION_AUDIT_FILE}"
    )

audit_df = pd.read_csv(
    PREPARATION_AUDIT_FILE
)

if len(audit_df) != 20:
    raise RuntimeError(
        f"Το preparation audit περιέχει {len(audit_df)} "
        "γραμμές αντί για 20."
    )

success_mask = audit_df[
    "Preparation_Success"
].apply(
    convert_to_boolean
)

successful_count = int(
    success_mask.sum()
)

failed_count = int(
    (~success_mask).sum()
)

if failed_count > 0:
    failed_candidates = audit_df.loc[
        ~success_mask,
        "Candidate_ID"
    ].tolist()

    raise RuntimeError(
        "Παραμένουν αποτυχημένα candidates:\n"
        f"{failed_candidates}"
    )


# ============================================================
# 4. Έλεγχος ότι υπάρχουν και τα 20 PDBQT
# ============================================================

missing_pdbqt_candidates = []

for _, row in audit_df.iterrows():

    candidate_id = str(
        row["Candidate_ID"]
    )

    pdbqt_path = (
        OUTPUT_DIRECTORY
        / candidate_id
        / f"{candidate_id}.pdbqt"
    )

    if (
        not pdbqt_path.exists()
        or pdbqt_path.stat().st_size == 0
    ):
        missing_pdbqt_candidates.append(
            candidate_id
        )


if missing_pdbqt_candidates:
    raise RuntimeError(
        "Λείπουν PDBQT αρχεία για:\n"
        f"{missing_pdbqt_candidates}"
    )


available_pdbqt_count = (
    len(audit_df)
    - len(missing_pdbqt_candidates)
)


# ============================================================
# 5. Έλεγχος του combined SDF
# ============================================================

if not COMBINED_PREPARED_SDF.exists():
    raise FileNotFoundError(
        f"Δεν βρέθηκε: {COMBINED_PREPARED_SDF}"
    )


combined_supplier = Chem.SDMolSupplier(
    str(COMBINED_PREPARED_SDF),
    removeHs=False,
    sanitize=True
)

combined_molecules = [
    molecule
    for molecule in combined_supplier
    if molecule is not None
]

combined_molecule_count = len(
    combined_molecules
)

if combined_molecule_count != 20:
    raise RuntimeError(
        "Το combined SDF δεν περιέχει 20 έγκυρα molecules.\n"
        f"Βρέθηκαν: {combined_molecule_count}"
    )


# ============================================================
# 6. Ανάγνωση του stereochemistry audit
# ============================================================

if not STEREOCHEMISTRY_AUDIT_FILE.exists():
    raise FileNotFoundError(
        f"Δεν βρέθηκε: {STEREOCHEMISTRY_AUDIT_FILE}"
    )

stereochemistry_audit_df = pd.read_csv(
    STEREOCHEMISTRY_AUDIT_FILE
)

stereo_affected_count = len(
    stereochemistry_audit_df
)

stereo_affected_candidates = (
    stereochemistry_audit_df[
        "Candidate_ID"
    ]
    .astype(str)
    .tolist()
)


# ============================================================
# 7. Ανάγνωση ή δημιουργία metadata
# ============================================================

if PREPARATION_METADATA_FILE.exists():

    metadata_df = pd.read_csv(
        PREPARATION_METADATA_FILE
    )

else:

    metadata_df = pd.DataFrame(
        columns=[
            "Parameter",
            "Value"
        ]
    )


if "Parameter" not in metadata_df.columns:
    metadata_df["Parameter"] = ""

if "Value" not in metadata_df.columns:
    metadata_df["Value"] = ""


# Το κρίσιμο fix για pandas 3:
# η στήλη μετατρέπεται σε object και όλες οι νέες τιμές
# αποθηκεύονται ως strings.
metadata_df["Parameter"] = (
    metadata_df["Parameter"]
    .astype("object")
)

metadata_df["Value"] = (
    metadata_df["Value"]
    .astype("object")
)


# ============================================================
# 8. Ασφαλής συνάρτηση ενημέρωσης metadata
# ============================================================

def set_metadata_value(
    dataframe,
    parameter,
    value
):
    """
    Ενημερώνει ή προσθέτει metadata.

    Όλες οι τιμές μετατρέπονται σε string, ώστε το κελί
    να λειτουργεί σωστά και με pandas 3.
    """

    parameter = str(
        parameter
    )

    value = str(
        value
    )

    parameter_mask = (
        dataframe["Parameter"].astype(str)
        == parameter
    )

    if parameter_mask.any():

        dataframe.loc[
            parameter_mask,
            "Value"
        ] = value

    else:

        new_row = pd.DataFrame(
            [
                {
                    "Parameter": parameter,
                    "Value": value
                }
            ],
            dtype="object"
        )

        dataframe = pd.concat(
            [
                dataframe,
                new_row
            ],
            ignore_index=True
        )

    return dataframe


# ============================================================
# 9. Ενημέρωση preparation metadata
# ============================================================

metadata_df = set_metadata_value(
    metadata_df,
    "Successful preparations",
    successful_count
)

metadata_df = set_metadata_value(
    metadata_df,
    "Failed preparations",
    failed_count
)

metadata_df = set_metadata_value(
    metadata_df,
    "Available PDBQT files",
    available_pdbqt_count
)

metadata_df = set_metadata_value(
    metadata_df,
    "Candidates with stereo-only 3D assignment",
    stereo_affected_count
)

metadata_df = set_metadata_value(
    metadata_df,
    "Stereo-affected candidates",
    ", ".join(
        stereo_affected_candidates
    )
)

metadata_df = set_metadata_value(
    metadata_df,
    "Stereo handling",
    (
        "Unspecified input stereochemistry; "
        "one OpenBabel 3D stereochemical realization "
        "used for docking"
    )
)

metadata_df = set_metadata_value(
    metadata_df,
    "Stereoisomer enumeration performed",
    False
)

metadata_df = set_metadata_value(
    metadata_df,
    "Combined prepared SDF",
    COMBINED_PREPARED_SDF
)

metadata_df = set_metadata_value(
    metadata_df,
    "Combined prepared SDF molecule count",
    combined_molecule_count
)

metadata_df.to_csv(
    PREPARATION_METADATA_FILE,
    index=False
)


# ============================================================
# 10. Δημιουργία ειδικού stereochemistry metadata
# ============================================================

stereo_metadata_df = pd.DataFrame(
    [
        {
            "Parameter": "Affected candidate count",
            "Value": str(
                stereo_affected_count
            )
        },
        {
            "Parameter": "Affected candidates",
            "Value": ", ".join(
                stereo_affected_candidates
            )
        },
        {
            "Parameter": "Original stereochemistry",
            "Value": (
                "Unspecified in generated canonical SMILES"
            )
        },
        {
            "Parameter": "3D stereochemistry",
            "Value": (
                "One concrete realization assigned during "
                "OpenBabel 3D generation"
            )
        },
        {
            "Parameter": "Acceptance criteria",
            "Value": (
                "Non-isomeric canonical SMILES, molecular "
                "formula, formal charge and heavy-atom count "
                "were preserved"
            )
        },
        {
            "Parameter": "Stereoisomer enumeration performed",
            "Value": "False"
        },
        {
            "Parameter": "Methodological interpretation",
            "Value": (
                "Single-conformer preparation; stereochemical "
                "ambiguity retained as a limitation"
            )
        }
    ],
    dtype="object"
)

stereo_metadata_df.to_csv(
    STEREOCHEMISTRY_METADATA_FILE,
    index=False
)


# ============================================================
# 11. Τελικό preparation summary
# ============================================================

final_summary_df = pd.DataFrame([
    {
        "Parameter": "Total candidates",
        "Value": 20
    },
    {
        "Parameter": "Successful preparations",
        "Value": successful_count
    },
    {
        "Parameter": "Failed preparations",
        "Value": failed_count
    },
    {
        "Parameter": "Available PDBQT files",
        "Value": available_pdbqt_count
    },
    {
        "Parameter": "Combined SDF molecules",
        "Value": combined_molecule_count
    },
    {
        "Parameter": "Stereo-only accepted cases",
        "Value": stereo_affected_count
    },
    {
        "Parameter": "Ready for docking",
        "Value": (
            successful_count == 20
            and available_pdbqt_count == 20
            and combined_molecule_count == 20
        )
    }
])

final_summary_df.to_csv(
    FINAL_SUMMARY_FILE,
    index=False
)


# ============================================================
# 12. Εμφάνιση αποτελεσμάτων
# ============================================================

print("=" * 100)
print("FINAL PREPARATION SUMMARY")
print("=" * 100)

display(
    final_summary_df
)


print("\n" + "=" * 100)
print("STEREOCHEMISTRY-AFFECTED CANDIDATES")
print("=" * 100)

display(
    stereochemistry_audit_df
)


print("\n" + "-" * 100)
print(
    "Η ΠΡΟΕΤΟΙΜΑΣΙΑ ΚΑΙ ΤΩΝ 20 CANDIDATES "
    "ΟΛΟΚΛΗΡΩΘΗΚΕ"
)
print("-" * 100)

print(
    f"Successful preparations: "
    f"{successful_count}/20"
)

print(
    f"Prepared PDBQT files: "
    f"{available_pdbqt_count}/20"
)

print(
    f"Combined SDF molecules: "
    f"{combined_molecule_count}/20"
)

print(
    f"Stereo-only accepted cases: "
    f"{stereo_affected_count}"
)

print(
    "\nΔεν πραγματοποιήθηκε ακόμη docking."
)

In [ ]:
# ============================================================
# ΒΗΜΑ 7D.2:
# Molecular docking των 20 generated candidates στο VEGFR2
#
# Παγωμένο protocol από το redocking validation:
# - Receptor: 3BE2_RECEPTOR_PREPARED.pdbqt
# - Search space: crystallographic RAJ + 4 Å ανά πλευρά
# - SMINA default scoring function
# - Exhaustiveness: 8
# - Num modes: 9
# - Energy range: 3 kcal/mol
# - Seed: 42
# - CPU: 1
#
# Σημαντικό:
# - Δεν τροποποιείται καμία παράμετρος ανά candidate.
# - Το docking αποτελεί post-generation screening.
# - Το docking score δεν αποτελεί πειραματική binding affinity.
# - Δεν γίνεται RMSD για τα generated molecules, επειδή
#   δεν υπάρχει γνωστή crystallographic reference pose.
# ============================================================

import os
import re
import time
import shutil
import hashlib
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 1. Βασικές διαδρομές
# ============================================================

PROJECT_DIRECTORY = Path(
    "/home/afroditi/ptixiaki"
)

os.chdir(
    PROJECT_DIRECTORY
)

RECEPTOR_PDBQT = Path(
    "3BE2_RECEPTOR_PREPARED.pdbqt"
)

AUTOBOX_REFERENCE_LIGAND = Path(
    "3BE2_RAJ_REFERENCE_CRYSTAL.pdbqt"
)

PREPARATION_AUDIT_FILE = Path(
    "SELECTED_20_LIGAND_PREPARATION_AUDIT.csv"
)

SELECTED_CANDIDATES_FILE = Path(
    "SELECTED_20_GENERATED_CANDIDATES.csv"
)

RAJ_VALIDATION_SUMMARY_FILE = Path(
    "3BE2_RAJ_REDOCKING_VALIDATION_SUMMARY.csv"
)

INPUT_DIRECTORY = Path(
    "SELECTED_20_DOCKING_INPUTS"
)

OUTPUT_DIRECTORY = Path(
    "SELECTED_20_DOCKING_RESULTS"
)

POSE_SCORES_OUTPUT = Path(
    "SELECTED_20_DOCKING_ALL_POSE_SCORES.csv"
)

RANKING_OUTPUT = Path(
    "SELECTED_20_DOCKING_RANKING.csv"
)

METADATA_OUTPUT = Path(
    "SELECTED_20_DOCKING_METADATA.csv"
)

FAILURES_OUTPUT = Path(
    "SELECTED_20_DOCKING_FAILURES.csv"
)

RUN_MANIFEST_OUTPUT = Path(
    "SELECTED_20_DOCKING_RUN_MANIFEST.csv"
)


# ============================================================
# 2. Παγωμένες παράμετροι
# ============================================================

AUTOBOX_PADDING_ANGSTROM = 4.0

EXHAUSTIVENESS = 8

NUM_MODES = 9

ENERGY_RANGE_KCAL_MOL = 3

RANDOM_SEED = 42

CPU_COUNT = 1

EXPECTED_CANDIDATE_COUNT = 20

TIMEOUT_PER_CANDIDATE_SECONDS = 3600


# ============================================================
# 3. Έλεγχος εργαλείου και αρχείων
# ============================================================

smina_path = shutil.which(
    "smina"
)

if smina_path is None:
    raise RuntimeError(
        "Δεν βρέθηκε το SMINA στο ενεργό docking environment."
    )


required_files = [
    RECEPTOR_PDBQT,
    AUTOBOX_REFERENCE_LIGAND,
    PREPARATION_AUDIT_FILE,
    SELECTED_CANDIDATES_FILE,
    RAJ_VALIDATION_SUMMARY_FILE
]


missing_files = [
    str(file_path)
    for file_path in required_files
    if not file_path.exists()
]


if missing_files:
    raise FileNotFoundError(
        "Λείπουν τα ακόλουθα αρχεία:\n"
        f"{missing_files}"
    )


OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 4. Βοηθητικές συναρτήσεις
# ============================================================

def convert_to_boolean(value):
    """
    Μετατρέπει τιμές CSV σε πραγματικό boolean.
    """

    if isinstance(value, bool):
        return value

    return str(value).strip().lower() in {
        "true",
        "1",
        "yes"
    }


def calculate_sha256(file_path):
    """
    Υπολογίζει SHA256 checksum αρχείου.
    """

    hash_object = hashlib.sha256()

    with open(file_path, "rb") as file_handle:

        for block in iter(
            lambda: file_handle.read(1024 * 1024),
            b""
        ):
            hash_object.update(
                block
            )

    return hash_object.hexdigest()


def extract_pose_score(model_lines):
    """
    Εξάγει το SMINA/Vina affinity από τα REMARK records
    μίας docking pose.
    """

    score_patterns = [
        r"minimizedAffinity\s+(-?\d+(?:\.\d+)?)",
        r"VINA\s+RESULT:\s+(-?\d+(?:\.\d+)?)",
        r"Affinity:\s+(-?\d+(?:\.\d+)?)"
    ]

    for line in model_lines:

        for pattern in score_patterns:

            match = re.search(
                pattern,
                line,
                flags=re.IGNORECASE
            )

            if match:
                return float(
                    match.group(1)
                )

    return np.nan


def split_pdbqt_models(lines):
    """
    Διαχωρίζει multi-model PDBQT σε ανεξάρτητες poses.
    """

    models = []

    current_lines = []

    current_model_number = None

    inside_model = False


    for line in lines:

        if line.startswith("MODEL"):

            if current_lines:

                models.append({
                    "Model_Number": current_model_number,
                    "Lines": current_lines
                })

            parts = line.split()

            current_model_number = (
                int(parts[1])
                if (
                    len(parts) >= 2
                    and parts[1].isdigit()
                )
                else len(models) + 1
            )

            current_lines = [
                line
            ]

            inside_model = True

            continue


        if line.startswith("ENDMDL"):

            current_lines.append(
                line
            )

            models.append({
                "Model_Number": current_model_number,
                "Lines": current_lines
            })

            current_lines = []

            current_model_number = None

            inside_model = False

            continue


        if inside_model:

            current_lines.append(
                line
            )


    if current_lines:

        models.append({
            "Model_Number": (
                current_model_number
                if current_model_number is not None
                else len(models) + 1
            ),
            "Lines": current_lines
        })


    if not models:

        models = [{
            "Model_Number": 1,
            "Lines": list(lines)
        }]


    return models


def extract_scores_from_log(log_path):
    """
    Εξάγει mode και affinity από τον πίνακα του SMINA log.
    """

    score_dictionary = {}

    if not log_path.exists():
        return score_dictionary


    log_lines = log_path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()


    for line in log_lines:

        match = re.match(
            r"^\s*(\d+)\s+(-?\d+(?:\.\d+)?)\s+",
            line
        )

        if match:

            pose_id = int(
                match.group(1)
            )

            affinity = float(
                match.group(2)
            )

            score_dictionary[
                pose_id
            ] = affinity


    return score_dictionary


def write_single_pose(
    pose_model,
    output_path
):
    """
    Αποθηκεύει μία επιλεγμένη pose σε ξεχωριστό PDBQT.
    """

    pose_lines = list(
        pose_model["Lines"]
    )

    output_path.write_text(
        "\n".join(pose_lines) + "\n",
        encoding="utf-8"
    )


def resolve_candidate_pdbqt(
    candidate_row
):
    """
    Εντοπίζει το prepared PDBQT του candidate.
    """

    candidate_id = str(
        candidate_row["Candidate_ID"]
    )


    audit_path_value = candidate_row.get(
        "PDBQT_File",
        ""
    )


    if (
        pd.notna(audit_path_value)
        and str(audit_path_value).strip()
        and str(audit_path_value).lower() != "nan"
    ):

        candidate_path = Path(
            str(audit_path_value)
        )

        if not candidate_path.is_absolute():

            candidate_path = (
                PROJECT_DIRECTORY
                / candidate_path
            )

    else:

        candidate_path = (
            INPUT_DIRECTORY
            / candidate_id
            / f"{candidate_id}.pdbqt"
        )


    return candidate_path


# ============================================================
# 5. Ανάγνωση και έλεγχος preparation audit
# ============================================================

preparation_audit_df = pd.read_csv(
    PREPARATION_AUDIT_FILE
)


if len(preparation_audit_df) != EXPECTED_CANDIDATE_COUNT:

    raise RuntimeError(
        f"Αναμένονταν {EXPECTED_CANDIDATE_COUNT} candidates, "
        f"αλλά το audit περιέχει "
        f"{len(preparation_audit_df)}."
    )


preparation_audit_df[
    "Preparation_Success_Boolean"
] = (
    preparation_audit_df[
        "Preparation_Success"
    ].apply(
        convert_to_boolean
    )
)


failed_preparations = preparation_audit_df[
    ~preparation_audit_df[
        "Preparation_Success_Boolean"
    ]
]


if not failed_preparations.empty:

    raise RuntimeError(
        "Υπάρχουν candidates χωρίς επιτυχημένη preparation:\n"
        f"{failed_preparations['Candidate_ID'].tolist()}"
    )


preparation_audit_df[
    "Selection_Order"
] = (
    preparation_audit_df[
        "Selection_Order"
    ].astype(int)
)


preparation_audit_df = (
    preparation_audit_df
    .sort_values(
        by="Selection_Order"
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 6. Έλεγχος μοναδικότητας candidate IDs και PDBQT
# ============================================================

if (
    preparation_audit_df[
        "Candidate_ID"
    ].nunique()
    != EXPECTED_CANDIDATE_COUNT
):

    raise RuntimeError(
        "Τα Candidate_ID δεν είναι μοναδικά."
    )


candidate_input_paths = {}


for _, candidate_row in (
    preparation_audit_df.iterrows()
):

    candidate_id = str(
        candidate_row["Candidate_ID"]
    )

    candidate_pdbqt = resolve_candidate_pdbqt(
        candidate_row
    )

    if (
        not candidate_pdbqt.exists()
        or candidate_pdbqt.stat().st_size == 0
    ):

        raise FileNotFoundError(
            f"Δεν βρέθηκε έγκυρο PDBQT για "
            f"{candidate_id}:\n{candidate_pdbqt}"
        )

    candidate_input_paths[
        candidate_id
    ] = candidate_pdbqt


# ============================================================
# 7. Ανάγνωση του RAJ reference docking score
# ============================================================

raj_summary_df = pd.read_csv(
    RAJ_VALIDATION_SUMMARY_FILE
)


raj_summary_dictionary = dict(
    zip(
        raj_summary_df["Parameter"].astype(str),
        raj_summary_df["Value"]
    )
)


raj_top_scoring_affinity = float(
    raj_summary_dictionary[
        "Top-scoring affinity kcal/mol"
    ]
)


raj_top_scoring_rmsd = float(
    raj_summary_dictionary[
        "Top-scoring pose symmetry-corrected RMSD Angstrom"
    ]
)


raj_best_sampled_rmsd = float(
    raj_summary_dictionary[
        "Best symmetry-corrected RMSD Angstrom"
    ]
)


print("=" * 100)
print("FROZEN DOCKING PROTOCOL")
print("=" * 100)

print(
    f"Receptor: {RECEPTOR_PDBQT}"
)

print(
    f"Autobox reference: {AUTOBOX_REFERENCE_LIGAND}"
)

print(
    f"Autobox padding: {AUTOBOX_PADDING_ANGSTROM} Å"
)

print(
    f"Exhaustiveness: {EXHAUSTIVENESS}"
)

print(
    f"Num modes: {NUM_MODES}"
)

print(
    f"Energy range: {ENERGY_RANGE_KCAL_MOL} kcal/mol"
)

print(
    f"Seed: {RANDOM_SEED}"
)

print(
    f"CPU: {CPU_COUNT}"
)

print(
    f"RAJ top-scoring affinity: "
    f"{raj_top_scoring_affinity:.6f} kcal/mol"
)


# ============================================================
# 8. Δημιουργία reproducibility manifest
# ============================================================

manifest_rows = []


for _, candidate_row in (
    preparation_audit_df.iterrows()
):

    candidate_id = str(
        candidate_row["Candidate_ID"]
    )

    candidate_pdbqt = candidate_input_paths[
        candidate_id
    ]

    manifest_rows.append({
        "Candidate_ID": candidate_id,
        "Selection_Order": int(
            candidate_row["Selection_Order"]
        ),
        "Input_PDBQT": str(
            candidate_pdbqt
        ),
        "Input_PDBQT_SHA256": calculate_sha256(
            candidate_pdbqt
        ),
        "Receptor_PDBQT": str(
            RECEPTOR_PDBQT
        ),
        "Receptor_SHA256": calculate_sha256(
            RECEPTOR_PDBQT
        ),
        "Autobox_Ligand": str(
            AUTOBOX_REFERENCE_LIGAND
        ),
        "Autobox_Ligand_SHA256": calculate_sha256(
            AUTOBOX_REFERENCE_LIGAND
        ),
        "Autobox_Add_Angstrom":
            AUTOBOX_PADDING_ANGSTROM,
        "Exhaustiveness": EXHAUSTIVENESS,
        "Num_Modes": NUM_MODES,
        "Energy_Range_kcal_mol":
            ENERGY_RANGE_KCAL_MOL,
        "Random_Seed": RANDOM_SEED,
        "CPU_Count": CPU_COUNT,
        "SMINA_Path": str(
            smina_path
        ),
        "SMINA_Binary_SHA256": calculate_sha256(
            Path(smina_path)
        )
    })


manifest_df = pd.DataFrame(
    manifest_rows
)


manifest_df.to_csv(
    RUN_MANIFEST_OUTPUT,
    index=False
)


# ============================================================
# 9. Docking των 20 candidates
# ============================================================

pose_score_rows = []

candidate_summary_rows = []

failure_rows = []


total_start_time = time.perf_counter()


for candidate_position, candidate_row in (
    preparation_audit_df.iterrows()
):

    candidate_id = str(
        candidate_row["Candidate_ID"]
    )

    selection_order = int(
        candidate_row["Selection_Order"]
    )

    candidate_pdbqt = candidate_input_paths[
        candidate_id
    ]


    candidate_output_directory = (
        OUTPUT_DIRECTORY
        / candidate_id
    )

    candidate_output_directory.mkdir(
        parents=True,
        exist_ok=True
    )


    docked_output = (
        candidate_output_directory
        / f"{candidate_id}_DOCKED.pdbqt"
    )

    docking_log = (
        candidate_output_directory
        / f"{candidate_id}_DOCKING.log"
    )

    docking_command_file = (
        candidate_output_directory
        / f"{candidate_id}_DOCKING_COMMAND.txt"
    )

    top_pose_output = (
        candidate_output_directory
        / f"{candidate_id}_TOP_POSE.pdbqt"
    )


    # Απομάκρυνση παλαιών outputs.
    for previous_output in [
        docked_output,
        docking_log,
        docking_command_file,
        top_pose_output
    ]:

        if previous_output.exists():
            previous_output.unlink()


    smina_command = [
        smina_path,
        "--receptor",
        str(
            RECEPTOR_PDBQT.resolve()
        ),
        "--ligand",
        str(
            candidate_pdbqt.resolve()
        ),
        "--autobox_ligand",
        str(
            AUTOBOX_REFERENCE_LIGAND.resolve()
        ),
        "--autobox_add",
        str(
            AUTOBOX_PADDING_ANGSTROM
        ),
        "--exhaustiveness",
        str(
            EXHAUSTIVENESS
        ),
        "--num_modes",
        str(
            NUM_MODES
        ),
        "--energy_range",
        str(
            ENERGY_RANGE_KCAL_MOL
        ),
        "--seed",
        str(
            RANDOM_SEED
        ),
        "--cpu",
        str(
            CPU_COUNT
        ),
        "--out",
        str(
            docked_output.resolve()
        ),
        "--log",
        str(
            docking_log.resolve()
        )
    ]


    docking_command_file.write_text(
        " ".join(smina_command) + "\n",
        encoding="utf-8"
    )


    print(
        f"[{candidate_position + 1:02d}/20] "
        f"Docking {candidate_id}...",
        flush=True
    )


    candidate_start_time = time.perf_counter()


    try:

        smina_process = subprocess.run(
            smina_command,
            capture_output=True,
            text=True,
            cwd=PROJECT_DIRECTORY,
            timeout=TIMEOUT_PER_CANDIDATE_SECONDS
        )


        candidate_runtime_seconds = (
            time.perf_counter()
            - candidate_start_time
        )


        if smina_process.returncode != 0:

            raise RuntimeError(
                "Το SMINA επέστρεψε return code "
                f"{smina_process.returncode}.\n"
                f"STDERR:\n{smina_process.stderr[-2000:]}"
            )


        if (
            not docked_output.exists()
            or docked_output.stat().st_size == 0
        ):

            raise RuntimeError(
                "Δεν δημιουργήθηκε έγκυρο docked PDBQT."
            )


        docked_lines = docked_output.read_text(
            encoding="utf-8",
            errors="replace"
        ).splitlines()


        pose_models = split_pdbqt_models(
            docked_lines
        )


        log_scores = extract_scores_from_log(
            docking_log
        )


        candidate_pose_rows = []


        for output_position, pose_model in enumerate(
            pose_models,
            start=1
        ):

            pose_id = int(
                pose_model["Model_Number"]
            )

            affinity = extract_pose_score(
                pose_model["Lines"]
            )


            if np.isnan(affinity):

                affinity = log_scores.get(
                    pose_id,
                    np.nan
                )


            pose_record = {
                "Candidate_ID": candidate_id,
                "Selection_Order":
                    selection_order,
                "Pose_ID": pose_id,
                "Output_Position":
                    output_position,
                "SMINA_Affinity_kcal_mol":
                    affinity
            }


            candidate_pose_rows.append(
                pose_record
            )

            pose_score_rows.append(
                pose_record
            )


        candidate_pose_df = pd.DataFrame(
            candidate_pose_rows
        )


        valid_pose_scores_df = candidate_pose_df[
            candidate_pose_df[
                "SMINA_Affinity_kcal_mol"
            ].notna()
        ].copy()


        if valid_pose_scores_df.empty:

            raise RuntimeError(
                "Δεν εντοπίστηκε έγκυρο docking score."
            )


        valid_pose_scores_df = (
            valid_pose_scores_df
            .sort_values(
                by=[
                    "SMINA_Affinity_kcal_mol",
                    "Pose_ID"
                ],
                ascending=[
                    True,
                    True
                ]
            )
            .reset_index(
                drop=True
            )
        )


        top_pose_row = (
            valid_pose_scores_df.iloc[0]
        )

        top_pose_id = int(
            top_pose_row["Pose_ID"]
        )

        top_affinity = float(
            top_pose_row[
                "SMINA_Affinity_kcal_mol"
            ]
        )


        if len(valid_pose_scores_df) >= 2:

            second_best_affinity = float(
                valid_pose_scores_df.iloc[1][
                    "SMINA_Affinity_kcal_mol"
                ]
            )

            score_gap = float(
                second_best_affinity
                - top_affinity
            )

        else:

            second_best_affinity = np.nan

            score_gap = np.nan


        top_pose_model = next(
            (
                model
                for model in pose_models
                if int(
                    model["Model_Number"]
                ) == top_pose_id
            ),
            None
        )


        if top_pose_model is None:

            raise RuntimeError(
                "Δεν βρέθηκε το model της top-scoring pose."
            )


        write_single_pose(
            top_pose_model,
            top_pose_output
        )


        stereo_only_difference = (
            convert_to_boolean(
                candidate_row.get(
                    "Stereo_Only_Difference",
                    False
                )
            )
        )


        candidate_summary_rows.append({
            "Candidate_ID": candidate_id,
            "Selection_Order":
                selection_order,
            "Input_SMILES": str(
                candidate_row.get(
                    "Input_SMILES",
                    ""
                )
            ),
            "Molecular_Formula": str(
                candidate_row.get(
                    "Molecular_Formula",
                    ""
                )
            ),
            "Formal_Charge": candidate_row.get(
                "Formal_Charge",
                np.nan
            ),
            "Heavy_Atom_Count":
                candidate_row.get(
                    "Heavy_Atom_Count",
                    np.nan
                ),
            "PDBQT_TORSDOF":
                candidate_row.get(
                    "PDBQT_TORSDOF",
                    np.nan
                ),
            "Stereo_Only_3D_Assignment":
                stereo_only_difference,
            "Generated_Pose_Count": int(
                len(candidate_pose_df)
            ),
            "Valid_Score_Count": int(
                len(valid_pose_scores_df)
            ),
            "Top_Pose_ID": top_pose_id,
            "Top_SMINA_Affinity_kcal_mol":
                top_affinity,
            "Second_Best_Affinity_kcal_mol":
                second_best_affinity,
            "Top_to_Second_Score_Gap_kcal_mol":
                score_gap,
            "Delta_vs_RAJ_Top_Affinity_kcal_mol":
                float(
                    top_affinity
                    - raj_top_scoring_affinity
                ),
            "Numerically_Lower_Score_Than_RAJ":
                bool(
                    top_affinity
                    < raj_top_scoring_affinity
                ),
            "Runtime_Seconds":
                candidate_runtime_seconds,
            "Docking_Return_Code":
                smina_process.returncode,
            "Input_PDBQT": str(
                candidate_pdbqt
            ),
            "Docked_PDBQT": str(
                docked_output
            ),
            "Top_Pose_PDBQT": str(
                top_pose_output
            ),
            "Docking_Log": str(
                docking_log
            ),
            "Docking_Command_File": str(
                docking_command_file
            ),
            "Docked_PDBQT_SHA256":
                calculate_sha256(
                    docked_output
                )
        })


        print(
            f"    OK | Top score: "
            f"{top_affinity:.4f} kcal/mol | "
            f"Poses: {len(candidate_pose_df)} | "
            f"Runtime: "
            f"{candidate_runtime_seconds:.1f} s",
            flush=True
        )


    except Exception as error:

        candidate_runtime_seconds = (
            time.perf_counter()
            - candidate_start_time
        )


        failure_rows.append({
            "Candidate_ID": candidate_id,
            "Selection_Order":
                selection_order,
            "Input_PDBQT": str(
                candidate_pdbqt
            ),
            "Runtime_Seconds":
                candidate_runtime_seconds,
            "Error": (
                f"{type(error).__name__}: {error}"
            )
        })


        print(
            f"    FAILED | "
            f"{type(error).__name__}: {error}",
            flush=True
        )


total_runtime_seconds = (
    time.perf_counter()
    - total_start_time
)


# ============================================================
# 10. Αποθήκευση όλων των pose scores
# ============================================================

pose_scores_df = pd.DataFrame(
    pose_score_rows
)


if not pose_scores_df.empty:

    pose_scores_df = (
        pose_scores_df
        .sort_values(
            by=[
                "Selection_Order",
                "SMINA_Affinity_kcal_mol",
                "Pose_ID"
            ],
            ascending=[
                True,
                True,
                True
            ],
            na_position="last"
        )
        .reset_index(
            drop=True
        )
    )


pose_scores_df.to_csv(
    POSE_SCORES_OUTPUT,
    index=False
)


# ============================================================
# 11. Έλεγχος πιθανών αποτυχιών
# ============================================================

failures_df = pd.DataFrame(
    failure_rows
)


failures_df.to_csv(
    FAILURES_OUTPUT,
    index=False
)


if not failures_df.empty:

    print("\n" + "=" * 100)
    print("DOCKING FAILURES")
    print("=" * 100)

    display(
        failures_df
    )

    raise RuntimeError(
        f"Απέτυχε το docking για "
        f"{len(failures_df)} candidates. "
        "Τα επιτυχημένα αποτελέσματα αποθηκεύτηκαν, "
        "αλλά δεν δημιουργείται τελική κατάταξη."
    )


# ============================================================
# 12. Δημιουργία τελικής κατάταξης
# ============================================================

candidate_summary_df = pd.DataFrame(
    candidate_summary_rows
)


if len(candidate_summary_df) != EXPECTED_CANDIDATE_COUNT:

    raise RuntimeError(
        "Ο τελικός πίνακας δεν περιέχει 20 candidates.\n"
        f"Βρέθηκαν: {len(candidate_summary_df)}"
    )


candidate_summary_df = (
    candidate_summary_df
    .sort_values(
        by=[
            "Top_SMINA_Affinity_kcal_mol",
            "Candidate_ID"
        ],
        ascending=[
            True,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


candidate_summary_df.insert(
    0,
    "Docking_Rank",
    range(
        1,
        len(candidate_summary_df) + 1
    )
)


# ============================================================
# 13. Προσθήκη των αρχικών selection properties
# ============================================================

selected_candidates_df = pd.read_csv(
    SELECTED_CANDIDATES_FILE
)


if "Selection_Order" not in selected_candidates_df.columns:

    raise KeyError(
        "Το SELECTED_20_GENERATED_CANDIDATES.csv "
        "δεν περιέχει Selection_Order."
    )


selected_candidates_df[
    "Selection_Order"
] = (
    selected_candidates_df[
        "Selection_Order"
    ].astype(int)
)


ranking_df = candidate_summary_df.merge(
    selected_candidates_df,
    on="Selection_Order",
    how="left",
    suffixes=(
        "",
        "_Selection"
    ),
    validate="one_to_one"
)


ranking_df.to_csv(
    RANKING_OUTPUT,
    index=False
)


# ============================================================
# 14. Δημιουργία metadata
# ============================================================

metadata_rows = [
    {
        "Parameter": "Target",
        "Value": "VEGFR2/KDR"
    },
    {
        "Parameter": "PDB structure",
        "Value": "3BE2"
    },
    {
        "Parameter": "Generated model protein-conditioned",
        "Value": "False"
    },
    {
        "Parameter": "Interpretation",
        "Value": (
            "Post-generation structure-based screening"
        )
    },
    {
        "Parameter": "Candidate count",
        "Value": str(
            EXPECTED_CANDIDATE_COUNT
        )
    },
    {
        "Parameter": "Successful docking runs",
        "Value": str(
            len(candidate_summary_df)
        )
    },
    {
        "Parameter": "Failed docking runs",
        "Value": str(
            len(failures_df)
        )
    },
    {
        "Parameter": "Receptor",
        "Value": str(
            RECEPTOR_PDBQT
        )
    },
    {
        "Parameter": "Autobox ligand",
        "Value": str(
            AUTOBOX_REFERENCE_LIGAND
        )
    },
    {
        "Parameter": "Autobox padding each side Angstrom",
        "Value": str(
            AUTOBOX_PADDING_ANGSTROM
        )
    },
    {
        "Parameter": "SMINA scoring function",
        "Value": "Default"
    },
    {
        "Parameter": "Exhaustiveness",
        "Value": str(
            EXHAUSTIVENESS
        )
    },
    {
        "Parameter": "Requested poses per candidate",
        "Value": str(
            NUM_MODES
        )
    },
    {
        "Parameter": "Energy range kcal/mol",
        "Value": str(
            ENERGY_RANGE_KCAL_MOL
        )
    },
    {
        "Parameter": "Random seed",
        "Value": str(
            RANDOM_SEED
        )
    },
    {
        "Parameter": "CPU count",
        "Value": str(
            CPU_COUNT
        )
    },
    {
        "Parameter": "Total runtime seconds",
        "Value": str(
            total_runtime_seconds
        )
    },
    {
        "Parameter": "RAJ top-scoring affinity kcal/mol",
        "Value": str(
            raj_top_scoring_affinity
        )
    },
    {
        "Parameter": "RAJ top-scoring pose RMSD Angstrom",
        "Value": str(
            raj_top_scoring_rmsd
        )
    },
    {
        "Parameter": "RAJ best sampled RMSD Angstrom",
        "Value": str(
            raj_best_sampled_rmsd
        )
    },
    {
        "Parameter": "Stereoisomer enumeration performed",
        "Value": "False"
    },
    {
        "Parameter": "Candidates with one assigned 3D stereo realization",
        "Value": "3"
    },
    {
        "Parameter": "Docking score interpretation",
        "Value": (
            "Computational ranking metric; "
            "not experimental binding affinity"
        )
    }
]


metadata_df = pd.DataFrame(
    metadata_rows,
    dtype="object"
)


metadata_df.to_csv(
    METADATA_OUTPUT,
    index=False
)


# ============================================================
# 15. Εμφάνιση της κατάταξης
# ============================================================

display_columns = [
    "Docking_Rank",
    "Candidate_ID",
    "Selection_Order",
    "Top_SMINA_Affinity_kcal_mol",
    "Delta_vs_RAJ_Top_Affinity_kcal_mol",
    "Numerically_Lower_Score_Than_RAJ",
    "Generated_Pose_Count",
    "PDBQT_TORSDOF",
    "Stereo_Only_3D_Assignment",
    "Input_SMILES"
]


print("\n" + "=" * 110)
print("SELECTED 20 — SMINA DOCKING RANKING")
print("=" * 110)

display(
    ranking_df[
        display_columns
    ]
)


# ============================================================
# 16. Συνοπτικά αριθμητικά αποτελέσματα
# ============================================================

score_series = ranking_df[
    "Top_SMINA_Affinity_kcal_mol"
]


numerically_better_than_raj_count = int(
    ranking_df[
        "Numerically_Lower_Score_Than_RAJ"
    ].sum()
)


summary_df = pd.DataFrame([
    {
        "Parameter": "Docked candidates",
        "Value": len(
            ranking_df
        )
    },
    {
        "Parameter": "Best generated score kcal/mol",
        "Value": float(
            score_series.min()
        )
    },
    {
        "Parameter": "Median generated score kcal/mol",
        "Value": float(
            score_series.median()
        )
    },
    {
        "Parameter": "Mean generated score kcal/mol",
        "Value": float(
            score_series.mean()
        )
    },
    {
        "Parameter": "Worst generated score kcal/mol",
        "Value": float(
            score_series.max()
        )
    },
    {
        "Parameter": "RAJ top score kcal/mol",
        "Value": raj_top_scoring_affinity
    },
    {
        "Parameter": (
            "Generated candidates with numerically "
            "lower score than RAJ"
        ),
        "Value": numerically_better_than_raj_count
    },
    {
        "Parameter": "Total runtime seconds",
        "Value": total_runtime_seconds
    }
])


print("\n" + "=" * 110)
print("DOCKING SUMMARY")
print("=" * 110)

display(
    summary_df
)


# ============================================================
# 17. Τελική αναφορά αρχείων
# ============================================================

print("\n" + "-" * 110)
print(
    "ΤΟ DOCKING ΚΑΙ ΤΩΝ 20 GENERATED CANDIDATES "
    "ΟΛΟΚΛΗΡΩΘΗΚΕ"
)
print("-" * 110)

print(
    f"Successful runs: "
    f"{len(ranking_df)}/20"
)

print(
    f"Total runtime: "
    f"{total_runtime_seconds:.2f} seconds"
)

print(
    f"Ranking file: {RANKING_OUTPUT}"
)

print(
    f"All pose scores: {POSE_SCORES_OUTPUT}"
)

print(
    f"Metadata: {METADATA_OUTPUT}"
)

print(
    f"Run manifest: {RUN_MANIFEST_OUTPUT}"
)

print(
    f"Results directory: {OUTPUT_DIRECTORY}"
)

print(
    "\nΔεν έγινε επιλογή candidate ή βιολογικός "
    "ισχυρισμός μόνο από το docking score."
)

In [ ]:
# ============================================================
# ΒΗΜΑ 7D.3:
# Post-docking ανάλυση των 20 generated candidates
#
# Στόχοι:
# 1. Υπολογισμός ligand efficiency
# 2. Σύγκριση με τον crystallographic ligand RAJ
# 3. Ξεχωριστή κατάταξη raw score και ligand efficiency
# 4. Έλεγχος πιθανής επίδρασης του μοριακού μεγέθους
# 5. Δημιουργία πινάκων και διαγραμμάτων
#
# Δεν πραγματοποιείται:
# - νέο docking,
# - αλλαγή docking παραμέτρων,
# - weighted συνολικό score,
# - τελική επιλογή lead molecule.
# ============================================================

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from IPython.display import display


# ============================================================
# 1. Βασικές διαδρομές
# ============================================================

PROJECT_DIRECTORY = Path(
    "/home/afroditi/ptixiaki"
)

os.chdir(
    PROJECT_DIRECTORY
)

DOCKING_RANKING_FILE = Path(
    "SELECTED_20_DOCKING_RANKING.csv"
)

RAJ_VALIDATION_FILE = Path(
    "3BE2_RAJ_REDOCKING_VALIDATION_SUMMARY.csv"
)

POST_DOCKING_OUTPUT = Path(
    "SELECTED_20_POST_DOCKING_ANALYSIS.csv"
)

TOP_SCORE_OUTPUT = Path(
    "SELECTED_20_TOP_5_BY_DOCKING_SCORE.csv"
)

TOP_LIGAND_EFFICIENCY_OUTPUT = Path(
    "SELECTED_20_TOP_5_BY_LIGAND_EFFICIENCY.csv"
)

PRIORITY_REVIEW_OUTPUT = Path(
    "SELECTED_20_POST_DOCKING_PRIORITY_REVIEW.csv"
)

CORRELATION_OUTPUT = Path(
    "SELECTED_20_DOCKING_SPEARMAN_CORRELATIONS.csv"
)

SUMMARY_OUTPUT = Path(
    "SELECTED_20_POST_DOCKING_SUMMARY.csv"
)

DOCKING_SCORE_PLOT = Path(
    "SELECTED_20_DOCKING_SCORE_RANKING.png"
)

LIGAND_EFFICIENCY_PLOT = Path(
    "SELECTED_20_LIGAND_EFFICIENCY_RANKING.png"
)

SCORE_VS_HEAVY_ATOMS_PLOT = Path(
    "SELECTED_20_DOCKING_SCORE_VS_HEAVY_ATOMS.png"
)


# ============================================================
# 2. Έλεγχος input αρχείων
# ============================================================

required_files = [
    DOCKING_RANKING_FILE,
    RAJ_VALIDATION_FILE
]

missing_files = [
    str(file_path)
    for file_path in required_files
    if not file_path.exists()
]

if missing_files:

    raise FileNotFoundError(
        "Λείπουν τα ακόλουθα αρχεία:\n"
        f"{missing_files}"
    )


# ============================================================
# 3. Ανάγνωση docking αποτελεσμάτων
# ============================================================

docking_df = pd.read_csv(
    DOCKING_RANKING_FILE
)

if len(docking_df) != 20:

    raise RuntimeError(
        f"Αναμένονταν 20 candidates, "
        f"αλλά βρέθηκαν {len(docking_df)}."
    )


required_columns = [
    "Docking_Rank",
    "Candidate_ID",
    "Selection_Order",
    "Top_SMINA_Affinity_kcal_mol",
    "Heavy_Atom_Count",
    "PDBQT_TORSDOF"
]

missing_columns = [
    column
    for column in required_columns
    if column not in docking_df.columns
]

if missing_columns:

    raise KeyError(
        "Λείπουν οι ακόλουθες στήλες από το ranking:\n"
        f"{missing_columns}"
    )


numeric_columns = [
    "Docking_Rank",
    "Selection_Order",
    "Top_SMINA_Affinity_kcal_mol",
    "Heavy_Atom_Count",
    "PDBQT_TORSDOF"
]

for column in numeric_columns:

    docking_df[column] = pd.to_numeric(
        docking_df[column],
        errors="raise"
    )


if docking_df["Candidate_ID"].nunique() != 20:

    raise RuntimeError(
        "Τα Candidate_ID δεν είναι μοναδικά."
    )


if (
    docking_df["Heavy_Atom_Count"] <= 0
).any():

    raise RuntimeError(
        "Βρέθηκε μη έγκυρος αριθμός heavy atoms."
    )


# ============================================================
# 4. Ανάγνωση των RAJ reference αποτελεσμάτων
# ============================================================

raj_summary_df = pd.read_csv(
    RAJ_VALIDATION_FILE
)

raj_summary_dictionary = dict(
    zip(
        raj_summary_df["Parameter"].astype(str),
        raj_summary_df["Value"]
    )
)


required_raj_parameters = [
    "Top-scoring affinity kcal/mol",
    "Heavy-atom count"
]

missing_raj_parameters = [
    parameter
    for parameter in required_raj_parameters
    if parameter not in raj_summary_dictionary
]

if missing_raj_parameters:

    raise KeyError(
        "Λείπουν από το RAJ validation summary:\n"
        f"{missing_raj_parameters}"
    )


raj_score = float(
    raj_summary_dictionary[
        "Top-scoring affinity kcal/mol"
    ]
)

raj_heavy_atom_count = int(
    float(
        raj_summary_dictionary[
            "Heavy-atom count"
        ]
    )
)

raj_ligand_efficiency = (
    -raj_score
    / raj_heavy_atom_count
)


# ============================================================
# 5. Υπολογισμός ligand efficiency
#
# Μεγαλύτερη θετική τιμή σημαίνει περισσότερη
# predicted binding energy ανά heavy atom.
# ============================================================

docking_df[
    "Ligand_Efficiency_kcal_mol_per_Heavy_Atom"
] = (
    -docking_df[
        "Top_SMINA_Affinity_kcal_mol"
    ]
    /
    docking_df[
        "Heavy_Atom_Count"
    ]
)


docking_df[
    "Ligand_Efficiency_Rank"
] = (
    docking_df[
        "Ligand_Efficiency_kcal_mol_per_Heavy_Atom"
    ]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


docking_df[
    "Higher_Ligand_Efficiency_Than_RAJ"
] = (
    docking_df[
        "Ligand_Efficiency_kcal_mol_per_Heavy_Atom"
    ]
    >
    raj_ligand_efficiency
)


# ============================================================
# 6. Προσδιορισμός top-5 ομάδων
#
# Δεν δημιουργείται weighted ή συνολικό score.
# Οι δύο κατατάξεις παραμένουν ανεξάρτητες.
# ============================================================

docking_df[
    "Top_5_By_Absolute_Docking_Score"
] = (
    docking_df[
        "Docking_Rank"
    ] <= 5
)


docking_df[
    "Top_5_By_Ligand_Efficiency"
] = (
    docking_df[
        "Ligand_Efficiency_Rank"
    ] <= 5
)


docking_df[
    "Present_In_Both_Top_5_Lists"
] = (
    docking_df[
        "Top_5_By_Absolute_Docking_Score"
    ]
    &
    docking_df[
        "Top_5_By_Ligand_Efficiency"
    ]
)


# ============================================================
# 7. Αποθήκευση πλήρους post-docking πίνακα
# ============================================================

post_docking_df = docking_df.sort_values(
    by=[
        "Docking_Rank",
        "Candidate_ID"
    ]
).reset_index(
    drop=True
)


post_docking_df.to_csv(
    POST_DOCKING_OUTPUT,
    index=False
)


# ============================================================
# 8. Top-5 με βάση το απόλυτο docking score
# ============================================================

top_score_df = (
    docking_df
    .sort_values(
        by=[
            "Top_SMINA_Affinity_kcal_mol",
            "Candidate_ID"
        ],
        ascending=[
            True,
            True
        ]
    )
    .head(5)
    .copy()
)


top_score_df.to_csv(
    TOP_SCORE_OUTPUT,
    index=False
)


# ============================================================
# 9. Top-5 με βάση το ligand efficiency
# ============================================================

top_ligand_efficiency_df = (
    docking_df
    .sort_values(
        by=[
            "Ligand_Efficiency_kcal_mol_per_Heavy_Atom",
            "Candidate_ID"
        ],
        ascending=[
            False,
            True
        ]
    )
    .head(5)
    .copy()
)


top_ligand_efficiency_df.to_csv(
    TOP_LIGAND_EFFICIENCY_OUTPUT,
    index=False
)


# ============================================================
# 10. Priority review set
#
# Περιλαμβάνει το union των δύο top-5 κατατάξεων.
# Δεν αποτελεί τελική επιλογή lead.
# ============================================================

priority_review_df = docking_df[
    docking_df[
        "Top_5_By_Absolute_Docking_Score"
    ]
    |
    docking_df[
        "Top_5_By_Ligand_Efficiency"
    ]
].copy()


priority_review_df = priority_review_df.sort_values(
    by=[
        "Present_In_Both_Top_5_Lists",
        "Docking_Rank",
        "Ligand_Efficiency_Rank"
    ],
    ascending=[
        False,
        True,
        True
    ]
).reset_index(
    drop=True
)


priority_review_df.to_csv(
    PRIORITY_REVIEW_OUTPUT,
    index=False
)


# ============================================================
# 11. Εντοπισμός προαιρετικών physicochemical columns
# ============================================================

def find_optional_column(
    dataframe,
    candidate_names
):
    """
    Επιστρέφει την πρώτη διαθέσιμη στήλη
    από μία λίστα πιθανών ονομάτων.
    """

    for candidate_name in candidate_names:

        if candidate_name in dataframe.columns:
            return candidate_name

    return None


optional_property_columns = {
    "Molecular_Weight": find_optional_column(
        docking_df,
        [
            "MW",
            "Molecular_Weight",
            "MolWt",
            "MolecularWeight"
        ]
    ),
    "QED": find_optional_column(
        docking_df,
        [
            "QED",
            "qed"
        ]
    ),
    "SAS": find_optional_column(
        docking_df,
        [
            "SAS",
            "SA_Score",
            "Synthetic_Accessibility_Score"
        ]
    ),
    "Absolute_LogP_Error": find_optional_column(
        docking_df,
        [
            "Abs_LogP_Error",
            "Absolute_LogP_Error",
            "Absolute_LogP_Difference",
            "abs_logp_error"
        ]
    ),
    "Rotatable_Bonds": find_optional_column(
        docking_df,
        [
            "RotB",
            "Rotatable_Bonds",
            "NumRotatableBonds"
        ]
    )
}


# ============================================================
# 12. Spearman correlations
#
# Οι συσχετίσεις είναι exploratory λόγω n=20.
#
# Επειδή τα καλύτερα scores είναι πιο αρνητικά:
# - αρνητικό rho: μεγαλύτερη property value συνδέεται
#   με πιο ευνοϊκό score,
# - θετικό rho: μεγαλύτερη property value συνδέεται
#   με λιγότερο ευνοϊκό score.
# ============================================================

correlation_properties = {
    "Heavy_Atom_Count": "Heavy_Atom_Count",
    "PDBQT_TORSDOF": "PDBQT_TORSDOF"
}


for property_label, column_name in (
    optional_property_columns.items()
):

    if column_name is not None:

        correlation_properties[
            property_label
        ] = column_name


correlation_rows = []


score_values = pd.to_numeric(
    docking_df[
        "Top_SMINA_Affinity_kcal_mol"
    ],
    errors="coerce"
)


for property_label, column_name in (
    correlation_properties.items()
):

    property_values = pd.to_numeric(
        docking_df[column_name],
        errors="coerce"
    )

    valid_mask = (
        score_values.notna()
        &
        property_values.notna()
    )

    valid_count = int(
        valid_mask.sum()
    )


    if valid_count < 3:

        spearman_rho = np.nan
        p_value = np.nan

    else:

        spearman_rho, p_value = spearmanr(
            property_values[
                valid_mask
            ],
            score_values[
                valid_mask
            ]
        )


    correlation_rows.append({
        "Property": property_label,
        "Source_Column": column_name,
        "Valid_Candidate_Count": valid_count,
        "Spearman_Rho_vs_Docking_Score":
            spearman_rho,
        "P_Value_Exploratory": p_value,
        "Interpretation_Note": (
            "Exploratory correlation; n=20. "
            "More negative docking score is considered "
            "numerically more favorable."
        )
    })


correlation_df = pd.DataFrame(
    correlation_rows
)


correlation_df.to_csv(
    CORRELATION_OUTPUT,
    index=False
)


# ============================================================
# 13. Συνοπτικά αποτελέσματα
# ============================================================

best_score_row = (
    docking_df
    .sort_values(
        by="Top_SMINA_Affinity_kcal_mol",
        ascending=True
    )
    .iloc[0]
)


best_ligand_efficiency_row = (
    docking_df
    .sort_values(
        by=(
            "Ligand_Efficiency_kcal_mol_per_Heavy_Atom"
        ),
        ascending=False
    )
    .iloc[0]
)


top_three_scores = (
    docking_df
    .sort_values(
        by="Top_SMINA_Affinity_kcal_mol",
        ascending=True
    )
    .head(3)[
        "Top_SMINA_Affinity_kcal_mol"
    ]
)


top_three_score_range = float(
    top_three_scores.max()
    - top_three_scores.min()
)


higher_efficiency_than_raj_count = int(
    docking_df[
        "Higher_Ligand_Efficiency_Than_RAJ"
    ].sum()
)


overlap_candidate_ids = (
    docking_df.loc[
        docking_df[
            "Present_In_Both_Top_5_Lists"
        ],
        "Candidate_ID"
    ]
    .astype(str)
    .tolist()
)


summary_df = pd.DataFrame([
    {
        "Parameter": "Generated candidate count",
        "Value": 20
    },
    {
        "Parameter": "Best absolute docking candidate",
        "Value": str(
            best_score_row["Candidate_ID"]
        )
    },
    {
        "Parameter": "Best generated docking score kcal/mol",
        "Value": float(
            best_score_row[
                "Top_SMINA_Affinity_kcal_mol"
            ]
        )
    },
    {
        "Parameter": "Top-three score range kcal/mol",
        "Value": top_three_score_range
    },
    {
        "Parameter": "RAJ docking score kcal/mol",
        "Value": raj_score
    },
    {
        "Parameter": "RAJ heavy-atom count",
        "Value": raj_heavy_atom_count
    },
    {
        "Parameter": (
            "RAJ ligand efficiency kcal/mol/heavy atom"
        ),
        "Value": raj_ligand_efficiency
    },
    {
        "Parameter": "Best ligand-efficiency candidate",
        "Value": str(
            best_ligand_efficiency_row[
                "Candidate_ID"
            ]
        )
    },
    {
        "Parameter": (
            "Best generated ligand efficiency "
            "kcal/mol/heavy atom"
        ),
        "Value": float(
            best_ligand_efficiency_row[
                "Ligand_Efficiency_kcal_mol_per_Heavy_Atom"
            ]
        )
    },
    {
        "Parameter": (
            "Generated candidates with higher "
            "ligand efficiency than RAJ"
        ),
        "Value": higher_efficiency_than_raj_count
    },
    {
        "Parameter": (
            "Candidates appearing in both top-5 lists"
        ),
        "Value": ", ".join(
            overlap_candidate_ids
        )
    },
    {
        "Parameter": "Final lead selected",
        "Value": False
    }
])


summary_df.to_csv(
    SUMMARY_OUTPUT,
    index=False
)


# ============================================================
# 14. Διάγραμμα docking scores
# ============================================================

score_plot_df = docking_df.sort_values(
    by="Top_SMINA_Affinity_kcal_mol",
    ascending=True
).reset_index(
    drop=True
)


figure, axis = plt.subplots(
    figsize=(11, 8)
)


axis.barh(
    score_plot_df["Candidate_ID"],
    score_plot_df[
        "Top_SMINA_Affinity_kcal_mol"
    ]
)


axis.axvline(
    raj_score,
    linestyle="--",
    label=(
        f"RAJ reference: "
        f"{raj_score:.2f} kcal/mol"
    )
)


axis.invert_yaxis()

axis.set_xlabel(
    "Top SMINA affinity (kcal/mol)"
)

axis.set_ylabel(
    "Generated candidate"
)

axis.set_title(
    "SMINA docking scores of the 20 generated candidates"
)

axis.legend()

figure.tight_layout()

figure.savefig(
    DOCKING_SCORE_PLOT,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.close(
    figure
)


# ============================================================
# 15. Διάγραμμα ligand efficiency
# ============================================================

efficiency_plot_df = docking_df.sort_values(
    by=(
        "Ligand_Efficiency_kcal_mol_per_Heavy_Atom"
    ),
    ascending=False
).reset_index(
    drop=True
)


figure, axis = plt.subplots(
    figsize=(11, 8)
)


axis.barh(
    efficiency_plot_df["Candidate_ID"],
    efficiency_plot_df[
        "Ligand_Efficiency_kcal_mol_per_Heavy_Atom"
    ]
)


axis.axvline(
    raj_ligand_efficiency,
    linestyle="--",
    label=(
        f"RAJ ligand efficiency: "
        f"{raj_ligand_efficiency:.3f}"
    )
)


axis.invert_yaxis()

axis.set_xlabel(
    "Ligand efficiency "
    "(−score / heavy atoms)"
)

axis.set_ylabel(
    "Generated candidate"
)

axis.set_title(
    "Ligand-efficiency ranking of generated candidates"
)

axis.legend()

figure.tight_layout()

figure.savefig(
    LIGAND_EFFICIENCY_PLOT,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.close(
    figure
)


# ============================================================
# 16. Docking score έναντι heavy-atom count
# ============================================================

figure, axis = plt.subplots(
    figsize=(10, 7)
)


axis.scatter(
    docking_df["Heavy_Atom_Count"],
    docking_df[
        "Top_SMINA_Affinity_kcal_mol"
    ],
    s=70
)


for _, candidate_row in docking_df.iterrows():

    axis.annotate(
        str(
            candidate_row["Candidate_ID"]
        ),
        (
            candidate_row[
                "Heavy_Atom_Count"
            ],
            candidate_row[
                "Top_SMINA_Affinity_kcal_mol"
            ]
        ),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=8
    )


axis.axhline(
    raj_score,
    linestyle="--",
    label=(
        f"RAJ score: "
        f"{raj_score:.2f} kcal/mol"
    )
)


axis.axvline(
    raj_heavy_atom_count,
    linestyle=":",
    label=(
        f"RAJ heavy atoms: "
        f"{raj_heavy_atom_count}"
    )
)


axis.set_xlabel(
    "Heavy-atom count"
)

axis.set_ylabel(
    "Top SMINA affinity (kcal/mol)"
)

axis.set_title(
    "Docking score versus molecular size"
)

axis.legend()

figure.tight_layout()

figure.savefig(
    SCORE_VS_HEAVY_ATOMS_PLOT,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.close(
    figure
)


# ============================================================
# 17. Εμφάνιση αποτελεσμάτων
# ============================================================

display_columns = [
    "Docking_Rank",
    "Ligand_Efficiency_Rank",
    "Candidate_ID",
    "Top_SMINA_Affinity_kcal_mol",
    "Heavy_Atom_Count",
    "Ligand_Efficiency_kcal_mol_per_Heavy_Atom",
    "Higher_Ligand_Efficiency_Than_RAJ",
    "Stereo_Only_3D_Assignment"
]


print("\n" + "=" * 110)
print("TOP 5 BY ABSOLUTE DOCKING SCORE")
print("=" * 110)

display(
    top_score_df[
        display_columns
    ]
)


print("\n" + "=" * 110)
print("TOP 5 BY LIGAND EFFICIENCY")
print("=" * 110)

display(
    top_ligand_efficiency_df[
        display_columns
    ]
)


print("\n" + "=" * 110)
print("PRIORITY REVIEW SET — UNION OF BOTH TOP-5 LISTS")
print("=" * 110)

display(
    priority_review_df[
        display_columns
        + [
            "Present_In_Both_Top_5_Lists"
        ]
    ]
)


print("\n" + "=" * 110)
print("EXPLORATORY SPEARMAN CORRELATIONS")
print("=" * 110)

display(
    correlation_df
)


print("\n" + "=" * 110)
print("POST-DOCKING SUMMARY")
print("=" * 110)

display(
    summary_df
)


# ============================================================
# 18. Τελική αναφορά
# ============================================================

print("\n" + "-" * 110)
print("Η POST-DOCKING ΑΝΑΛΥΣΗ ΟΛΟΚΛΗΡΩΘΗΚΕ")
print("-" * 110)

print(
    f"Best absolute score: "
    f"{best_score_row['Candidate_ID']} | "
    f"{best_score_row['Top_SMINA_Affinity_kcal_mol']:.4f} "
    "kcal/mol"
)

print(
    f"Best ligand efficiency: "
    f"{best_ligand_efficiency_row['Candidate_ID']} | "
    f"{best_ligand_efficiency_row['Ligand_Efficiency_kcal_mol_per_Heavy_Atom']:.4f}"
)

print(
    f"RAJ ligand efficiency: "
    f"{raj_ligand_efficiency:.4f}"
)

print(
    f"Candidates with higher ligand efficiency than RAJ: "
    f"{higher_efficiency_than_raj_count}"
)

print(
    "Candidates in both top-5 lists: "
    + (
        ", ".join(overlap_candidate_ids)
        if overlap_candidate_ids
        else "None"
    )
)

print(
    "\nΔεν επιλέχθηκε ακόμη τελικό lead molecule."
)

In [ ]:
# ============================================================
# ΒΗΜΑ 7D.4:
# Pareto analysis των docking αποτελεσμάτων
#
# Κριτήρια:
# 1. Ελαχιστοποίηση του SMINA docking score
#    (πιο αρνητικό = αριθμητικά ευνοϊκότερο)
#
# 2. Μεγιστοποίηση του ligand efficiency
#    (-score / αριθμός heavy atoms)
#
# Δεν χρησιμοποιείται:
# - weighted συνολικό score,
# - αυθαίρετος συνδυασμός βαρών,
# - αλλαγή των docking αποτελεσμάτων,
# - τελικός βιολογικός χαρακτηρισμός.
# ============================================================

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display


# ============================================================
# 1. Βασικές διαδρομές
# ============================================================

PROJECT_DIRECTORY = Path(
    "/home/afroditi/ptixiaki"
)

os.chdir(
    PROJECT_DIRECTORY
)

POST_DOCKING_FILE = Path(
    "SELECTED_20_POST_DOCKING_ANALYSIS.csv"
)

RAJ_VALIDATION_FILE = Path(
    "3BE2_RAJ_REDOCKING_VALIDATION_SUMMARY.csv"
)

PARETO_ALL_LAYERS_OUTPUT = Path(
    "SELECTED_20_DOCKING_PARETO_LAYERS.csv"
)

PARETO_FRONT_OUTPUT = Path(
    "SELECTED_20_DOCKING_PARETO_FRONT.csv"
)

PARETO_SUMMARY_OUTPUT = Path(
    "SELECTED_20_DOCKING_PARETO_SUMMARY.csv"
)

PARETO_PLOT_OUTPUT = Path(
    "SELECTED_20_DOCKING_PARETO_FRONT.png"
)


# ============================================================
# 2. Έλεγχος αρχείων
# ============================================================

required_files = [
    POST_DOCKING_FILE,
    RAJ_VALIDATION_FILE
]

missing_files = [
    str(file_path)
    for file_path in required_files
    if not file_path.exists()
]

if missing_files:

    raise FileNotFoundError(
        "Λείπουν τα ακόλουθα αρχεία:\n"
        f"{missing_files}"
    )


# ============================================================
# 3. Ανάγνωση post-docking αποτελεσμάτων
# ============================================================

docking_df = pd.read_csv(
    POST_DOCKING_FILE
)


required_columns = [
    "Candidate_ID",
    "Docking_Rank",
    "Top_SMINA_Affinity_kcal_mol",
    "Heavy_Atom_Count",
    "Ligand_Efficiency_kcal_mol_per_Heavy_Atom"
]


missing_columns = [
    column
    for column in required_columns
    if column not in docking_df.columns
]


if missing_columns:

    raise KeyError(
        "Λείπουν οι ακόλουθες στήλες:\n"
        f"{missing_columns}"
    )


if len(docking_df) != 20:

    raise RuntimeError(
        f"Αναμένονταν 20 candidates, "
        f"αλλά βρέθηκαν {len(docking_df)}."
    )


numeric_columns = [
    "Docking_Rank",
    "Top_SMINA_Affinity_kcal_mol",
    "Heavy_Atom_Count",
    "Ligand_Efficiency_kcal_mol_per_Heavy_Atom"
]


for column in numeric_columns:

    docking_df[column] = pd.to_numeric(
        docking_df[column],
        errors="raise"
    )


if docking_df["Candidate_ID"].nunique() != 20:

    raise RuntimeError(
        "Τα Candidate_ID δεν είναι μοναδικά."
    )


# ============================================================
# 4. Ανάγνωση των RAJ reference τιμών
# ============================================================

raj_summary_df = pd.read_csv(
    RAJ_VALIDATION_FILE
)


raj_values = dict(
    zip(
        raj_summary_df["Parameter"].astype(str),
        raj_summary_df["Value"]
    )
)


raj_score = float(
    raj_values[
        "Top-scoring affinity kcal/mol"
    ]
)

raj_heavy_atoms = int(
    float(
        raj_values[
            "Heavy-atom count"
        ]
    )
)

raj_ligand_efficiency = (
    -raj_score
    / raj_heavy_atoms
)


# ============================================================
# 5. Συνάρτηση εντοπισμού μη κυριαρχούμενων μορίων
#
# Το molecule A κυριαρχεί το B όταν:
# - το score του A είναι μικρότερο ή ίσο από του B,
# - το ligand efficiency του A είναι μεγαλύτερο ή ίσο,
# - και τουλάχιστον ένα από τα δύο είναι αυστηρά καλύτερο.
# ============================================================

def find_pareto_front_indices(
    dataframe
):
    """
    Επιστρέφει τα indices των μη κυριαρχούμενων candidates.
    """

    dataframe_indices = list(
        dataframe.index
    )

    pareto_indices = []


    for current_index in dataframe_indices:

        current_score = float(
            dataframe.loc[
                current_index,
                "Top_SMINA_Affinity_kcal_mol"
            ]
        )

        current_efficiency = float(
            dataframe.loc[
                current_index,
                (
                    "Ligand_Efficiency_"
                    "kcal_mol_per_Heavy_Atom"
                )
            ]
        )

        is_dominated = False


        for comparison_index in dataframe_indices:

            if comparison_index == current_index:
                continue


            comparison_score = float(
                dataframe.loc[
                    comparison_index,
                    "Top_SMINA_Affinity_kcal_mol"
                ]
            )

            comparison_efficiency = float(
                dataframe.loc[
                    comparison_index,
                    (
                        "Ligand_Efficiency_"
                        "kcal_mol_per_Heavy_Atom"
                    )
                ]
            )


            score_not_worse = (
                comparison_score
                <= current_score
            )

            efficiency_not_worse = (
                comparison_efficiency
                >= current_efficiency
            )

            at_least_one_strictly_better = (
                comparison_score
                < current_score
                or
                comparison_efficiency
                > current_efficiency
            )


            if (
                score_not_worse
                and efficiency_not_worse
                and at_least_one_strictly_better
            ):

                is_dominated = True
                break


        if not is_dominated:

            pareto_indices.append(
                current_index
            )


    return pareto_indices


# ============================================================
# 6. Υπολογισμός διαδοχικών Pareto layers
# ============================================================

remaining_df = docking_df.copy()

pareto_layer_assignments = {}

current_layer = 1


while not remaining_df.empty:

    current_front_indices = (
        find_pareto_front_indices(
            remaining_df
        )
    )


    if not current_front_indices:

        raise RuntimeError(
            "Δεν ήταν δυνατός ο υπολογισμός "
            "του επόμενου Pareto layer."
        )


    for dataframe_index in current_front_indices:

        candidate_id = str(
            remaining_df.loc[
                dataframe_index,
                "Candidate_ID"
            ]
        )

        pareto_layer_assignments[
            candidate_id
        ] = current_layer


    remaining_df = remaining_df.drop(
        index=current_front_indices
    )


    current_layer += 1


docking_df[
    "Docking_Pareto_Layer"
] = (
    docking_df["Candidate_ID"]
    .map(
        pareto_layer_assignments
    )
    .astype(int)
)


docking_df[
    "On_First_Pareto_Front"
] = (
    docking_df[
        "Docking_Pareto_Layer"
    ] == 1
)


# ============================================================
# 7. Πρόσθετα περιγραφικά μεγέθη
#
# Δεν χρησιμοποιούνται για τον υπολογισμό του Pareto front.
# ============================================================

best_generated_score = float(
    docking_df[
        "Top_SMINA_Affinity_kcal_mol"
    ].min()
)


best_generated_efficiency = float(
    docking_df[
        "Ligand_Efficiency_kcal_mol_per_Heavy_Atom"
    ].max()
)


docking_df[
    "Score_Difference_From_Best_Generated_kcal_mol"
] = (
    docking_df[
        "Top_SMINA_Affinity_kcal_mol"
    ]
    - best_generated_score
)


docking_df[
    "Ligand_Efficiency_Difference_From_Best"
] = (
    best_generated_efficiency
    -
    docking_df[
        "Ligand_Efficiency_kcal_mol_per_Heavy_Atom"
    ]
)


docking_df[
    "Higher_Ligand_Efficiency_Than_RAJ"
] = (
    docking_df[
        "Ligand_Efficiency_kcal_mol_per_Heavy_Atom"
    ]
    >
    raj_ligand_efficiency
)


# ============================================================
# 8. Δημιουργία πρώτου Pareto front
# ============================================================

pareto_front_df = docking_df[
    docking_df[
        "On_First_Pareto_Front"
    ]
].copy()


pareto_front_df = pareto_front_df.sort_values(
    by=[
        "Top_SMINA_Affinity_kcal_mol",
        "Ligand_Efficiency_kcal_mol_per_Heavy_Atom",
        "Candidate_ID"
    ],
    ascending=[
        True,
        False,
        True
    ]
).reset_index(
    drop=True
)


pareto_front_df.insert(
    0,
    "Pareto_Display_Order",
    range(
        1,
        len(pareto_front_df) + 1
    )
)


# ============================================================
# 9. Αποθήκευση πινάκων
# ============================================================

all_layers_df = docking_df.sort_values(
    by=[
        "Docking_Pareto_Layer",
        "Top_SMINA_Affinity_kcal_mol",
        "Ligand_Efficiency_kcal_mol_per_Heavy_Atom"
    ],
    ascending=[
        True,
        True,
        False
    ]
).reset_index(
    drop=True
)


all_layers_df.to_csv(
    PARETO_ALL_LAYERS_OUTPUT,
    index=False
)


pareto_front_df.to_csv(
    PARETO_FRONT_OUTPUT,
    index=False
)


# ============================================================
# 10. Δημιουργία Pareto summary
# ============================================================

pareto_candidate_ids = (
    pareto_front_df[
        "Candidate_ID"
    ]
    .astype(str)
    .tolist()
)


candidates_above_raj_efficiency = int(
    docking_df[
        "Higher_Ligand_Efficiency_Than_RAJ"
    ].sum()
)


summary_df = pd.DataFrame(
    [
        {
            "Parameter": "Total generated candidates",
            "Value": "20"
        },
        {
            "Parameter": "Pareto-front candidate count",
            "Value": str(
                len(pareto_front_df)
            )
        },
        {
            "Parameter": "Pareto-front candidates",
            "Value": ", ".join(
                pareto_candidate_ids
            )
        },
        {
            "Parameter": "Best absolute-score candidate",
            "Value": str(
                docking_df.loc[
                    docking_df[
                        "Top_SMINA_Affinity_kcal_mol"
                    ].idxmin(),
                    "Candidate_ID"
                ]
            )
        },
        {
            "Parameter": "Best ligand-efficiency candidate",
            "Value": str(
                docking_df.loc[
                    docking_df[
                        (
                            "Ligand_Efficiency_"
                            "kcal_mol_per_Heavy_Atom"
                        )
                    ].idxmax(),
                    "Candidate_ID"
                ]
            )
        },
        {
            "Parameter": "RAJ docking score kcal/mol",
            "Value": str(
                raj_score
            )
        },
        {
            "Parameter": "RAJ heavy atoms",
            "Value": str(
                raj_heavy_atoms
            )
        },
        {
            "Parameter": "RAJ ligand efficiency",
            "Value": str(
                raj_ligand_efficiency
            )
        },
        {
            "Parameter": (
                "Generated candidates with higher "
                "ligand efficiency than RAJ"
            ),
            "Value": str(
                candidates_above_raj_efficiency
            )
        },
        {
            "Parameter": "Weighted combined score used",
            "Value": "False"
        },
        {
            "Parameter": "Final biological lead selected",
            "Value": "False"
        }
    ],
    dtype="object"
)


summary_df.to_csv(
    PARETO_SUMMARY_OUTPUT,
    index=False
)


# ============================================================
# 11. Διάγραμμα Pareto front
#
# X: ligand efficiency — μεγαλύτερο είναι καλύτερο
# Y: docking score — μικρότερο/πιο αρνητικό είναι καλύτερο
# ============================================================

non_pareto_df = docking_df[
    ~docking_df[
        "On_First_Pareto_Front"
    ]
]


figure, axis = plt.subplots(
    figsize=(11, 8)
)


axis.scatter(
    non_pareto_df[
        "Ligand_Efficiency_kcal_mol_per_Heavy_Atom"
    ],
    non_pareto_df[
        "Top_SMINA_Affinity_kcal_mol"
    ],
    s=65,
    label="Dominated candidates"
)


axis.scatter(
    pareto_front_df[
        "Ligand_Efficiency_kcal_mol_per_Heavy_Atom"
    ],
    pareto_front_df[
        "Top_SMINA_Affinity_kcal_mol"
    ],
    s=110,
    marker="D",
    label="First Pareto front"
)


axis.scatter(
    [
        raj_ligand_efficiency
    ],
    [
        raj_score
    ],
    s=130,
    marker="*",
    label="RAJ reference"
)


for _, candidate_row in docking_df.iterrows():

    axis.annotate(
        str(
            candidate_row["Candidate_ID"]
        ),
        (
            candidate_row[
                "Ligand_Efficiency_kcal_mol_per_Heavy_Atom"
            ],
            candidate_row[
                "Top_SMINA_Affinity_kcal_mol"
            ]
        ),
        xytext=(5, 4),
        textcoords="offset points",
        fontsize=8
    )


axis.set_xlabel(
    "Ligand efficiency (−SMINA score / heavy atoms)"
)

axis.set_ylabel(
    "Top SMINA affinity (kcal/mol)"
)

axis.set_title(
    "Pareto analysis of docking score and ligand efficiency"
)

axis.legend()

figure.tight_layout()

figure.savefig(
    PARETO_PLOT_OUTPUT,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.close(
    figure
)


# ============================================================
# 12. Εμφάνιση αποτελεσμάτων
# ============================================================

display_columns = [
    "Pareto_Display_Order",
    "Candidate_ID",
    "Docking_Rank",
    "Top_SMINA_Affinity_kcal_mol",
    "Heavy_Atom_Count",
    "Ligand_Efficiency_kcal_mol_per_Heavy_Atom",
    "Ligand_Efficiency_Rank",
    "Stereo_Only_3D_Assignment"
]


print("\n" + "=" * 110)
print("FIRST DOCKING PARETO FRONT")
print("=" * 110)

display(
    pareto_front_df[
        [
            column
            for column in display_columns
            if column in pareto_front_df.columns
        ]
    ]
)


print("\n" + "=" * 110)
print("DOCKING PARETO SUMMARY")
print("=" * 110)

display(
    summary_df
)


# ============================================================
# 13. Τελική αναφορά
# ============================================================

print("\n" + "-" * 110)
print("Η DOCKING PARETO ANALYSIS ΟΛΟΚΛΗΡΩΘΗΚΕ")
print("-" * 110)

print(
    f"Pareto-front candidates: "
    f"{', '.join(pareto_candidate_ids)}"
)

print(
    f"Pareto-front size: "
    f"{len(pareto_front_df)}"
)

print(
    f"Candidates with ligand efficiency "
    f"higher than RAJ: "
    f"{candidates_above_raj_efficiency}/20"
)

print(
    "\nΔεν χρησιμοποιήθηκε weighted score "
    "και δεν επιλέχθηκε ακόμη τελικό lead."
)

In [ ]:
# ============================================================
# ΒΗΜΑ 7D.5:
# Interaction analysis των Pareto docking candidates
#
# Αναλύονται:
# - residues με heavy-atom απόσταση <= 4.0 Å,
# - πιθανοί δεσμοί υδρογόνου βάσει PDBQT atom types,
# - πιθανές στερεοχημικές συγκρούσεις,
# - τρισδιάστατες poses σε σύγκριση με τον RAJ.
#
# Δεν πραγματοποιείται νέο docking.
# Δεν αλλάζει καμία pose ή παράμετρος.
# Δεν επιλέγεται ακόμη τελικό lead.
# ============================================================

import os
import math
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image as PILImage
from PIL import ImageDraw, ImageOps

from IPython.display import display
from IPython.display import Image as IPythonImage


# ============================================================
# 1. Βασικές διαδρομές
# ============================================================

PROJECT_DIRECTORY = Path(
    "/home/afroditi/ptixiaki"
)

os.chdir(
    PROJECT_DIRECTORY
)

PARETO_FRONT_FILE = Path(
    "SELECTED_20_DOCKING_PARETO_FRONT.csv"
)

DOCKING_RANKING_FILE = Path(
    "SELECTED_20_DOCKING_RANKING.csv"
)

RAJ_VALIDATION_FILE = Path(
    "3BE2_RAJ_REDOCKING_VALIDATION_SUMMARY.csv"
)

RECEPTOR_PDBQT = Path(
    "3BE2_RECEPTOR_PREPARED.pdbqt"
)

RECEPTOR_PDB = Path(
    "3BE2_RECEPTOR_MEEKO_PREPARED.pdb"
)

RAJ_REFERENCE_PDBQT = Path(
    "3BE2_RAJ_REFERENCE_CRYSTAL.pdbqt"
)

RAJ_REFERENCE_SDF = Path(
    "3BE2_RAJ_REFERENCE_CRYSTAL_HEAVY.sdf"
)

DOCKING_RESULTS_DIRECTORY = Path(
    "SELECTED_20_DOCKING_RESULTS"
)

OUTPUT_DIRECTORY = Path(
    "PARETO_4_INTERACTION_ANALYSIS"
)

CONTACT_RESIDUES_OUTPUT = Path(
    "PARETO_4_PLUS_RAJ_CONTACT_RESIDUES.csv"
)

HYDROGEN_BONDS_OUTPUT = Path(
    "PARETO_4_PLUS_RAJ_GEOMETRIC_HYDROGEN_BONDS.csv"
)

INTERACTION_SUMMARY_OUTPUT = Path(
    "PARETO_4_PLUS_RAJ_INTERACTION_SUMMARY.csv"
)

INTERACTION_METADATA_OUTPUT = Path(
    "PARETO_4_INTERACTION_METADATA.csv"
)

IMAGE_GRID_OUTPUT = Path(
    "PARETO_4_DOCKING_POSES_GRID.png"
)

PYMOL_LOG_OUTPUT = Path(
    "PARETO_4_PYMOL_VISUALIZATION_LOG.txt"
)


# ============================================================
# 2. Προκαθορισμένα γεωμετρικά όρια
# ============================================================

CONTACT_DISTANCE_ANGSTROM = 4.0

SEVERE_CLASH_DISTANCE_ANGSTROM = 1.5

HBOND_HYDROGEN_ACCEPTOR_MAX = 2.5

HBOND_DONOR_ACCEPTOR_MAX = 3.5

HBOND_MINIMUM_ANGLE_DEGREES = 120.0

EXPECTED_PARETO_COUNT = 4


# ============================================================
# 3. Έλεγχος input αρχείων και PyMOL
# ============================================================

required_files = [
    PARETO_FRONT_FILE,
    DOCKING_RANKING_FILE,
    RAJ_VALIDATION_FILE,
    RECEPTOR_PDBQT,
    RECEPTOR_PDB,
    RAJ_REFERENCE_PDBQT,
    RAJ_REFERENCE_SDF
]

missing_files = [
    str(file_path)
    for file_path in required_files
    if not file_path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Λείπουν τα ακόλουθα αρχεία:\n"
        f"{missing_files}"
    )


pymol_path = shutil.which(
    "pymol"
)

if pymol_path is None:
    raise RuntimeError(
        "Δεν βρέθηκε το PyMOL στο docking environment."
    )


OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 4. Ανάγνωση του Pareto front
# ============================================================

pareto_df = pd.read_csv(
    PARETO_FRONT_FILE
)

if len(pareto_df) != EXPECTED_PARETO_COUNT:
    raise RuntimeError(
        f"Αναμένονταν {EXPECTED_PARETO_COUNT} Pareto candidates, "
        f"αλλά βρέθηκαν {len(pareto_df)}."
    )


pareto_candidate_ids = (
    pareto_df["Candidate_ID"]
    .astype(str)
    .tolist()
)


print("=" * 100)
print("PARETO CANDIDATES ΓΙΑ INTERACTION ANALYSIS")
print("=" * 100)

print(
    ", ".join(
        pareto_candidate_ids
    )
)


# ============================================================
# 5. Ανάγνωση docking ranking και RAJ score
# ============================================================

ranking_df = pd.read_csv(
    DOCKING_RANKING_FILE
)

raj_validation_df = pd.read_csv(
    RAJ_VALIDATION_FILE
)

raj_values = dict(
    zip(
        raj_validation_df["Parameter"].astype(str),
        raj_validation_df["Value"]
    )
)

raj_score = float(
    raj_values[
        "Top-scoring affinity kcal/mol"
    ]
)


# ============================================================
# 6. Βοηθητικές συναρτήσεις ανάγνωσης PDBQT
# ============================================================

def infer_element_from_pdbqt_type(
    atom_type,
    atom_name
):
    """
    Εκτιμά το χημικό στοιχείο από τον AutoDock atom type.
    """

    atom_type = str(
        atom_type
    ).strip().upper()

    atom_name = str(
        atom_name
    ).strip().upper()

    if atom_type.startswith("H"):
        return "H"

    if atom_type == "CL":
        return "CL"

    if atom_type == "BR":
        return "BR"

    if atom_type.startswith("O"):
        return "O"

    if atom_type.startswith("N"):
        return "N"

    if atom_type.startswith("S"):
        return "S"

    if atom_type.startswith("P"):
        return "P"

    if atom_type in {"C", "A"}:
        return "C"

    letters = "".join(
        character
        for character in atom_name
        if character.isalpha()
    )

    return (
        letters[0]
        if letters
        else ""
    )


def parse_pdbqt(
    pdbqt_path,
    source_name
):
    """
    Διαβάζει τα ATOM/HETATM records ενός PDBQT.
    """

    atom_rows = []

    lines = pdbqt_path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    for line in lines:

        if not line.startswith(
            ("ATOM", "HETATM")
        ):
            continue

        if len(line) < 54:
            continue

        try:
            x = float(
                line[30:38]
            )

            y = float(
                line[38:46]
            )

            z = float(
                line[46:54]
            )

        except ValueError:
            continue


        split_fields = line.split()

        atom_type = (
            split_fields[-1]
            if split_fields
            else ""
        )


        try:
            atom_serial = int(
                line[6:11]
            )
        except ValueError:
            atom_serial = len(atom_rows) + 1


        atom_name = line[
            12:16
        ].strip()

        residue_name = line[
            17:20
        ].strip()

        chain_id = (
            line[21:22].strip()
            or "_"
        )

        residue_number = line[
            22:26
        ].strip()

        insertion_code = line[
            26:27
        ].strip()


        element = infer_element_from_pdbqt_type(
            atom_type,
            atom_name
        )


        atom_rows.append({
            "Source": source_name,
            "Atom_Serial": atom_serial,
            "Atom_Name": atom_name,
            "Residue_Name": residue_name,
            "Chain_ID": chain_id,
            "Residue_Number": residue_number,
            "Insertion_Code": insertion_code,
            "X": x,
            "Y": y,
            "Z": z,
            "Atom_Type": atom_type,
            "Element": element
        })


    atom_df = pd.DataFrame(
        atom_rows
    )

    if atom_df.empty:
        raise RuntimeError(
            f"Δεν βρέθηκαν atoms στο {pdbqt_path}."
        )

    return atom_df


# ============================================================
# 7. Βοηθητικές συναρτήσεις γεωμετρίας
# ============================================================

def euclidean_distance(
    first_coordinates,
    second_coordinates
):
    """
    Ευκλείδεια απόσταση δύο τρισδιάστατων σημείων.
    """

    return float(
        np.linalg.norm(
            np.asarray(first_coordinates, dtype=float)
            -
            np.asarray(second_coordinates, dtype=float)
        )
    )


def calculate_angle_degrees(
    donor_coordinates,
    hydrogen_coordinates,
    acceptor_coordinates
):
    """
    Υπολογίζει τη γωνία D-H-A με κορυφή το hydrogen.
    """

    vector_hd = (
        np.asarray(donor_coordinates, dtype=float)
        -
        np.asarray(hydrogen_coordinates, dtype=float)
    )

    vector_ha = (
        np.asarray(acceptor_coordinates, dtype=float)
        -
        np.asarray(hydrogen_coordinates, dtype=float)
    )

    denominator = (
        np.linalg.norm(vector_hd)
        *
        np.linalg.norm(vector_ha)
    )

    if denominator == 0:
        return np.nan


    cosine_value = float(
        np.dot(
            vector_hd,
            vector_ha
        )
        /
        denominator
    )

    cosine_value = np.clip(
        cosine_value,
        -1.0,
        1.0
    )

    return float(
        np.degrees(
            np.arccos(
                cosine_value
            )
        )
    )


# ============================================================
# 8. Εντοπισμός donor hydrogens
# ============================================================

def infer_donor_hydrogens(
    atom_df
):
    """
    Εντοπίζει polar hydrogens και το συνδεδεμένο
    donor heavy atom.

    Τα PDBQT polar hydrogens έχουν συνήθως atom type HD.
    """

    hydrogen_df = atom_df[
        atom_df["Atom_Type"]
        .astype(str)
        .str.upper()
        .str.startswith("H")
    ].copy()


    heavy_df = atom_df[
        atom_df["Element"] != "H"
    ].copy()


    donor_rows = []


    for _, hydrogen_row in hydrogen_df.iterrows():

        same_residue_mask = (
            (heavy_df["Chain_ID"] == hydrogen_row["Chain_ID"])
            &
            (
                heavy_df["Residue_Number"]
                == hydrogen_row["Residue_Number"]
            )
            &
            (
                heavy_df["Insertion_Code"]
                == hydrogen_row["Insertion_Code"]
            )
        )


        candidate_heavy_df = heavy_df[
            same_residue_mask
            &
            heavy_df["Element"].isin(
                ["N", "O", "S"]
            )
        ].copy()


        if candidate_heavy_df.empty:
            continue


        hydrogen_coordinates = np.array([
            hydrogen_row["X"],
            hydrogen_row["Y"],
            hydrogen_row["Z"]
        ])


        candidate_heavy_df[
            "Distance_To_Hydrogen"
        ] = candidate_heavy_df.apply(
            lambda heavy_row: euclidean_distance(
                hydrogen_coordinates,
                [
                    heavy_row["X"],
                    heavy_row["Y"],
                    heavy_row["Z"]
                ]
            ),
            axis=1
        )


        candidate_heavy_df = (
            candidate_heavy_df
            .sort_values(
                by="Distance_To_Hydrogen"
            )
        )


        nearest_heavy_row = (
            candidate_heavy_df.iloc[0]
        )


        donor_h_distance = float(
            nearest_heavy_row[
                "Distance_To_Hydrogen"
            ]
        )


        if not (
            0.7
            <= donor_h_distance
            <= 1.35
        ):
            continue


        donor_rows.append({
            "Donor_Atom_Serial": int(
                nearest_heavy_row["Atom_Serial"]
            ),
            "Donor_Atom_Name": str(
                nearest_heavy_row["Atom_Name"]
            ),
            "Donor_Element": str(
                nearest_heavy_row["Element"]
            ),
            "Donor_X": float(
                nearest_heavy_row["X"]
            ),
            "Donor_Y": float(
                nearest_heavy_row["Y"]
            ),
            "Donor_Z": float(
                nearest_heavy_row["Z"]
            ),
            "Hydrogen_Atom_Serial": int(
                hydrogen_row["Atom_Serial"]
            ),
            "Hydrogen_Atom_Name": str(
                hydrogen_row["Atom_Name"]
            ),
            "Hydrogen_X": float(
                hydrogen_row["X"]
            ),
            "Hydrogen_Y": float(
                hydrogen_row["Y"]
            ),
            "Hydrogen_Z": float(
                hydrogen_row["Z"]
            ),
            "Residue_Name": str(
                nearest_heavy_row["Residue_Name"]
            ),
            "Chain_ID": str(
                nearest_heavy_row["Chain_ID"]
            ),
            "Residue_Number": str(
                nearest_heavy_row["Residue_Number"]
            ),
            "Insertion_Code": str(
                nearest_heavy_row["Insertion_Code"]
            )
        })


    return pd.DataFrame(
        donor_rows
    )


def get_acceptor_atoms(
    atom_df
):
    """
    Εντοπίζει acceptors βάσει AutoDock atom types.
    """

    acceptor_types = {
        "OA",
        "NA",
        "SA"
    }

    return atom_df[
        atom_df["Atom_Type"]
        .astype(str)
        .str.upper()
        .isin(
            acceptor_types
        )
    ].copy()


# ============================================================
# 9. Ανάλυση επαφών residue–ligand
# ============================================================

def calculate_contact_residues(
    receptor_atoms,
    ligand_atoms,
    ligand_id
):
    """
    Εντοπίζει receptor residues με heavy-atom επαφή <= 4 Å.
    """

    receptor_heavy = receptor_atoms[
        receptor_atoms["Element"] != "H"
    ].reset_index(
        drop=True
    )

    ligand_heavy = ligand_atoms[
        ligand_atoms["Element"] != "H"
    ].reset_index(
        drop=True
    )


    receptor_coordinates = receptor_heavy[
        ["X", "Y", "Z"]
    ].to_numpy(
        dtype=float
    )

    ligand_coordinates = ligand_heavy[
        ["X", "Y", "Z"]
    ].to_numpy(
        dtype=float
    )


    distance_matrix = np.linalg.norm(
        receptor_coordinates[
            :,
            np.newaxis,
            :
        ]
        -
        ligand_coordinates[
            np.newaxis,
            :,
            :
        ],
        axis=2
    )


    grouping_columns = [
        "Chain_ID",
        "Residue_Number",
        "Insertion_Code",
        "Residue_Name"
    ]


    contact_rows = []


    for residue_key, residue_indices in (
        receptor_heavy.groupby(
            grouping_columns,
            sort=True
        ).groups.items()
    ):

        residue_indices = list(
            residue_indices
        )

        residue_distances = distance_matrix[
            residue_indices,
            :
        ]


        minimum_distance = float(
            residue_distances.min()
        )


        if minimum_distance > CONTACT_DISTANCE_ANGSTROM:
            continue


        close_pair_count = int(
            (
                residue_distances
                <= CONTACT_DISTANCE_ANGSTROM
            ).sum()
        )


        (
            chain_id,
            residue_number,
            insertion_code,
            residue_name
        ) = residue_key


        contact_rows.append({
            "Ligand_ID": ligand_id,
            "Chain_ID": chain_id,
            "Residue_Number": residue_number,
            "Insertion_Code": insertion_code,
            "Residue_Name": residue_name,
            "Residue_Label": (
                f"{residue_name}"
                f"{residue_number}"
                f"{insertion_code}"
            ),
            "Minimum_Heavy_Atom_Distance_Angstrom":
                minimum_distance,
            "Heavy_Atom_Contact_Pair_Count":
                close_pair_count
        })


    severe_clash_pair_count = int(
        (
            distance_matrix
            < SEVERE_CLASH_DISTANCE_ANGSTROM
        ).sum()
    )


    minimum_receptor_ligand_distance = float(
        distance_matrix.min()
    )


    return (
        pd.DataFrame(
            contact_rows
        ),
        severe_clash_pair_count,
        minimum_receptor_ligand_distance
    )


# ============================================================
# 10. Ανάλυση πιθανών δεσμών υδρογόνου
# ============================================================

def calculate_hydrogen_bonds(
    receptor_atoms,
    ligand_atoms,
    ligand_id
):
    """
    Εντοπίζει γεωμετρικά συμβατούς δεσμούς υδρογόνου.

    Κριτήρια:
    H···A <= 2.5 Å
    D···A <= 3.5 Å
    D-H-A >= 120°
    """

    receptor_donors = infer_donor_hydrogens(
        receptor_atoms
    )

    ligand_donors = infer_donor_hydrogens(
        ligand_atoms
    )

    receptor_acceptors = get_acceptor_atoms(
        receptor_atoms
    )

    ligand_acceptors = get_acceptor_atoms(
        ligand_atoms
    )


    hbond_rows = []


    # --------------------------------------------------------
    # Ligand donor προς receptor acceptor
    # --------------------------------------------------------

    for _, donor_row in ligand_donors.iterrows():

        donor_coordinates = np.array([
            donor_row["Donor_X"],
            donor_row["Donor_Y"],
            donor_row["Donor_Z"]
        ])

        hydrogen_coordinates = np.array([
            donor_row["Hydrogen_X"],
            donor_row["Hydrogen_Y"],
            donor_row["Hydrogen_Z"]
        ])


        for _, acceptor_row in receptor_acceptors.iterrows():

            acceptor_coordinates = np.array([
                acceptor_row["X"],
                acceptor_row["Y"],
                acceptor_row["Z"]
            ])


            hydrogen_acceptor_distance = euclidean_distance(
                hydrogen_coordinates,
                acceptor_coordinates
            )

            donor_acceptor_distance = euclidean_distance(
                donor_coordinates,
                acceptor_coordinates
            )


            if (
                hydrogen_acceptor_distance
                > HBOND_HYDROGEN_ACCEPTOR_MAX
                or
                donor_acceptor_distance
                > HBOND_DONOR_ACCEPTOR_MAX
            ):
                continue


            angle_degrees = calculate_angle_degrees(
                donor_coordinates,
                hydrogen_coordinates,
                acceptor_coordinates
            )


            if (
                np.isnan(angle_degrees)
                or
                angle_degrees
                < HBOND_MINIMUM_ANGLE_DEGREES
            ):
                continue


            hbond_rows.append({
                "Ligand_ID": ligand_id,
                "Direction": (
                    "Ligand donor → receptor acceptor"
                ),
                "Receptor_Residue_Name":
                    acceptor_row["Residue_Name"],
                "Receptor_Chain_ID":
                    acceptor_row["Chain_ID"],
                "Receptor_Residue_Number":
                    acceptor_row["Residue_Number"],
                "Receptor_Atom_Name":
                    acceptor_row["Atom_Name"],
                "Ligand_Atom_Name":
                    donor_row["Donor_Atom_Name"],
                "Hydrogen_Acceptor_Distance_Angstrom":
                    hydrogen_acceptor_distance,
                "Donor_Acceptor_Distance_Angstrom":
                    donor_acceptor_distance,
                "D_H_A_Angle_Degrees":
                    angle_degrees
            })


    # --------------------------------------------------------
    # Receptor donor προς ligand acceptor
    # --------------------------------------------------------

    for _, donor_row in receptor_donors.iterrows():

        donor_coordinates = np.array([
            donor_row["Donor_X"],
            donor_row["Donor_Y"],
            donor_row["Donor_Z"]
        ])

        hydrogen_coordinates = np.array([
            donor_row["Hydrogen_X"],
            donor_row["Hydrogen_Y"],
            donor_row["Hydrogen_Z"]
        ])


        for _, acceptor_row in ligand_acceptors.iterrows():

            acceptor_coordinates = np.array([
                acceptor_row["X"],
                acceptor_row["Y"],
                acceptor_row["Z"]
            ])


            hydrogen_acceptor_distance = euclidean_distance(
                hydrogen_coordinates,
                acceptor_coordinates
            )

            donor_acceptor_distance = euclidean_distance(
                donor_coordinates,
                acceptor_coordinates
            )


            if (
                hydrogen_acceptor_distance
                > HBOND_HYDROGEN_ACCEPTOR_MAX
                or
                donor_acceptor_distance
                > HBOND_DONOR_ACCEPTOR_MAX
            ):
                continue


            angle_degrees = calculate_angle_degrees(
                donor_coordinates,
                hydrogen_coordinates,
                acceptor_coordinates
            )


            if (
                np.isnan(angle_degrees)
                or
                angle_degrees
                < HBOND_MINIMUM_ANGLE_DEGREES
            ):
                continue


            hbond_rows.append({
                "Ligand_ID": ligand_id,
                "Direction": (
                    "Receptor donor → ligand acceptor"
                ),
                "Receptor_Residue_Name":
                    donor_row["Residue_Name"],
                "Receptor_Chain_ID":
                    donor_row["Chain_ID"],
                "Receptor_Residue_Number":
                    donor_row["Residue_Number"],
                "Receptor_Atom_Name":
                    donor_row["Donor_Atom_Name"],
                "Ligand_Atom_Name":
                    acceptor_row["Atom_Name"],
                "Hydrogen_Acceptor_Distance_Angstrom":
                    hydrogen_acceptor_distance,
                "Donor_Acceptor_Distance_Angstrom":
                    donor_acceptor_distance,
                "D_H_A_Angle_Degrees":
                    angle_degrees
            })


    return pd.DataFrame(
        hbond_rows
    )


# ============================================================
# 11. Προετοιμασία των ligand entries
# ============================================================

ligand_entries = [
    {
        "Ligand_ID": "RAJ_REFERENCE",
        "Ligand_PDBQT": RAJ_REFERENCE_PDBQT,
        "Docking_Score": raj_score,
        "Is_Pareto_Candidate": False,
        "Stereo_Only_Assignment": False
    }
]


for candidate_id in pareto_candidate_ids:

    candidate_ranking_row = ranking_df[
        ranking_df["Candidate_ID"].astype(str)
        == candidate_id
    ]


    if len(candidate_ranking_row) != 1:
        raise RuntimeError(
            f"Δεν βρέθηκε μοναδική ranking row για "
            f"{candidate_id}."
        )


    candidate_ranking_row = (
        candidate_ranking_row.iloc[0]
    )


    candidate_pose_path = (
        DOCKING_RESULTS_DIRECTORY
        / candidate_id
        / f"{candidate_id}_TOP_POSE.pdbqt"
    )


    if (
        not candidate_pose_path.exists()
        or candidate_pose_path.stat().st_size == 0
    ):
        raise FileNotFoundError(
            f"Δεν βρέθηκε η top pose του {candidate_id}:\n"
            f"{candidate_pose_path}"
        )


    ligand_entries.append({
        "Ligand_ID": candidate_id,
        "Ligand_PDBQT": candidate_pose_path,
        "Docking_Score": float(
            candidate_ranking_row[
                "Top_SMINA_Affinity_kcal_mol"
            ]
        ),
        "Is_Pareto_Candidate": True,
        "Stereo_Only_Assignment": bool(
            candidate_ranking_row.get(
                "Stereo_Only_3D_Assignment",
                False
            )
        )
    })


# ============================================================
# 12. Εκτέλεση contact και hydrogen-bond analysis
# ============================================================

receptor_atoms_df = parse_pdbqt(
    RECEPTOR_PDBQT,
    source_name="3BE2_RECEPTOR"
)


all_contact_frames = []

all_hbond_frames = []

summary_rows = []


for ligand_entry in ligand_entries:

    ligand_id = ligand_entry[
        "Ligand_ID"
    ]

    ligand_atoms_df = parse_pdbqt(
        ligand_entry["Ligand_PDBQT"],
        source_name=ligand_id
    )


    (
        contact_df,
        severe_clash_count,
        minimum_distance
    ) = calculate_contact_residues(
        receptor_atoms_df,
        ligand_atoms_df,
        ligand_id
    )


    hbond_df = calculate_hydrogen_bonds(
        receptor_atoms_df,
        ligand_atoms_df,
        ligand_id
    )


    if not contact_df.empty:
        all_contact_frames.append(
            contact_df
        )


    if not hbond_df.empty:
        all_hbond_frames.append(
            hbond_df
        )


    contact_residue_labels = (
        contact_df
        .sort_values(
            by="Minimum_Heavy_Atom_Distance_Angstrom"
        )[
            "Residue_Label"
        ]
        .astype(str)
        .tolist()
        if not contact_df.empty
        else []
    )


    hbond_residue_labels = (
        hbond_df.apply(
            lambda row: (
                f"{row['Receptor_Residue_Name']}"
                f"{row['Receptor_Residue_Number']}"
            ),
            axis=1
        )
        .drop_duplicates()
        .tolist()
        if not hbond_df.empty
        else []
    )


    ligand_heavy_atom_count = int(
        (
            ligand_atoms_df["Element"]
            != "H"
        ).sum()
    )


    summary_rows.append({
        "Ligand_ID": ligand_id,
        "Is_Pareto_Candidate":
            ligand_entry["Is_Pareto_Candidate"],
        "Top_SMINA_Affinity_kcal_mol":
            ligand_entry["Docking_Score"],
        "Heavy_Atom_Count":
            ligand_heavy_atom_count,
        "Contact_Residue_Count":
            len(contact_df),
        "Geometric_Hydrogen_Bond_Count":
            len(hbond_df),
        "Minimum_Receptor_Ligand_Heavy_Atom_Distance_Angstrom":
            minimum_distance,
        "Severe_Clash_Pair_Count_Below_1_5_Angstrom":
            severe_clash_count,
        "Contact_Residues":
            ", ".join(contact_residue_labels),
        "Hydrogen_Bond_Residues":
            ", ".join(hbond_residue_labels),
        "Stereo_Only_3D_Assignment":
            ligand_entry["Stereo_Only_Assignment"]
    })


# ============================================================
# 13. Ενοποίηση και αποθήκευση αποτελεσμάτων
# ============================================================

contact_residues_df = (
    pd.concat(
        all_contact_frames,
        ignore_index=True
    )
    if all_contact_frames
    else pd.DataFrame()
)


hydrogen_bonds_df = (
    pd.concat(
        all_hbond_frames,
        ignore_index=True
    )
    if all_hbond_frames
    else pd.DataFrame()
)


interaction_summary_df = pd.DataFrame(
    summary_rows
)


contact_residues_df.to_csv(
    CONTACT_RESIDUES_OUTPUT,
    index=False
)

hydrogen_bonds_df.to_csv(
    HYDROGEN_BONDS_OUTPUT,
    index=False
)

interaction_summary_df.to_csv(
    INTERACTION_SUMMARY_OUTPUT,
    index=False
)


# ============================================================
# 14. Δημιουργία PyMOL εικόνων
# ============================================================

def create_pymol_script(
    candidate_id,
    candidate_pose_path,
    output_png_path
):
    """
    Δημιουργεί PyMOL script για μία Pareto pose.
    """

    return f"""
reinitialize

load {RECEPTOR_PDB.resolve().as_posix()}, receptor
load {RAJ_REFERENCE_SDF.resolve().as_posix()}, reference_raj
load {candidate_pose_path.resolve().as_posix()}, candidate_pose

hide everything

show cartoon, receptor
color gray70, receptor

select candidate_pocket, byres (
    receptor within {CONTACT_DISTANCE_ANGSTROM} of candidate_pose
)

show sticks, candidate_pocket
color gray50, candidate_pocket

show sticks, candidate_pose
color cyan, candidate_pose

show lines, reference_raj
color green, reference_raj

set transparency, 0.55, reference_raj
set cartoon_transparency, 0.72
set stick_radius, 0.20
set line_width, 2.0

set bg_rgb, [1.0, 1.0, 1.0]
set ray_opaque_background, on
set orthoscopic, on
set antialias, 2
set ray_trace_mode, 1
set ambient, 0.45
set specular, 0.25
set depth_cue, 0

orient candidate_pose or reference_raj
zoom candidate_pose or reference_raj, 7

ray 1800, 1400
png {output_png_path.resolve().as_posix()}, dpi=300

quit
""".strip()


pymol_log_sections = []

candidate_image_paths = []


for candidate_id in pareto_candidate_ids:

    candidate_pose_path = (
        DOCKING_RESULTS_DIRECTORY
        / candidate_id
        / f"{candidate_id}_TOP_POSE.pdbqt"
    )

    pymol_script_path = (
        OUTPUT_DIRECTORY
        / f"{candidate_id}_POSE_VISUALIZATION.pml"
    )

    output_png_path = (
        OUTPUT_DIRECTORY
        / f"{candidate_id}_TOP_POSE_VS_RAJ.png"
    )


    pymol_script_path.write_text(
        create_pymol_script(
            candidate_id,
            candidate_pose_path,
            output_png_path
        ) + "\n",
        encoding="utf-8"
    )


    pymol_process = subprocess.run(
        [
            pymol_path,
            "-cq",
            str(
                pymol_script_path.resolve()
            )
        ],
        capture_output=True,
        text=True,
        cwd=PROJECT_DIRECTORY,
        timeout=600
    )


    pymol_log_sections.append(
        (
            f"\n{'=' * 100}\n"
            f"{candidate_id}\n"
            f"{'=' * 100}\n"
            f"RETURN CODE: {pymol_process.returncode}\n\n"
            f"STDOUT\n{pymol_process.stdout}\n\n"
            f"STDERR\n{pymol_process.stderr}\n"
        )
    )


    if pymol_process.returncode != 0:
        raise RuntimeError(
            f"Απέτυχε η εικόνα του {candidate_id}."
        )


    if (
        not output_png_path.exists()
        or output_png_path.stat().st_size == 0
    ):
        raise RuntimeError(
            f"Δεν δημιουργήθηκε η εικόνα: "
            f"{output_png_path}"
        )


    candidate_image_paths.append(
        (
            candidate_id,
            output_png_path
        )
    )


PYMOL_LOG_OUTPUT.write_text(
    "\n".join(
        pymol_log_sections
    ),
    encoding="utf-8"
)


# ============================================================
# 15. Δημιουργία ενιαίου panel 2 × 2
# ============================================================

panel_width = 900

panel_height = 740

caption_height = 50


panel_images = []


for candidate_id, image_path in candidate_image_paths:

    source_image = PILImage.open(
        image_path
    ).convert(
        "RGB"
    )


    source_image.thumbnail(
        (
            panel_width,
            panel_height - caption_height
        )
    )


    canvas = PILImage.new(
        "RGB",
        (
            panel_width,
            panel_height
        ),
        "white"
    )


    x_offset = (
        panel_width
        - source_image.width
    ) // 2

    y_offset = caption_height


    canvas.paste(
        source_image,
        (
            x_offset,
            y_offset
        )
    )


    drawing = ImageDraw.Draw(
        canvas
    )


    drawing.text(
        (
            20,
            15
        ),
        candidate_id,
        fill="black"
    )


    panel_images.append(
        canvas
    )


grid_image = PILImage.new(
    "RGB",
    (
        panel_width * 2,
        panel_height * 2
    ),
    "white"
)


for image_index, panel_image in enumerate(
    panel_images
):

    column_index = (
        image_index % 2
    )

    row_index = (
        image_index // 2
    )


    grid_image.paste(
        panel_image,
        (
            column_index * panel_width,
            row_index * panel_height
        )
    )


grid_image.save(
    IMAGE_GRID_OUTPUT,
    dpi=(
        300,
        300
    )
)


# ============================================================
# 16. Metadata
# ============================================================

metadata_df = pd.DataFrame(
    [
        {
            "Parameter": "Pareto candidate count",
            "Value": str(
                len(pareto_candidate_ids)
            )
        },
        {
            "Parameter": "Pareto candidates",
            "Value": ", ".join(
                pareto_candidate_ids
            )
        },
        {
            "Parameter": "Contact cutoff Angstrom",
            "Value": str(
                CONTACT_DISTANCE_ANGSTROM
            )
        },
        {
            "Parameter": "Severe clash cutoff Angstrom",
            "Value": str(
                SEVERE_CLASH_DISTANCE_ANGSTROM
            )
        },
        {
            "Parameter": "Hydrogen-acceptor cutoff Angstrom",
            "Value": str(
                HBOND_HYDROGEN_ACCEPTOR_MAX
            )
        },
        {
            "Parameter": "Donor-acceptor cutoff Angstrom",
            "Value": str(
                HBOND_DONOR_ACCEPTOR_MAX
            )
        },
        {
            "Parameter": "Minimum D-H-A angle degrees",
            "Value": str(
                HBOND_MINIMUM_ANGLE_DEGREES
            )
        },
        {
            "Parameter": "Water-mediated interactions included",
            "Value": "False"
        },
        {
            "Parameter": "Interaction interpretation",
            "Value": (
                "Geometry-based analysis of docking poses; "
                "not experimental interaction evidence"
            )
        },
        {
            "Parameter": "Final lead selected",
            "Value": "False"
        }
    ],
    dtype="object"
)


metadata_df.to_csv(
    INTERACTION_METADATA_OUTPUT,
    index=False
)


# ============================================================
# 17. Εμφάνιση αποτελεσμάτων
# ============================================================

print("\n" + "=" * 110)
print("PARETO CANDIDATE INTERACTION SUMMARY")
print("=" * 110)

display(
    interaction_summary_df
)


print("\n" + "=" * 110)
print("GEOMETRIC HYDROGEN-BOND CANDIDATES")
print("=" * 110)

if hydrogen_bonds_df.empty:

    print(
        "Δεν εντοπίστηκαν hydrogen bonds "
        "με τα προκαθορισμένα γεωμετρικά κριτήρια."
    )

else:

    display(
        hydrogen_bonds_df
    )


print("\n" + "=" * 110)
print("PARETO DOCKING POSE PANEL")
print("=" * 110)

display(
    IPythonImage(
        filename=str(
            IMAGE_GRID_OUTPUT
        )
    )
)


# ============================================================
# 18. Τελική αναφορά
# ============================================================

print("\n" + "-" * 110)
print("Η INTERACTION ANALYSIS ΟΛΟΚΛΗΡΩΘΗΚΕ")
print("-" * 110)

print(
    f"Pareto candidates: "
    f"{', '.join(pareto_candidate_ids)}"
)

print(
    f"Interaction summary: "
    f"{INTERACTION_SUMMARY_OUTPUT}"
)

print(
    f"Contact residues: "
    f"{CONTACT_RESIDUES_OUTPUT}"
)

print(
    f"Hydrogen bonds: "
    f"{HYDROGEN_BONDS_OUTPUT}"
)

print(
    f"Pose grid: "
    f"{IMAGE_GRID_OUTPUT}"
)

print(
    "\nΔεν επιλέχθηκε ακόμη τελικό lead molecule."
)

In [ ]:
# ============================================================
# ΕΛΕΓΧΟΣ ΤΗΣ ΑΚΡΙΒΟΥΣ PARETO ΛΙΣΤΑΣ ΚΑΙ ΤΩΝ INTERACTIONS
#
# Δεν πραγματοποιείται κανένας νέος υπολογισμός ή docking.
# ============================================================

from pathlib import Path

import pandas as pd
from IPython.display import display


PROJECT_DIRECTORY = Path(
    "/home/afroditi/ptixiaki"
)

PARETO_FILE = (
    PROJECT_DIRECTORY
    / "SELECTED_20_DOCKING_PARETO_FRONT.csv"
)

INTERACTION_FILE = (
    PROJECT_DIRECTORY
    / "PARETO_4_PLUS_RAJ_INTERACTION_SUMMARY.csv"
)

HYDROGEN_BOND_FILE = (
    PROJECT_DIRECTORY
    / "PARETO_4_PLUS_RAJ_GEOMETRIC_HYDROGEN_BONDS.csv"
)


pareto_df = pd.read_csv(
    PARETO_FILE
)

interaction_df = pd.read_csv(
    INTERACTION_FILE
)

hydrogen_bond_df = pd.read_csv(
    HYDROGEN_BOND_FILE
)


pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_colwidth",
    None
)

pd.set_option(
    "display.width",
    250
)


print("=" * 100)
print("ΑΚΡΙΒΗΣ ΠΡΩΤΟΣ PARETO FRONT")
print("=" * 100)

display(
    pareto_df[
        [
            "Candidate_ID",
            "Top_SMINA_Affinity_kcal_mol",
            "Heavy_Atom_Count",
            "Ligand_Efficiency_kcal_mol_per_Heavy_Atom"
        ]
    ]
)


print("\n" + "=" * 100)
print("ΠΛΗΡΕΣ INTERACTION SUMMARY")
print("=" * 100)

display(
    interaction_df
)


print("\n" + "=" * 100)
print("ΠΛΗΡΕΙΣ ΓΕΩΜΕΤΡΙΚΟΙ HYDROGEN BONDS")
print("=" * 100)

display(
    hydrogen_bond_df
)


print("\nPareto IDs από το Pareto CSV:")
print(
    pareto_df["Candidate_ID"]
    .astype(str)
    .tolist()
)

print("\nCandidates που αναλύθηκαν:")
print(
    interaction_df.loc[
        interaction_df["Is_Pareto_Candidate"] == True,
        "Ligand_ID"
    ]
    .astype(str)
    .tolist()
)